# Large-FOV Nuclear Pipeline — v18

**v18** — Section 36 rebuilt as the analysis figure set: rose plots, membrane
intensity by angle and distance, membrane asymmetry vs time, and cross-sectional
area boxed into fixed acquisition-time intervals, and a new Section 37 with the
shell-resolved rose facets, the normalised asymmetry score, and the
direction-agnostic aggregate. All figures save to
`<run>/exports/figures/`. Data handling is unchanged from v17.

---

# Large-FOV Nuclear Pipeline — v18 (v9 segmentation contract)

**v16.2** — the training-set generator's Stage-2 gate (NPC-shell organisation +
membrane co-localisation) is now applied in the segmentation block. Ported from
`vulcan_training_1.1.ipynb` sections 4b / 5b / 6; see Section 7a. Adding config
fields changes the run signature, so this lands in a NEW `Runs/<run_id>/` and
v16.1 results are not overwritten.


**Improvements over the previous version**

- All imports consolidated in one place; no re-imports scattered across cells.
- `gc` and `skimage.filters` (required by the focus scorer) added to the import block.
- Critical logic bug in `assign_track_ids_hybrid_dbscan` fixed (misindented `continue`
  caused the match loop body to be skipped for every match).
- Duplicate `plot_largest_cross_sectional_area_vs_time` definition removed.
- `label_and_measure_objects` (Section 7) merged into the existing `regionprops_to_rows`
  path — one code path instead of two diverging implementations.
- `extract_objects_from_saved_masks` now skips `included=False` rows instead of loading
  empty mask files.
- Debug `print` statements removed from the Hungarian-assignment inner loop.
- `segment_single_plane_with_overlap` no longer called with an unsupported `batch_size=`
  kwarg; the config value is used instead.
- Model is loaded exactly once and reused across the debug and full-segmentation cells.
- Focus-scoring utilities moved above the cells that call them.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: Z floor/ceiling (`focus_min_z` / `focus_max_z`) applied to all focus scoring to exclude coverslip artefacts.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: NPC-ring focus scoring added for early timepoints where nuclear channel signal is absent.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: Adaptive three-way channel selection matching the training notebook logic.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: `npc_channel_index` added to `PipelineConfig`.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: `focus_score_npc_ring` added to focus scoring utilities.
- **v14**: Full 4-class model compatibility (nucleus=class 3, NPC=class 2, droplet=class 1).
  Channel input order corrected to match training (NLS, NPC, Membrane).
  All hardcoded class counts replaced with config.num_classes.
- **v13**: `get_best_focus_z_indices_adaptive` replaces single-channel `get_best_focus_z_indices` in the segmentation loop.
- `configure_tensorflow_for_gpu` called once at runtime initialisation (Section 19).
- `plot_largest_cross_sectional_area_vs_time` unified: scatter + mean-line in one function.

**Workflow**

Run top-to-bottom the first time.  After that, flip stage flags in **Section 18** and
rerun only the cells you need — every stage writes intermediate artefacts to disk.

*v16: integrates the droplet-bounded radial sweep (Section 19 + 35b) directly on tracked nuclei; RadialProfile now populated in the database export (Section 44).*


## 1. Imports

In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import shutil

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage import filters, measure, morphology

try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    import tifffile as tiff
except Exception:
    tiff = None

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None


2026-09-02 13:45:47.791949: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-02 13:45:47.792017: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-02 13:45:47.793062: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-02 13:45:47.800121: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-02 13:45:49.610833: W tensorflow/compiler/tf2

## 2. Configuration

In [2]:
def get_n_workers() -> int:
    """"Return the number of CPU Cores Avalible for Paralell Processing"""
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # ── Root paths ────────────────────────────────────────────────────────
    model_name: str = "Vulcan_1.1_best.keras"
    input_image_name: str = "control_extract_1.1.tif"

    # ── Imaging ───────────────────────────────────────────────────────────
    pixel_size_um: float = 0.1625
    z_step_um: float = 1.0
    nuclear_channel_index: int = 1
    membrane_channel_index: int = 0
    npc_channel_index: int = 2

    # ── Halo analysis ─────────────────────────────────────────────────────
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 4

    # ── Radial sweep (v16) ───────────────────────────────────────
    # Rays cast from each tracked nucleus centroid to its droplet edge,
    # sampling the membrane / NE channel. One row per ray sample point.
    radial_channel_index: int = 0        # membrane / NE channel (matches membrane_channel_index)
    radial_n_angles: int = 180
    radial_step_size_px: float = 1.0
    radial_exclude_nucleus: bool = True  # drop ray points falling inside the nucleus
    radial_max_ray_steps: int = 4000

    # ── Grouping / tracking ───────────────────────────────────────────────
    z_group_tolerance_um: float = 1.0
    time_track_tolerance_um: float = 5.0
    multi_nucleus_exclusion_um: float = 1.0

    track_dbscan_eps_um: float = 6.0
    track_dbscan_min_samples: int = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0

    # ── Stitched acquisition layout ───────────────────────────────────────
    tile_rows: int = 2
    tile_cols: int = 3
    minutes_per_tile: float = 1.0
    serpentine_scan: bool = True

    # ── Segmentation ──────────────────────────────────────────────────────
    # ── v9 per-channel sigmoid extraction thresholds ──────────────────────
    # v9 head = 4 INDEPENDENT sigmoids. Each class thresholded on its OWN
    # channel, never argmax (classes overlap: nucleus subset of droplet, NPC
    # ring on the envelope). These are the per-experiment knobs to sweep.
    droplet_threshold: float = 0.5
    npc_threshold:     float = 0.3
    nucleus_threshold: float = 0.5
    # ── Re-extraction source (layer 3) ────────────────────────────────────
    # None -> run inference from input_image_name, write a probability hyperstack.
    # str  -> skip the model; load this probability hyperstack and re-extract
    #         masks at the current thresholds (cheap threshold sweeps). Points
    #         at any dataset's saved probability map.
    probability_hyperstack_input: Optional[str] = None
    min_nucleus_area_px: int = 1065
    nucleus_hole_size_px: int = 10647
    nucleus_opening_radius_px: int = 5
    min_nucleus_circularity: float = 0.65
    batch_size: int = 8   # tune down if GPU OOM, raise to 16-32 if VRAM permits
    # ── Stage-2 gate: NPC-shell organisation + membrane co-localisation ───
    # PORT OF THE TRAINING-SET GATE (vulcan_training_1.1, sections 5b + 6).
    # Field names and defaults are copied so a diff between the two configs is
    # meaningful. Change them in BOTH places or the model and the pipeline stop
    # agreeing on what a nucleus is.
    #
    # NOTE ON WHAT THIS ACTUALLY TESTS. The generator gate is NPC + MEMBRANE,
    # not NLS + NPC. NLS enters upstream, as the detector that proposes the
    # nucleus (detect_nucleus_adaptive); in v16 the U-Net's nucleus channel
    # plays that role. The gate then asks the independent question "is there an
    # organised envelope around this thing?" — which is exactly what a droplet
    # cap fails. NLS enrichment is recorded as a diagnostic column but is NOT
    # part of the pass/fail decision, matching the generator.
    require_stage2_gate: bool = True

    # Skip the gate for t < this (None = gate every timepoint). The generator
    # sets generation_min_timepoint = 2 and never gates t = 0-1: too few puncta
    # for a shell fit and the NE probe is too faint to co-localise. Applying the
    # gate there will therefore reject nearly everything. Leaving this at None
    # means unverified early objects are dropped rather than emitted — set it to
    # 2 to pass t = 0-1 through ungated instead.
    gate_min_timepoint: Optional[int] = None

    # NPC puncta (RAW channel, nucleus-boundary anchored)
    npc_margin_px: int = 5            # annular half-width around the nucleus edge
    npc_std_mult: float = 2.0         # thresh = mean + k*std of RAW NPC in droplet

    # Shell organisation (primary test)
    gate_min_puncta: int = 8
    gate_shell_tol_px: float = 6.0
    gate_shell_min_inlier_frac: float = 0.45
    ransac_n_iter: int = 200

    # Membrane co-localisation (secondary test)
    gate_membrane_nn_radius_px: int = 6
    gate_membrane_min_coloc_frac: float = 0.30
    gate_membrane_k_std: float = 1.0

    # Droplet-relative area window, from the generator's nucleus detector
    # (_clean_and_gate). v16 only ever applied an ABSOLUTE min_nucleus_area_px,
    # so a whole-droplet false positive (frac ~ 1.0) and a merged multi-droplet
    # blob both sailed through. This ratio is calibration-free — it does not
    # move when pixel_size_um is finally settled.
    gate_use_droplet_area_fraction: bool = True
    nucleus_min_frac: float = 0.01
    nucleus_max_frac: float = 0.50

    # ── Beyond generator parity — READ THE VERIFICATION NOTE IN SECTION 7a ──
    # The ported shell test is close to vacuous. Its puncta zone is ALREADY a
    # 10 px annulus (dilate 5 / erode 5) and the RANSAC tolerance is 6 px, so
    # every suprathreshold pixel in the zone is an inlier to a circle at the
    # nucleus boundary by construction. On a synthetic droplet cap with a FLAT
    # NPC channel the ported gate passes 8/8 trials on Gaussian noise alone.
    # It does not reject caps.
    #
    # These two knobs restore the discrimination the gate was designed to have.
    # Both default to exact generator parity (no effect). Turn them on
    # deliberately, and mirror the change in vulcan_training_1.1.
    gate_puncta_min_size_px: int = 0                 # drop noise puncta; try 4-9
    gate_min_ring_contrast: Optional[float] = None   # ring/core raw NPC; try 1.15

    # Optional shell-radius sanity check, absent from the generator. The
    # generator records shell_r and never tests it, so a ring of puncta on the
    # DROPLET wall fits a circle and passes. Bounds are fractions of the parent
    # droplet's equivalent radius; measured r/R ~ 0.53. None = generator parity.
    gate_shell_r_frac_min: Optional[float] = None
    gate_shell_r_frac_max: Optional[float] = None

    # Safety valve: if more than this fraction of a plane's pixels exceed the
    # nucleus_threshold the plane is almost certainly a bad Z (model
    # over-fired on bright droplet walls / out-of-focus texture).  Skip mask
    # cleaning entirely and return an empty mask.  7 ms to check vs 20 min to clean.
    max_nucleus_foreground_fraction: float = 0.25
    use_mixed_precision: bool = False
    patch_size: int = 512
    patch_stride: int = 256
    background_class_index: int = 0
    droplet_class_index: int = 1
    npc_class_index:     int = 2
    nucleus_class_index: int = 3
    num_classes: int = 4

    # ── Focus-aware Z selection ───────────────────────────────────────────
    use_focus_z_selection: bool = True
    focus_window_radius: int = 6
    focus_edge_z_exclusion: int = 2
    focus_channel_index: Optional[int] = None
    focus_metric: str = "object_based_nuclear"
    # Droplet-resistant focus scorer tuning (passed to focus_score_object_based_nuclear)
    focus_min_nucleus_area_um2: float = 20.0    # lower nucleus size bound in µm²
    focus_max_nucleus_area_um2: float = 800.0   # upper bound — set BELOW droplet area
    focus_threshold_k: float = 1.5             # mean + k*std threshold selectivity
    focus_sharpness_weight: float = 0.5        # 0=off, 0.5=moderate, 1.0=strong

    # ── v13: Z floor/ceiling — exclude coverslip artefacts ──────────────
    # Bright round objects near the coverslip produce high nuclear channel
    # focus scores even when no nucleus is present. Setting focus_min_z
    # to the first artefact-free plane eliminates these false peaks.
    # Must match focus_min_z / focus_max_z used during training patch generation.
    focus_min_z: int = 6        # exclude Z < focus_min_z from focus scoring
    focus_max_z: Optional[int] = None  # None = only upper edge exclusion applies

    # ── v13: Adaptive channel selection ─────────────────────────────────
    # Nucleus channel Laplacian peak (across all valid Z) must exceed this
    # floor to use nuclear-channel scoring. Below it: NPC-ring scoring.
    focus_nucleus_score_floor: float = 0.008
    # Above floor but below this threshold: contrast-based nuclear scoring.
    # Above this threshold: Laplacian nuclear scoring.
    focus_nucleus_laplacian_threshold: float = 0.012
    # NPC-ring Laplacian variance minimum for a Z plane to be accepted.
    focus_npc_ring_min_score: float = 0.001
    # Erosion radius (px) applied to droplet mask before NPC-ring scoring.
    # Should match droplet_focus_erosion_px used in training.
    focus_npc_erosion_px: int = 20

    # ── HPC ───────────────────────────────────────────────────────────────
    n_workers: Optional[int] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)
        self.paths = None   # set by the path-resolver bootstrap cell

    # ── Unit conversions (derived) ────────────────────────────────────────
    @property
    def z_group_tolerance_px(self) -> float:
        return self.z_group_tolerance_um / self.pixel_size_um

    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def multi_nucleus_exclusion_px(self) -> float:
        return self.multi_nucleus_exclusion_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    @property
    def effective_focus_channel_index(self) -> int:
        return self.nuclear_channel_index if self.focus_channel_index is None else self.focus_channel_index

    # ── Path layer — delegates to the run-scoped resolver ────────────────
    # cfg.paths (a RunPaths) is set once by the bootstrap cell after the run is
    # created. Nothing is hard-coded; every output is isolated under
    # Runs/<run_id>/, which is what makes stale-pickle mixing impossible.
    def _require_paths(self):
        if getattr(self, "paths", None) is None:
            raise RuntimeError(
                "cfg.paths is not set — run the path-resolver bootstrap cell "
                "before touching any path.")
        return self.paths

    # Immutable inputs (data_root/Inputs, via the parent ProjectPaths)
    @property
    def model_path(self) -> Path:
        return self._require_paths().project.models_dir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return self._require_paths().project.raw_images_dir / self.input_image_name

    # Run-scoped output directories
    @property
    def output_dir(self) -> Path:
        return self._require_paths().run_dir

    @property
    def seg_dir(self) -> Path:
        return self._require_paths().seg_dir

    @property
    def obj_dir(self) -> Path:
        return self._require_paths().obj_dir

    @property
    def track_dir(self) -> Path:
        return self._require_paths().track_dir

    @property
    def analysis_dir(self) -> Path:
        return self._require_paths().analysis_dir

    @property
    def qc_dir(self) -> Path:
        return self._require_paths().qc_dir

    @property
    def mask_tif_dir(self) -> Path:
        return self._require_paths().mask_tif_dir

    @property
    def exports_dir(self) -> Path:
        return self._require_paths().exports_dir

    # ── Segmentation artefact paths ───────────────────────────────────────
    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_probability_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_probability_hyperstack.tif"

    @property
    def probability_input_path(self) -> Optional[Path]:
        """Layer 3: if set, re-extract from this probability hyperstack instead of
        running inference. Any dataset's saved probability map can be pointed at."""
        if self.probability_hyperstack_input is None:
            return None
        return Path(self.probability_hyperstack_input)

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    # location-only fields excluded from the results signature (robust whether
    # or not the fields were deleted above)
    _LOCATION_FIELDS = ("project_root", "model_subdir", "image_subdir", "output_subdir")

    def to_run_signature(self) -> Dict[str, object]:
        """Config subset that defines a run's RESULTS (not where they're stored).
        Hashed into the run id and checked by the stale-guard."""
        d = asdict(self)
        for k in self._LOCATION_FIELDS:
            d.pop(k, None)
        return d

    def __repr__(self) -> str:
        keys = [
            "model_name", "input_image_name",
            "pixel_size_um", "nuclear_channel_index", "membrane_channel_index",
            "z_group_tolerance_um", "time_track_tolerance_um",
            "multi_nucleus_exclusion_um", "track_dbscan_eps_um",
            "min_nucleus_area_px", "patch_size", "patch_stride",
            "n_workers", "use_mixed_precision",
        ]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)



---
# v17 — CONFIG CORRECTIONS

Three values in `PipelineConfig` are wrong for this rig and every downstream
number depends on them. Applied here rather than edited in place so the
change is visible in the run log and in the config hash.

| field | was | is | why it matters |
|---|---|---|---|
| `pixel_size_um` | 0.2167 | **0.1625** | areas scale with the SQUARE — every area was 78 % too large |
| `z_step_um` | 1.0 | **2.0** | every z-extent and volume calculation |
| `nucleus_area_max_um2` | 300 (in the §31 debug call) | **600** | the §31 call used 300 while the function default was 600 |

The pixel size was confirmed optically (6.5 µm sensor / 40×) and
geometrically. Correcting it changes the run hash, so outputs land in a new
run directory rather than mixing with earlier results.

## 2b. Path resolver + run bootstrap

In [3]:
# =====================================================================
# Path resolver — single source of truth for every filesystem location.
# Reads the logical layout in STRUCTURE and materialises it under two
# physical roots. On Cheaha: code_root=$HOME/Projects (small, git),
# data_root=/data/user/$USER/<project> (5 TB, all inputs + outputs).
# Every run writes under Runs/<run_id>/, so a stale pickle from a
# different config can never be loaded by accident. (Mirrors structure.json.)
# =====================================================================
import getpass, hashlib, datetime as _dt
import json as _json
from pathlib import Path as _Path

STRUCTURE = {
    "code_root": {"Python": {"Environments": None,
                             "Model_Training": {"Training_Scripts": None},
                             "Image_Segmentation": None, "Image_Analysis": None},
                  "R_Projects": None},
    "data_root": {"Inputs": {"Raw_Images": None, "Models": None, "Training_Data": None},
                  "Runs": None, "Legacy": None},
    "run_template": {"segmentation": None, "objects": None, "tracking": None,
                     "analysis": None, "qc": None, "mask_tifs": None, "exports": None},
}

_RUN_STAGE_ATTRS = {"segmentation": "seg_dir", "objects": "obj_dir",
                    "tracking": "track_dir", "analysis": "analysis_dir",
                    "qc": "qc_dir", "mask_tifs": "mask_tif_dir",
                    "exports": "exports_dir"}


def _build_tree(base, spec):
    """Recursively create a directory tree under base according to the spec dict."""
    base.mkdir(parents=True, exist_ok=True)
    if not spec:
        return
    for name, child in spec.items():
        if str(name).startswith("_"):
            continue
        _build_tree(base / name, child)


class ProjectPaths:
    """Resolve all filesystem locations for a project, given two roots:"""
    def __init__(self, code_root, data_root, structure=STRUCTURE, create=True):
        self.spec = structure
        self.code_root = _Path(code_root).expanduser().resolve()
        self.data_root = _Path(data_root).expanduser().resolve()
        self.run_template = structure.get("run_template", {})
        self.raw_images_dir    = self.data_root / "Inputs" / "Raw_Images"
        self.models_dir        = self.data_root / "Inputs" / "Models"
        self.training_data_dir = self.data_root / "Inputs" / "Training_Data"
        self.runs_dir          = self.data_root / "Runs"
        self.legacy_dir        = self.data_root / "Legacy"
        if create:
            _build_tree(self.code_root, self.spec.get("code_root"))
            _build_tree(self.data_root, self.spec.get("data_root"))

    @classmethod
    def for_cheaha(cls, project_name, code_subdir="Projects", create=True):
        """Resolve paths for a Cheaha HPC. The code root is $HOME/<code_subdir>,"""
        user = getpass.getuser()
        home = _Path(os.environ.get("HOME", f"/home/{user}"))
        user_data = _Path(os.environ.get("USER_DATA", f"/data/user/{user}")).expanduser()
        return cls(home / code_subdir, user_data / project_name, create=create)

    @staticmethod
    def make_run_id(label, config_dict, timestamp=False):
        """Stable per config: <label>__cfg-<hash>. Same config -> same dir, so
        RUN_*=False loads reuse the existing run. Any config change flips the
        hash -> a new dir -> no stale mixing. Set timestamp=True to force a
        distinct dir for same-config reruns."""
        h = hashlib.sha1(
            _json.dumps(config_dict, sort_keys=True, default=str).encode()).hexdigest()[:8]
        safe = "".join(c if (c.isalnum() or c in "-._") else "_" for c in str(label))
        if timestamp:
            return f"{safe}__{_dt.datetime.now():%Y%m%d_%H%M}__cfg-{h}"
        return f"{safe}__cfg-{h}"

    def for_run(self, run_id, create=True):
        run_dir = self.runs_dir / run_id
        if create:
            _build_tree(run_dir, self.run_template)
        return RunPaths(self, run_id, run_dir)

    def list_runs(self):
        if not self.runs_dir.exists():
            return []
        return sorted(p.name for p in self.runs_dir.iterdir() if p.is_dir())


class RunPaths:
    def __init__(self, project, run_id, run_dir):
        self.project, self.run_id, self.run_dir = project, run_id, run_dir
        for stage, attr in _RUN_STAGE_ATTRS.items():
            setattr(self, attr, run_dir / stage)
        self.config_snapshot_path = run_dir / "config.json"
        self.manifest_path = run_dir / "manifest.json"

    def snapshot_config(self, config_dict):
        self.config_snapshot_path.write_text(
            _json.dumps(config_dict, indent=2, sort_keys=True, default=str))

    def load_snapshot(self):
        if self.config_snapshot_path.exists():
            return _json.loads(self.config_snapshot_path.read_text())
        return None

    def assert_config_matches(self, config_dict, strict=True):
        """Refuse to trust cached stages if the live config drifted from the
        snapshot that produced this run's data. Call before any RUN_*=False load."""
        snap = self.load_snapshot()
        if snap is None:
            return
        diffs = {k: (snap.get(k), config_dict.get(k))
                 for k in set(snap) | set(config_dict)
                 if snap.get(k) != config_dict.get(k)}
        if diffs:
            msg = ("[stale-guard] live config differs from snapshot in run '%s':\n%s"
                   % (self.run_id, "\n".join(f"    {k}: snapshot={s!r} live={l!r}"
                                             for k, (s, l) in diffs.items())))
            if strict:
                raise RuntimeError(msg)
            print("WARNING:", msg)

    def write_manifest(self, **fields):
        m = {}
        if self.manifest_path.exists():
            m = _json.loads(self.manifest_path.read_text())
        m.update(fields)
        m["updated_at"] = _dt.datetime.now().isoformat(timespec="seconds")
        self.manifest_path.write_text(_json.dumps(m, indent=2, default=str))


In [4]:
# ── Instantiate config, create the run, wire paths, snapshot ──────────
# Change project_name / label here if needed. run_id is stable per config,
# so RUN_*=False cells below reuse this run's directory.
cfg = PipelineConfig()

paths = ProjectPaths.for_cheaha(project_name="Nuclear_Scaling")
run = paths.for_run(ProjectPaths.make_run_id(
    label=Path(cfg.input_image_name).stem,
    config_dict=cfg.to_run_signature(),
))
cfg.paths = run
run.snapshot_config(cfg.to_run_signature())
run.write_manifest(input_image=cfg.input_image_name,
                   model=cfg.model_name, run_id=run.run_id)

print("Run id :", run.run_id)
print("Run dir:", run.run_dir)
print("Inputs :", paths.raw_images_dir, "|", paths.models_dir)
cfg


Run id : control_extract_1.1__cfg-a6725cc8
Run dir: /data/user/tdeibert/Nuclear_Scaling/Runs/control_extract_1.1__cfg-a6725cc8
Inputs : /data/user/tdeibert/Nuclear_Scaling/Inputs/Raw_Images | /data/user/tdeibert/Nuclear_Scaling/Inputs/Models


model_name: Vulcan_1.1_best.keras
input_image_name: control_extract_1.1.tif
pixel_size_um: 0.1625
nuclear_channel_index: 1
membrane_channel_index: 0
z_group_tolerance_um: 1.0
time_track_tolerance_um: 5.0
multi_nucleus_exclusion_um: 1.0
track_dbscan_eps_um: 6.0
min_nucleus_area_px: 1065
patch_size: 512
patch_stride: 256
n_workers: 8
use_mixed_precision: False

In [5]:
cfg.pixel_size_um = 0.1625
cfg.z_step_um = 2.0

# ---- fragmentation repair (v17) ----------------------------------------
cfg.repair_enabled = True
cfg.repair_flatten_um = 8.0        # reconstruction SE; sweep was flat 3-8
cfg.repair_npc_slack = 1.10        # allowed overshoot past the NPC ring
cfg.repair_min_solidity = 0.93     # fragment detector
cfg.repair_area_deficit = 0.50     # fragment detector, relative to timepoint p75
cfg.repair_z_pad = 6               # planes searched either side
cfg.repair_max_frac_droplet = 0.60
cfg.repair_min_signed_margin = 1.15
cfg.repair_edge_margin_um = 2.0
cfg.repair_npc_min_rays = 36       # of 72
cfg.repair_win_um = 45.0

print(f"pixel {cfg.pixel_size_um} um/px | z-step {cfg.z_step_um} um | "
      f"repair {'ON' if cfg.repair_enabled else 'OFF'} "
      f"(flatten {cfg.repair_flatten_um} um, NPC slack {cfg.repair_npc_slack})")

pixel 0.1625 um/px | z-step 2.0 um | repair ON (flatten 8.0 um, NPC slack 1.1)


## 3. Save / load configuration

In [6]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


save_config(cfg, cfg.output_dir / "pipeline_config.json")
print("Configuration saved.")


Configuration saved.


## 4. GPU setup

In [7]:
def configure_tensorflow_for_gpu(config: PipelineConfig) -> None:
    if tf is None:
        print("TensorFlow not available — skipping GPU setup.")
        return

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    if getattr(config, "use_mixed_precision", False):
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled.")
        except Exception as exc:
            print("Could not enable mixed precision:", exc)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print("Could not set memory growth:", exc)


## 5. Image loading helpers

In [8]:
def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print("Loaded image shape:", arr.shape)
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(
    img_5d: np.ndarray,
    t_idx: int,
    z_idx: int,
    config: "PipelineConfig" = None,
) -> np.ndarray:
    """
    Return a 3-channel plane in (Y, X, C) format ordered to match training input.

    Training channel order: (NLS/nucleus, NPC, Membrane)
    Hyperstack channel order: Ch0=Membrane, Ch1=NLS, Ch2=NPC

    If config is provided, uses config channel indices for explicit ordering.
    Falls back to default training order (1, 2, 0) if config is None.
    """
    plane_cyx = img_5d[t_idx, z_idx]  # shape: (C, Y, X)
    if config is not None:
        ch_order = [
            config.nuclear_channel_index,   # NLS first  (training ch0)
            config.npc_channel_index,        # NPC second (training ch1)
            config.membrane_channel_index,   # Mem third  (training ch2)
        ]
    else:
        ch_order = [1, 2, 0]  # default: NLS, NPC, Membrane
    return np.moveaxis(plane_cyx[ch_order], 0, -1)  # (Y, X, C)


## 6. Focus-scoring utilities

Placed before segmentation so they are available for Z-selection during the segmentation loop.

In [9]:
def _normalize_2d(img2d: np.ndarray) -> np.ndarray:
    """Normalise a 2-D plane to [0, 1] for focus scoring."""
    arr = np.asarray(img2d, dtype=np.float32)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)


def _area_um2_to_px(area_um2: float, pixel_size_um: float) -> int:
    """Convert an area in µm² to pixels²"""
    return max(1, int(np.ceil(area_um2 / pixel_size_um ** 2)))


def focus_score_variance_laplacian(img2d: np.ndarray) -> float:
    """Whole-plane variance-of-Laplacian (fallback / debug)."""
    return float(np.nanvar(ndi.laplace(_normalize_2d(img2d))))


def focus_score_nucleus_weighted_laplacian(
    img2d: np.ndarray,
    candidate_percentile: float = 97.5,
    min_candidate_pixels: int = 50,
) -> float:
    """Bright-pixel-weighted Laplacian focus score (retained for comparison)."""
    img_f = _normalize_2d(img2d)
    finite = np.isfinite(img_f)
    if not finite.any():
        return 0.0
    cutoff = np.nanpercentile(img_f[finite], candidate_percentile)
    candidate_mask = (img_f >= cutoff) & finite
    if int(candidate_mask.sum()) < min_candidate_pixels:
        return 0.25 * focus_score_variance_laplacian(img2d)
    lap = ndi.laplace(img_f)
    weights = np.maximum(img_f[candidate_mask], 1e-6)
    weighted_energy = np.average(lap[candidate_mask] ** 2, weights=weights)
    signal_bonus = np.sqrt(max(float(candidate_mask.mean()), 1e-8))
    return float(weighted_energy * signal_bonus)


def focus_score_hybrid_nuclear_laplacian(
    img2d: np.ndarray,
    global_weight: float = 0.2,
    nuclear_weight: float = 0.8,
) -> float:
    """Hybrid score (retained for comparison / debugging)."""
    return float(
        global_weight * focus_score_variance_laplacian(img2d)
        + nuclear_weight * focus_score_nucleus_weighted_laplacian(img2d)
    )


def focus_score_object_based_nuclear(
    img2d: np.ndarray,
    pixel_size_um: float,
    min_nucleus_area_um2: float = 20.0,
    max_nucleus_area_um2: float = 500.0,
    blur_sigma: float = 1.5,
    threshold_k: float = 1.5,
    min_signal_fraction: float = 0.005,
    sharpness_weight: float = 0.5,
    min_object_count: int = 1,
) -> float:
    """Object-based nuclear focus score — droplet-resistant version."""
    nuc = _normalize_2d(img2d)
    if np.nanmax(nuc) <= 0:
        return 0.0
    mean_val = float(np.nanmean(nuc))
    if float((nuc > mean_val).mean()) < min_signal_fraction:
        return 0.0
    nuc_blur = filters.gaussian(nuc, sigma=blur_sigma, preserve_range=True)
    thr = mean_val + threshold_k * float(np.nanstd(nuc_blur))
    if thr >= 1.0:
        return 0.0
    min_area_px = _area_um2_to_px(min_nucleus_area_um2, pixel_size_um)
    max_area_px = _area_um2_to_px(max_nucleus_area_um2, pixel_size_um)
    mask = morphology.remove_small_objects(
        (nuc_blur > thr).astype(bool), min_size=min_area_px)
    lap = ndi.laplace(nuc)
    plane_lap_var = float(np.var(lap)) + 1e-12
    labeled = measure.label(mask)
    score = 0.0
    valid_count = 0
    for p in measure.regionprops(labeled, intensity_image=nuc):
        if p.area < min_area_px or p.area > max_area_px:
            continue
        roundness_weight = max(0.05, 1.0 - float(p.eccentricity))
        solidity_weight  = max(0.05, float(getattr(p, "solidity", 1.0)))
        base = float(p.area) * float(p.mean_intensity) * roundness_weight * solidity_weight
        if sharpness_weight > 0:
            obj_pixels = lap[labeled == p.label]
            sharpness = float(np.var(obj_pixels)) / plane_lap_var
            combined_weight = 1.0 + sharpness_weight * sharpness
        else:
            combined_weight = 1.0
        score += base * combined_weight
        valid_count += 1
    return float(score) if valid_count >= min_object_count else 0.0


_FOCUS_METRICS = {
    "object_based_nuclear":       lambda img, pxum: focus_score_object_based_nuclear(img, pixel_size_um=pxum),
    "variance_laplacian":         lambda img, pxum: focus_score_variance_laplacian(img),
    "nucleus_weighted_laplacian": lambda img, pxum: focus_score_nucleus_weighted_laplacian(img),
    "hybrid_nuclear_laplacian":   lambda img, pxum: focus_score_hybrid_nuclear_laplacian(img),
}


# ── v13: NPC-ring and adaptive focus scoring ──────────────────────────────

def focus_score_npc_ring(
    npc2d: np.ndarray,
    droplet_mask: Optional[np.ndarray] = None,
    erosion_px: int = 20,
) -> float:
    """
    Laplacian-variance focus score on the NPC boundary ring.
    If droplet_mask provided, restricts scoring to the annular ring region.
    Falls back to whole-plane Laplacian if no mask or ring is empty.
    """
    npc_f = _normalize_2d(npc2d)
    if droplet_mask is None:
        return float(np.nanvar(ndi.laplace(npc_f)))
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    interior = dist >= erosion_px
    ring = droplet_mask.astype(bool) & ~interior
    if not ring.any():
        return float(np.nanvar(ndi.laplace(npc_f)))
    lap_sq = ndi.laplace(npc_f) ** 2
    return float(np.mean(lap_sq[ring]))


def _z_in_focus_range(z: int, n_z: int, config: "PipelineConfig") -> bool:
    """Combined Z validity check: edge exclusion + floor + ceiling."""
    excl = config.focus_edge_z_exclusion
    if excl > 0 and (z < excl or z >= n_z - excl):
        return False
    if config.focus_min_z is not None and z < config.focus_min_z:
        return False
    if config.focus_max_z is not None and z > config.focus_max_z:
        return False
    return True


def _focus_channel_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    config: "PipelineConfig",
    valid_zs: List[int],
) -> str:
    """
    Three-way adaptive channel selection for one timepoint.

    Tests the nuclear channel Laplacian peak across valid Z planes:
      peak < focus_nucleus_score_floor       -> 'npc_ring'
      floor <= peak < laplacian_threshold    -> 'nucleus_contrast'
      peak >= laplacian_threshold            -> 'nucleus_laplacian'
    """
    peak_scores = []
    for z in valid_zs:
        nuc2d = img_5d[t, z, config.nuclear_channel_index].astype(np.float32)
        nuc_f = _normalize_2d(nuc2d)
        lap_sq = ndi.laplace(nuc_f) ** 2
        peak_scores.append(float(np.mean(lap_sq)))

    peak_lap = float(np.max(peak_scores)) if peak_scores else 0.0

    if peak_lap < config.focus_nucleus_score_floor:
        return 'npc_ring'
    elif peak_lap < config.focus_nucleus_laplacian_threshold:
        return 'nucleus_contrast'
    else:
        return 'nucleus_laplacian'


def get_best_focus_z_indices_adaptive(
    img_5d: np.ndarray,
    t: int,
    config: "PipelineConfig",
) -> Tuple[List[int], np.ndarray, Optional[int], str]:
    """
    NPC-ring focus scoring with droplet-mask restriction.
    Universal across all timepoints — NPC ring is reliably sharp at the
    correct focal plane regardless of nuclear development stage.
    Reference droplet mask is computed from the Z with the highest droplet
    count in the valid range (matching training notebook get_reference_z logic).
    Returns (keep_z, scores, best_z, channel).
    """
    from skimage.filters import threshold_local

    n_z = img_5d.shape[1]
    valid_zs = [z for z in range(n_z) if _z_in_focus_range(z, n_z, config)]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    channel = 'npc_ring'

    if not valid_zs:
        return [], scores, None, channel

    # Find reference Z with highest droplet count in valid range
    # Mirrors get_reference_z from training notebook
    best_ref_z = valid_zs[len(valid_zs) // 2]  # fallback
    best_ref_count = 0
    for z in valid_zs:
        npc_z = _normalize_2d(img_5d[t, z, config.npc_channel_index])
        npc_blur_z = filters.gaussian(npc_z, sigma=8.0, preserve_range=True)
        local_thresh_z = threshold_local(npc_blur_z, block_size=101, offset=-0.02)
        mask_z = morphology.remove_small_objects(
            (npc_blur_z > local_thresh_z).astype(bool), min_size=4000)
        n_droplets = len(np.unique(measure.label(mask_z))) - 1
        if n_droplets > best_ref_count:
            best_ref_count = n_droplets
            best_ref_z = z

    # Build reference droplet mask at the best-count Z
    npc_ref = _normalize_2d(img_5d[t, best_ref_z, config.npc_channel_index])
    npc_blur = filters.gaussian(npc_ref, sigma=8.0, preserve_range=True)
    local_thresh = threshold_local(npc_blur, block_size=101, offset=-0.02)
    droplet_mask_ref = morphology.remove_small_objects(
        (npc_blur > local_thresh).astype(bool), min_size=4000)

    print(f"    ref_z={best_ref_z} ({best_ref_count} droplets)")

    # Score each valid Z using ring-restricted NPC Laplacian
    for z in valid_zs:
        npc2d = img_5d[t, z, config.npc_channel_index]
        scores[z] = focus_score_npc_ring(
            npc2d,
            droplet_mask=droplet_mask_ref,
            erosion_px=config.focus_npc_erosion_px)

    # Apply minimum score threshold
    scores = np.where(
        scores < config.focus_npc_ring_min_score,
        np.nan, scores)

    if np.all(np.isnan(scores)):
        return [], scores, None, channel

    best_z = int(np.nanargmax(scores))
    excl = config.focus_edge_z_exclusion
    z_start = max(excl, best_z - config.focus_window_radius)
    z_stop  = min(n_z - excl, best_z + config.focus_window_radius + 1)
    keep_z  = [z for z in range(z_start, z_stop)
               if _z_in_focus_range(z, n_z, config)]
    return keep_z, scores, best_z, channel

def focus_score_plane(img2d: np.ndarray, metric: str = "object_based_nuclear",
                      pixel_size_um: float = 0.108,
                      config: Optional["PipelineConfig"] = None) -> float:
    """Dispatch focus metric."""
    if metric not in _FOCUS_METRICS:
        raise ValueError(f"Unsupported focus metric: {metric!r}. Choose from {list(_FOCUS_METRICS)}")
    if metric == "object_based_nuclear" and config is not None:
        return focus_score_object_based_nuclear(
            img2d, pixel_size_um=pixel_size_um,
            min_nucleus_area_um2=config.focus_min_nucleus_area_um2,
            max_nucleus_area_um2=config.focus_max_nucleus_area_um2,
            threshold_k=config.focus_threshold_k,
            sharpness_weight=config.focus_sharpness_weight,
        )
    return _FOCUS_METRICS[metric](img2d, pixel_size_um)


def get_focus_scores_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> np.ndarray:
    """Score every Z plane; excluded planes set to NaN. Respects focus_min_z/max_z."""
    n_z = img_5d.shape[1]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    for z in range(n_z):
        if config is not None:
            if not _z_in_focus_range(z, n_z, config):
                continue
        elif z < exclude_edge_z or z >= n_z - exclude_edge_z:
            continue
        scores[z] = focus_score_plane(img_5d[t, z, nuc_channel_idx],
                                      metric, pixel_size_um, config=config)
    return scores


def get_best_focus_z_indices(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    window_radius: int = 1,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> Tuple[List[int], np.ndarray, Optional[int]]:
    """Return (keep_z, scores, best_z) for the best-focus window at time t."""
    scores = get_focus_scores_for_timepoint(
        img_5d, t, nuc_channel_idx, exclude_edge_z, metric, pixel_size_um,
        config=config)
    if np.all(np.isnan(scores)):
        return [], scores, None
    best_z = int(np.nanargmax(scores))
    z_start = max(exclude_edge_z, best_z - window_radius)
    z_stop  = min(len(scores) - exclude_edge_z, best_z + window_radius + 1)
    n_z = len(scores)
    if config is not None:
        keep_z = [z for z in range(z_start, z_stop)
                  if _z_in_focus_range(z, n_z, config)]
    else:
        keep_z = list(range(z_start, z_stop))
    return keep_z, scores, best_z


def plot_focus_scores_for_timepoint(img_5d: np.ndarray, t: int,
                                    config: PipelineConfig) -> Tuple:
    """Quick QC plot of focus scores across Z for one time point."""
    keep_z, scores, best_z = get_best_focus_z_indices(
        img_5d, t,
        nuc_channel_idx=config.effective_focus_channel_index,
        exclude_edge_z=config.focus_edge_z_exclusion,
        window_radius=config.focus_window_radius,
        metric=config.focus_metric,
        pixel_size_um=config.pixel_size_um,
        config=config,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(scores)), scores, marker="o")
    if best_z is not None:
        ax.axvline(best_z, linestyle="--", label=f"best z={best_z}")
    for z in keep_z:
        ax.axvspan(z - 0.45, z + 0.45, alpha=0.2)
    ax.set_xlabel("Z index")
    ax.set_ylabel("Focus score")
    ax.set_title(f"Focus scores t={t}  keep_z={keep_z}  metric={config.focus_metric}")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return keep_z, scores, best_z

## 7a. Stage-2 gate — NPC shell + membrane co-localisation (v16.2)

Ported from `vulcan_training_1.1.ipynb` §4b, §5b, §6. The generator has always
applied this gate; the segmentation block never did.

`clean_nucleus_mask` tests **area, solidity and holes** — all properties of the
*mask*. A droplet cap passes every one: in a single Z-plane it is a small, round,
solid, filled disc. That asymmetry is the t = 0–3 contamination. The gate instead
tests **organisation, not presence**:

1. **Shell (primary)** — NPC puncta in an annulus straddling the nucleus edge
   (`dilate(nuc, 5) & ~erode(nuc, 5)`, threshold `mean + 2σ` of *raw* NPC over
   the droplet interior) must admit a coherent RANSAC circle: ≥ 8 puncta, ≥ 45 %
   inliers at 6 px tolerance. An envelope fits; scattered cap signal does not.
2. **Membrane (secondary)** — ≥ 30 % of puncta must have membrane signal
   (`mean + 1σ` over the droplet interior) within 6 px.

Both must fire, plus a **droplet-relative area window** (1 %–50 % of the parent
droplet) taken from the generator's `_clean_and_gate`.

### Read this before turning it on

- **The gate is NPC + Membrane, not NLS + NPC.** NLS enters upstream in the
  generator, as `detect_nucleus_adaptive`, the detector that *proposes* the
  nucleus; in v16 the U-Net's nucleus channel plays that role. The gate then asks
  the independent question "is there an organised envelope here?" — which is
  precisely what a cap fails. NLS is not part of the pass/fail decision.
- **The generator never gates t = 0–1** (`generation_min_timepoint = 2`): too few
  puncta for a shell fit, NE probe too faint to co-localise. Applying it there
  will reject nearly everything. `cfg.gate_min_timepoint = 2` passes those
  timepoints through ungated instead; `None` (the default) drops unverified
  objects rather than emitting them. **Decide this deliberately** — it changes
  whether t = 0–1 is empty or unfiltered, and both are defensible.
- **Raw intensities only.** Thresholds are `mean + kσ` of *raw* NPC / membrane
  over the droplet interior. A per-patch contrast stretch invalidates them, which
  is why the gate reads `img_5d` directly and never `_preprocess_patch` output.
  This connects to the `normalization_mode` note in the training config.
### The ported gate does not, on its own, reject caps

Measured after the port, on synthetic data. **A droplet cap with a completely
flat NPC channel passes the ported gate on 8/8 Gaussian-noise trials**
(57–79 "puncta", shell fit r ≈ 50 px, 100 % inliers, coloc = 1.00).

Three compounding reasons:

1. **The shell test is close to vacuous.** Its puncta zone is *already* a 10 px
   annulus (`dilate(nuc,5) & ~erode(nuc,5)`) and the RANSAC tolerance is 6 px, so
   every suprathreshold pixel in the zone is an inlier to a circle at the nucleus
   boundary **by construction**. The test effectively asks "are there ≥ 8
   suprathreshold pixels in the annulus?", which noise satisfies.
2. **No size filter on puncta**, so a single noisy pixel is a punctum and
   `gate_min_puncta = 8` is trivially met.
3. **`gate_membrane_k_std = 1.0` dilated by 6 px** covers essentially the whole
   crop, so `coloc_frac` saturates at 1.00.

This is not a porting error — it is the generator's behaviour, faithfully
reproduced. It is invisible there because the generator never *sees* a cap: its
nucleus proposals come from `detect_nucleus_adaptive` at the best-Z per droplet,
and caps are handled on a separate track by `mine_cap_negatives`. v16 runs the
U-Net on every Z, so caps arrive at the gate — a case the gate was never exposed
to. The section-6 markdown's claim that it "tests organisation, not presence" is
the intent, not the implementation.

**What actually separates them is contrast, not geometry**: ring/core mean raw
NPC is 1.46 for a real envelope and 1.00 for a flat cap. Two opt-in knobs,
default off for exact parity:

- `gate_min_ring_contrast` (try 1.15) — enforce that ratio. Always recorded as
  `npc_ring_contrast` in the ROI table, so you can pick the threshold from your
  own data before enabling it.
- `gate_puncta_min_size_px` (try 4–9) — require puncta to be discrete objects.

Also: **the generator computes `shell_r` and never tests it**, so a ring of
puncta on the *droplet wall* fits a circle and passes. `gate_shell_r_frac_min/max`
bound the shell radius as a fraction of the parent droplet radius (measured
r/R ≈ 0.53). Default `None` = parity.
- **Deterministic**: `fit_circle_ransac` seeds at 0, so re-running a plane gives
  the same verdict.
- `require_stage2_gate = False` reproduces v16.1 exactly, while still writing all
  gate metrics to the ROI table — so the A/B costs one run and no code change.

### Auditing

Every candidate's `n_puncta`, `shell_r_px`, `shell_inliers`, `coloc_frac`,
`nucleus_frac_of_droplet` and the three sub-verdicts land in the ROI table via
`regionprops_to_rows(extra_by_label=...)`. Thresholds can be re-tuned from that
table without re-running segmentation. `seg_index_df` gains `gate_candidates` and
`gate_verified` per plane, which is the fastest way to see whether the gate is
eating a timepoint whole.

**If you change any gate constant, change it in `vulcan_training_1.1.ipynb` too.**
Field names and defaults were copied deliberately so the two configs can be
diffed; if they drift, the model and the pipeline stop agreeing on what a nucleus
is, which is the failure this port exists to remove.

## 7. Segmentation helpers

In [10]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def normalize_channel(ch2d: np.ndarray) -> np.ndarray:
    """v9 training-faithful per-channel normalisation: p1-p99.8 clip -> [0,1]."""
    ch = ch2d.astype(np.float32)
    lo, hi = np.percentile(ch, [1.0, 99.8])
    if hi <= lo:
        return np.zeros_like(ch)
    return np.clip((ch - lo) / (hi - lo), 0.0, 1.0)


def _preprocess_patch(patch_yxc: np.ndarray) -> np.ndarray:
    """
    Training contract (v9 build_input_patch): per-CHANNEL p1-p99.8 normalisation.
    Replaces the old global x/x.max() which fed the model an input distribution
    it never trained on (one scalar over all 3 channels crushed the dim membrane
    channel). patch_yxc is (512,512,3) in [NLS, NPC, Membrane] order.

    Per-PATCH, matching how v9 normalised each 512 tile. To move to per-PLANE
    later: normalise the whole plane in get_full_plane_yxc and make this a
    passthrough -- this function is the single point of change.
    """
    x = patch_yxc.astype(np.float32)
    return np.stack([normalize_channel(x[..., c]) for c in range(x.shape[-1])], axis=-1)


def collapse_label_map(droplet: np.ndarray, npc: np.ndarray,
                       nucleus: np.ndarray, config: PipelineConfig) -> np.ndarray:
    """Single-int DISPLAY label map. Priority Nucleus>NPC>Droplet>BG (v9
    collapse_to_integer_hwc). Display/QC only -- never an extraction source."""
    out = np.zeros(droplet.shape, dtype=np.uint8)
    out[droplet] = config.droplet_class_index
    out[npc]     = config.npc_class_index
    out[nucleus] = config.nucleus_class_index   # last write wins = highest priority
    return out


def extract_class_masks(class_prob_map: np.ndarray, config: PipelineConfig):
    """
    v9 contract: 4 INDEPENDENT sigmoid channels, each thresholded on its OWN
    channel. No argmax -- argmax forces mutual exclusivity and suppresses the
    faint NPC ring everywhere nucleus/droplet also fire.

    Returns (background, droplet, npc, nucleus_raw, label_map):
      droplet     = prob[...,1] > droplet_threshold, unioned with nucleus for a
                    SOLID FILLED disk (nucleus footprint included, as droplet|=nucleus
                    trained it). Union avoids fill_holes (documented segfault risk).
      npc         = prob[...,2] > npc_threshold       (envelope ring)
      nucleus_raw = prob[...,3] > nucleus_threshold   (caller cleans separately)
      background  = NOT droplet
      label_map   = display-only collapse (Nucleus>NPC>Droplet>BG)
    """
    droplet = class_prob_map[..., config.droplet_class_index] > config.droplet_threshold
    npc     = class_prob_map[..., config.npc_class_index]     > config.npc_threshold
    nucleus = class_prob_map[..., config.nucleus_class_index] > config.nucleus_threshold

    droplet = droplet | nucleus          # solid filled; nucleus subset of droplet
    background = ~droplet

    label_map = collapse_label_map(droplet, npc, nucleus, config)
    return background, droplet, npc, nucleus, label_map


def _generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """Generate patch start positions that always include the image end."""
    starts = list(range(0, max(length - patch_size + 1, 1), stride)) or [0]
    last = max(0, length - patch_size)
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def extract_overlapping_patches(
    image_yxc: np.ndarray,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    h, w, _ = image_yxc.shape
    y_starts = _generate_patch_starts(h, patch_size, stride)
    x_starts = _generate_patch_starts(w, patch_size, stride)
    patches, coords = [], []
    for y0 in y_starts:
        for x0 in x_starts:
            patches.append(image_yxc[y0:y0 + patch_size, x0:x0 + patch_size, :])
            coords.append((y0, x0))
    return patches, coords


def _predict_batch(model, batch_patches: np.ndarray) -> np.ndarray:
    """
    Use model(x, training=False) instead of model.predict().
    model.predict() spins up a fresh TF dataset pipeline and progress-bar
    machinery on every call — at batch_size=1 this overhead dominates.
    Direct __call__ avoids all of that and runs the graph immediately.
    """
    preds = model(batch_patches, training=False).numpy()
    if preds.ndim != 4:
        raise ValueError(f"Unexpected prediction shape: {preds.shape}")
    if preds.shape[-1] == 1:
        fg = preds[..., 0]
        preds = np.stack([1.0 - fg, fg], axis=-1)
    return preds.astype(np.float32)


def stitch_probability_patches(
    patch_probs: List[np.ndarray],
    coords: List[Tuple[int, int]],
    image_shape: Tuple[int, int],
    n_classes: int,
    patch_size: int = 512,
) -> np.ndarray:
    """
    Average overlapping patch probabilities into a full probability map (H, W, C).

    Accepts either a list of 2-D arrays or the stacked (N, pH, pW, C) array
    returned directly from the inference loop.
    """
    h, w = image_shape
    prob_sum   = np.zeros((h, w, n_classes), dtype=np.float32)
    prob_count = np.zeros((h, w),            dtype=np.float32)

    # Accept pre-stacked array or list
    probs_arr = (np.asarray(patch_probs) if not isinstance(patch_probs, np.ndarray)
                 else patch_probs)

    for idx, (y0, x0) in enumerate(coords):
        y1, x1 = y0 + patch_size, x0 + patch_size
        prob_sum  [y0:y1, x0:x1] += probs_arr[idx]
        prob_count[y0:y1, x0:x1] += 1.0

    return prob_sum / np.maximum(prob_count[:, :, np.newaxis], 1.0)


def _filter_by_area_and_circularity(
    binary_mask: np.ndarray,
    min_area_px: int = 50,
    min_circularity: float = 0.45,
) -> np.ndarray:
    """
    Filter objects by area and shape compactness.

    Uses SOLIDITY (area / convex_hull_area) instead of perimeter-based
    circularity.  Solidity is computed from the convex hull — a pure geometric
    operation with no C-level binary erosion — so it cannot segfault on large
    or irregular objects.  Nuclei are compact (solidity > 0.7); noise fragments,
    droplet-wall arcs, and merged blobs are not.

    The min_circularity parameter is reused as the solidity threshold so no
    config changes are needed (valid range and typical values are similar).
    """
    labeled = measure.label(binary_mask, connectivity=2)
    if labeled.max() == 0:
        return np.zeros_like(binary_mask, dtype=bool)

    tbl = measure.regionprops_table(
        labeled, properties=("label", "area", "solidity"))

    areas     = np.array(tbl["area"],     dtype=np.float64)
    solidities = np.array(tbl["solidity"], dtype=np.float64)
    labels    = np.array(tbl["label"],    dtype=np.int32)

    keep_mask   = (areas >= min_area_px) & (solidities >= min_circularity)
    keep_labels = labels[keep_mask]

    out = np.isin(labeled, keep_labels)
    return out


def clean_nucleus_mask(
    nucleus_mask: np.ndarray,
    min_size_px: int = 50,
    hole_size_px: int = 200,
    opening_radius: int = 1,
    min_circularity: float = 0.45,
    max_foreground_fraction: float = 0.25,
) -> np.ndarray:
    """
    Clean a raw nucleus probability mask.

    Operation order is intentional:
      1. Foreground-fraction guard  — bail out immediately on over-fired planes.
         A plane where >25% of pixels are foreground is almost always a bad Z
         (model firing on droplet walls / texture).  Returning empty here costs
         7 ms vs up to 20 min of morphology + regionprops on a near-solid mask.
      2. remove_small_objects first — reduces object count before any fill_holes,
         so subsequent operations work on a sparser mask.
      3. opening before fill_holes — breaks merged blobs before filling gaps,
         preventing fill_holes from joining large connected regions.
      4. fill_holes on the reduced mask — much cheaper than on the raw mask.
      5. Perimeter-based circularity filter last — only computed on surviving
         objects after area filtering, not on every initial fragment.
    """
    mask = nucleus_mask.astype(bool)

    # ── Guard: bail on over-fired planes ─────────────────────────────────
    if max_foreground_fraction < 1.0:
        fg_frac = float(mask.mean())
        if fg_frac > max_foreground_fraction:
            return np.zeros_like(mask, dtype=bool)

    # ── Step 1: remove small fragments early ─────────────────────────────
    mask = morphology.remove_small_objects(mask, min_size=min_size_px)
    if not mask.any():
        return mask

    # ── Step 2: opening before fill to break merged blobs ────────────────
    if opening_radius > 0:
        mask = morphology.opening(mask, footprint=morphology.disk(opening_radius))
    if not mask.any():
        return mask

    # ── Step 3: per-object hole fill (safe on large/irregular masks) ─────
    # ndi.binary_fill_holes on the full image flood-fills the entire background
    # which segfaults on complex maze-like backgrounds at mid-stack Z planes.
    # Filling per bounding-box is equivalent but operates on small arrays only.
    labeled_for_fill = measure.label(mask, connectivity=2)
    filled = np.zeros_like(mask, dtype=bool)
    for prop in measure.regionprops(labeled_for_fill):
        r0, c0, r1, c1 = prop.bbox
        r0m = max(0, r0 - 1); c0m = max(0, c0 - 1)
        r1m = min(mask.shape[0], r1 + 1); c1m = min(mask.shape[1], c1 + 1)
        patch = mask[r0m:r1m, c0m:c1m]
        filled[r0m:r1m, c0m:c1m] |= ndi.binary_fill_holes(patch)
    mask = filled
    del labeled_for_fill, filled

    # ── Step 4: circularity filter (perimeter only on survivors) ─────────
    return _filter_by_area_and_circularity(mask, min_area_px=min_size_px,
                                           min_circularity=min_circularity)


def _run_batched_inference(
    patch_arr: np.ndarray,
    model,
    batch_size: int,
) -> np.ndarray:
    """
    Run model inference on a (N, H, W, C) patch array in eager mode.

    tf.function has been intentionally removed.  With a large U-Net on GPU,
    inference FLOPS dominate completely — tf.function saves ~1-5 ms per batch
    in Python dispatch overhead but costs 60-300 s whenever it retraces
    (which it does on any new input shape or after gc/memory events).
    Eager mode via model(x, training=False) is consistent and deadlock-free.

    The model should be warmed up once before the segmentation loop with
    warm_up_model() so the first real batch does not pay the GPU JIT cost.
    """
    n_patches = patch_arr.shape[0]
    all_preds: List[np.ndarray] = []

    for i in range(0, n_patches, batch_size):
        chunk = patch_arr[i:i + batch_size].astype(np.float32)
        preds = model(chunk, training=False).numpy().astype(np.float32)

        if preds.shape[-1] == 1:
            fg = preds[..., 0]
            preds = np.stack([1.0 - fg, fg], axis=-1)

        all_preds.append(preds)

    return np.concatenate(all_preds, axis=0)


def warm_up_model(model, patch_size: int, n_channels: int, batch_size: int) -> None:
    """
    Run one dummy forward pass to initialise GPU kernels before the main loop.

    Without this, the very first model(x) call of the segmentation run pays
    the CUDA JIT compilation cost for every layer — typically 5-15 s on a
    large U-Net.  Running this once here means all subsequent calls hit the
    warm kernel cache immediately.
    """
    dummy = np.zeros((batch_size, patch_size, patch_size, n_channels), dtype=np.float32)
    _ = model(dummy, training=False)
    del dummy
    print(f"  Model warmed up (batch={batch_size}, patch={patch_size}, channels={n_channels})")



# =====================================================================
# Stage-2 gate — NPC-shell organisation + membrane co-localisation
# PORTED FROM vulcan_training_1.1.ipynb (sections 4b, 5b, 6)      [v16.2]
# ---------------------------------------------------------------------
# The training-set generator has always applied this gate. The segmentation
# block never did — it accepted any component surviving clean_nucleus_mask,
# which tests area, solidity and holes: all properties of the MASK, every one
# of which a droplet cap passes trivially (in a single Z a cap is a small,
# round, solid, filled disc). That asymmetry is the t = 0-3 contamination.
#
# The gate tests ORGANISATION, NOT PRESENCE:
#   1. Shell     — NPC puncta in an annulus around the nucleus edge must admit
#                  a coherent RANSAC circle. An envelope fits; scattered cap or
#                  antibody-clump signal does not.
#   2. Membrane  — each punctum needs membrane signal within R, and a minimum
#                  fraction must be supported. True NPC sits on the NE, which
#                  the dim membrane probe also marks.
# Both must fire.
#
# The functions below are transcribed from the generator so the two can be
# diffed. ONE deliberate deviation, marked inline: _punctum_centroids reshapes
# to (-1, 2) so the empty case has a 2-D shape. Behaviourally identical.
#
# Normalisation regions are the DROPLET INTERIOR in every case, exactly as in
# the generator, which is why the adapter pairs each candidate with its parent
# droplet instance rather than using a local annulus.
# =====================================================================


def _circle_from_3(p1, p2, p3):
    ax, ay = p1; bx, by = p2; cx, cy = p3
    d = 2.0 * (ax*(by-cy) + bx*(cy-ay) + cx*(ay-by))
    if abs(d) < 1e-9:
        return None
    a2, b2, c2 = ax*ax+ay*ay, bx*bx+by*by, cx*cx+cy*cy
    ux = (a2*(by-cy) + b2*(cy-ay) + c2*(ay-by)) / d
    uy = (a2*(cx-bx) + b2*(ax-cx) + c2*(bx-ax)) / d
    return ux, uy, float(np.hypot(ux-ax, uy-ay))


def _fit_circle_kasa(pts):
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[x, y, np.ones(len(x))]
    b = x*x + y*y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0]/2.0, sol[1]/2.0
    r = np.sqrt(max(sol[2] + cx*cx + cy*cy, 0.0))
    return cx, cy, r


def fit_circle_ransac(pts, tol_px=4.0, n_iter=200, min_inlier_frac=0.5, rng=None):
    """
    RANSAC circle fit to Nx2 (x, y). Returns (cx, cy, r, inlier_mask) or None.
    Tolerant of missing arcs (touching neighbours) and stray points.

    rng defaults to seed 0, so the gate is DETERMINISTIC — re-running the same
    plane gives the same verdict. Do not pass a fresh rng per call.
    """
    rng = rng or np.random.default_rng(0)
    n = len(pts)
    if n < 3:
        return None
    best_inliers, best_circle = None, None
    for _ in range(n_iter):
        idx = rng.choice(n, 3, replace=False)
        circ = _circle_from_3(pts[idx[0]], pts[idx[1]], pts[idx[2]])
        if circ is None:
            continue
        cx, cy, r = circ
        resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
        inliers = resid < tol_px
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers, best_circle = inliers, circ
    if best_inliers is None or best_inliers.sum() < max(3, int(min_inlier_frac*n)):
        return None
    cx, cy, r = _fit_circle_kasa(pts[best_inliers])      # refit on inliers
    resid = np.abs(np.hypot(pts[:, 0]-cx, pts[:, 1]-cy) - r)
    return cx, cy, r, resid < tol_px


def _punctum_centroids(npc_puncta_mask: np.ndarray) -> np.ndarray:
    """Centroids (x, y) of connected NPC puncta in crop coordinates."""
    lbl = measure.label(npc_puncta_mask)
    return np.array([[r.centroid[1], r.centroid[0]]
                     for r in measure.regionprops(lbl)],
                    dtype=float).reshape(-1, 2)   # <- the one deviation


def detect_npc_puncta(npc_crop_raw, nucleus_mask_crop, droplet_mask_crop,
                      config: PipelineConfig) -> np.ndarray:
    """
    NPC puncta in an annular zone straddling the nucleus edge.

    npc_crop_raw MUST be the RAW NPC plane — not a normalised patch and not a
    probability map. The threshold is mean + k*std of raw NPC over the DROPLET
    interior, so any per-patch contrast stretch invalidates it.
    """
    if nucleus_mask_crop.sum() == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    npc = np.asarray(npc_crop_raw, dtype=np.float32)
    selem = morphology.disk(config.npc_margin_px)
    outer = morphology.binary_dilation(nucleus_mask_crop, selem)
    inner = morphology.binary_erosion(nucleus_mask_crop, selem)
    zone = outer & ~inner
    interior_vals = npc[droplet_mask_crop]
    if interior_vals.size == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    thresh = interior_vals.mean() + config.npc_std_mult * interior_vals.std()
    puncta = (npc > thresh) & zone
    if config.gate_puncta_min_size_px > 0:
        # NOT in the generator (default 0 = parity). Without it a single noisy
        # pixel counts as a punctum, so gate_min_puncta=8 is met by noise.
        puncta = morphology.remove_small_objects(
            puncta, min_size=int(config.gate_puncta_min_size_px))
    return puncta


def gate_npc_shell(npc_puncta_mask, config: PipelineConfig):
    """
    Shell-organisation test. Returns (passed: bool, info: dict).
    Puncta must admit a coherent RANSAC circle fit (envelope), not scatter.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"n_puncta": len(cents), "shell_r": np.nan, "shell_inliers": 0}
    if len(cents) < config.gate_min_puncta:
        return False, info
    fit = fit_circle_ransac(cents, tol_px=config.gate_shell_tol_px,
                            n_iter=config.ransac_n_iter,
                            min_inlier_frac=config.gate_shell_min_inlier_frac)
    if fit is None:
        return False, info
    cx, cy, r, inliers = fit
    info["shell_r"], info["shell_inliers"] = float(r), int(inliers.sum())
    passed = inliers.sum() >= max(3, int(config.gate_shell_min_inlier_frac * len(cents)))
    return passed, info


def gate_membrane_coloc(npc_puncta_mask, mem_crop_raw, droplet_mask_crop,
                        config: PipelineConfig):
    """
    Membrane co-localisation test. Returns (passed: bool, info: dict).
    Each NPC punctum must have membrane signal (mean + k*std, dim probe) within
    gate_membrane_nn_radius_px; a minimum fraction of puncta must be supported.
    """
    cents = _punctum_centroids(npc_puncta_mask)
    info = {"coloc_frac": 0.0}
    if len(cents) == 0:
        return False, info
    mem = np.asarray(mem_crop_raw, dtype=np.float32)
    interior = mem[droplet_mask_crop]
    if interior.size == 0:
        return False, info
    mem_thresh = interior.mean() + config.gate_membrane_k_std * interior.std()
    mem_present = mem > mem_thresh
    mem_dil = morphology.binary_dilation(
        mem_present, morphology.disk(config.gate_membrane_nn_radius_px))
    H, W = mem.shape
    supported = 0
    for x, y in cents:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < H and 0 <= xi < W and mem_dil[yi, xi]:
            supported += 1
    frac = supported / len(cents)
    info["coloc_frac"] = float(frac)
    return frac >= config.gate_membrane_min_coloc_frac, info


def gate_droplet(npc_puncta_mask, mem_crop_raw, droplet_mask_crop,
                 config: PipelineConfig):
    """
    Full Stage-2 gate: shell-organisation AND membrane co-localisation.
    Returns (passed: bool, info: dict). A candidate passes only if both fire.
    """
    shell_ok, shell_info = gate_npc_shell(npc_puncta_mask, config)
    mem_ok, mem_info = gate_membrane_coloc(npc_puncta_mask, mem_crop_raw,
                                           droplet_mask_crop, config)
    info = {**shell_info, **mem_info, "shell_ok": shell_ok, "membrane_ok": mem_ok}
    return (shell_ok and mem_ok), info


# ---------------------------------------------------------------------
# v16 adapter — per-candidate, parent-droplet-scoped
# ---------------------------------------------------------------------
def apply_stage2_gate_to_plane(
    nucleus_mask: np.ndarray,
    npc_plane_raw: Optional[np.ndarray],
    mem_plane_raw: Optional[np.ndarray],
    droplet_labels: Optional[np.ndarray],
    config: PipelineConfig,
    t_idx: Optional[int] = None,
) -> Tuple[np.ndarray, np.ndarray, Dict[int, dict], int, int]:
    """
    Run the generator's Stage-2 gate on every candidate in `nucleus_mask`.

    The generator gates per DROPLET; v16's unit is a nucleus candidate, so each
    candidate is paired with the droplet instance it sits in and that droplet's
    interior supplies the NPC and membrane thresholds. Since extract_class_masks
    does `droplet |= nucleus`, every candidate has a parent by construction.

    Parameters
    ----------
    npc_plane_raw, mem_plane_raw : RAW Ch2 / Ch0 for this (t, z). Not normalised,
        not probabilities — the thresholds are mean + k*std of raw intensity.
    droplet_labels : instance labels from measure.label(droplet_mask). Must be
        computed BEFORE this call.
    t_idx : used only for config.gate_min_timepoint.

    Returns
    -------
    (verified_mask, instance_labels, metrics_by_label, n_candidates, n_verified)

    `instance_labels` retains each survivor's ORIGINAL label, so metric keys stay
    valid and a rejected object leaves a gap in the numbering instead of silently
    renaming its neighbours. Labels are therefore non-contiguous; regionprops,
    napari and ImageJ all handle that.

    Inputs missing, gate disabled, or t below gate_min_timepoint -> everything
    passes through untouched, so the flag is a clean A/B switch.
    """
    mask_bool = nucleus_mask.astype(bool)
    if not mask_bool.any():
        return mask_bool, np.zeros(mask_bool.shape, np.uint16), {}, 0, 0

    lab = measure.label(mask_bool, connectivity=2)
    n_cand = int(lab.max())

    skip_t = (config.gate_min_timepoint is not None
              and t_idx is not None
              and t_idx < config.gate_min_timepoint)
    if (npc_plane_raw is None or mem_plane_raw is None
            or droplet_labels is None or skip_t):
        return mask_bool, lab.astype(np.uint16), {}, n_cand, n_cand

    npc = np.asarray(npc_plane_raw)
    mem = np.asarray(mem_plane_raw)
    H, W = lab.shape

    # ── Droplet bboxes: ONE vectorised pass, not one per candidate ────────
    # Doing `droplet_labels == parent` then regionprops inside the loop scans
    # the full 5732x3889 frame for every nucleus — thousands of full-frame
    # passes per timepoint. regionprops_table gets every droplet's bbox and
    # area in a single C-level pass.
    dtab = measure.regionprops_table(
        droplet_labels, properties=("label", "bbox", "area"))
    droplet_bbox = {
        int(l): (int(r0), int(c0), int(r1), int(c1), int(a))
        for l, r0, c0, r1, c1, a in zip(
            dtab["label"], dtab["bbox-0"], dtab["bbox-1"],
            dtab["bbox-2"], dtab["bbox-3"], dtab["area"])}

    metrics_by_label: Dict[int, dict] = {}
    keep_labels: List[int] = []

    for prop in measure.regionprops(lab):
        # ── parent droplet = the instance covering most of this candidate ──
        coords = prop.coords
        dvals = droplet_labels[coords[:, 0], coords[:, 1]]
        dvals = dvals[dvals > 0]
        if dvals.size and int(np.bincount(dvals).argmax()) in droplet_bbox:
            parent = int(np.bincount(dvals).argmax())
            r0, c0, r1, c1, droplet_area = droplet_bbox[parent]
            dmask_full = None          # crop is taken lazily below
        else:
            # no parent droplet — should not happen (droplet |= nucleus), but
            # fall back to a padded candidate bbox rather than crashing
            parent, droplet_area = 0, 0
            pad = int(config.npc_margin_px + config.gate_membrane_nn_radius_px + 4)
            r0, c0, r1, c1 = prop.bbox
            r0 = max(0, r0-pad); c0 = max(0, c0-pad)
            r1 = min(H, r1+pad); c1 = min(W, c1+pad)

        sl = (slice(r0, r1), slice(c0, c1))
        nuc_crop = lab[sl] == prop.label
        # Droplet mask cropped FIRST, compared second — the comparison touches
        # only the droplet bbox, never the full frame.
        dm_crop = (droplet_labels[sl] == parent if parent
                   else np.ones((r1-r0, c1-c0), bool))

        # ── generator's Stage-2 gate, unmodified ──────────────────────────
        npc_puncta = detect_npc_puncta(npc[sl], nuc_crop, dm_crop, config)
        passed, info = gate_droplet(npc_puncta, mem[sl], dm_crop, config)

        # ── ring/core raw-NPC contrast (NOT in the generator) ─────────────
        # The measurement that actually separates a real envelope from a flat
        # cap: 1.46 vs 1.00 on the synthetic check. Always recorded; only
        # enforced when gate_min_ring_contrast is set.
        sel = morphology.disk(config.npc_margin_px)
        zone = (morphology.binary_dilation(nuc_crop, sel)
                & ~morphology.binary_erosion(nuc_crop, sel))
        core = morphology.binary_erosion(nuc_crop, sel)
        if not core.any():
            core = nuc_crop
        npc_c = np.asarray(npc[sl], dtype=np.float32)
        ring_mean = float(npc_c[zone].mean()) if zone.any() else np.nan
        core_mean = float(npc_c[core].mean()) if core.any() else np.nan
        ring_contrast = (ring_mean / core_mean
                         if np.isfinite(ring_mean) and np.isfinite(core_mean)
                         and core_mean > 0 else np.nan)
        contrast_ok = True
        if config.gate_min_ring_contrast is not None:
            contrast_ok = (np.isfinite(ring_contrast)
                           and ring_contrast >= config.gate_min_ring_contrast)

        # ── droplet-relative area window (generator's _clean_and_gate) ────
        frac = (float(prop.area) / droplet_area) if droplet_area else np.nan
        frac_ok = True
        if config.gate_use_droplet_area_fraction and droplet_area:
            frac_ok = (config.nucleus_min_frac <= frac <= config.nucleus_max_frac)

        # ── optional shell-radius sanity check (NOT in the generator) ─────
        radius_ok = True
        r_eq_droplet = np.sqrt(droplet_area / np.pi) if droplet_area else np.nan
        shell_frac = (info["shell_r"] / r_eq_droplet
                      if np.isfinite(info.get("shell_r", np.nan))
                      and np.isfinite(r_eq_droplet) and r_eq_droplet > 0
                      else np.nan)
        if np.isfinite(shell_frac):
            if (config.gate_shell_r_frac_min is not None
                    and shell_frac < config.gate_shell_r_frac_min):
                radius_ok = False
            if (config.gate_shell_r_frac_max is not None
                    and shell_frac > config.gate_shell_r_frac_max):
                radius_ok = False

        m = {
            "parent_droplet_label":  parent,
            "droplet_area_px":       droplet_area,
            "nucleus_frac_of_droplet": frac,
            "n_puncta":       int(info["n_puncta"]),
            "shell_r_px":     float(info["shell_r"]),
            "shell_inliers":  int(info["shell_inliers"]),
            "shell_r_frac_of_droplet": float(shell_frac),
            "coloc_frac":     float(info["coloc_frac"]),
            "npc_ring_contrast": float(ring_contrast),
            "shell_ok":       bool(info["shell_ok"]),
            "membrane_ok":    bool(info["membrane_ok"]),
            "area_frac_ok":   bool(frac_ok),
            "shell_radius_ok": bool(radius_ok),
            "ring_contrast_ok": bool(contrast_ok),
        }
        m["stage2_verified"] = bool(passed and frac_ok and radius_ok and contrast_ok)
        metrics_by_label[int(prop.label)] = m
        if m["stage2_verified"]:
            keep_labels.append(int(prop.label))

    if not config.require_stage2_gate:
        return mask_bool, lab.astype(np.uint16), metrics_by_label, n_cand, n_cand

    if keep_labels:
        instance_labels = np.where(np.isin(lab, keep_labels), lab, 0).astype(np.uint16)
    else:
        instance_labels = np.zeros(lab.shape, np.uint16)
    return (instance_labels > 0, instance_labels, metrics_by_label,
            n_cand, len(keep_labels))


def segment_all_planes_for_timepoint(
    img_5d: np.ndarray,
    t_idx: int,
    keep_z: List[int],
    model,
    config: PipelineConfig,
) -> Dict[int, Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Segment all keep_z planes for one timepoint, one plane at a time.

    Memory architecture
    -------------------
    The previous "mega-batch" design collected ALL patches from ALL planes
    into one tensor before running inference.  This caused peak RAM usage of
    5-20 GB for a 3-plane window on a large FOV (all_patch_arr + all_probs_arr
    + results dict all live simultaneously), which killed the kernel at t=3
    where mid-stack planes have much higher foreground density.

    This version processes one plane at a time and explicitly deletes the
    patch array before allocating the output probability array, keeping peak
    RAM to ~1-4 GB regardless of how many keep_z planes there are.  The GPU
    stays warm between planes because the model is called repeatedly without
    any Python-level sleep or teardown between calls.

    Returns
    -------
    Dict mapping z_idx → (class_prob_map, class_label_map, nucleus_mask, nucleus_labels)
    """
    if not keep_z:
        return {}

    patch_size = config.patch_size
    stride     = config.patch_stride
    Y, X       = img_5d.shape[-2], img_5d.shape[-1]
    batch_size = max(1, int(config.batch_size))

    results: Dict[int, Tuple] = {}

    for z in keep_z:
        # ── Extract and preprocess patches for this plane ─────────────────
        plane_yxc = get_full_plane_yxc(img_5d, t_idx, z, config=config)
        patches, coords = extract_overlapping_patches(plane_yxc, patch_size, stride)
        patch_arr = np.stack([_preprocess_patch(p) for p in patches], axis=0)
        del plane_yxc, patches   # free input before allocating output

        # ── Inference ────────────────────────────────────────────────────
        probs_arr = _run_batched_inference(patch_arr, model, batch_size)
        del patch_arr            # free patches immediately after inference

        # ── Stitch and post-process ───────────────────────────────────────
        n_classes = probs_arr.shape[-1]
        class_prob_map = stitch_probability_patches(
            probs_arr, coords, (Y, X), n_classes, patch_size)
        del probs_arr            # free patch probs after stitching

        # ── v9 per-channel extraction (no argmax) ─────────────────────────
        _, droplet_mask, npc_mask, nucleus_raw, _ = extract_class_masks(
            class_prob_map, config)
        nucleus_mask = clean_nucleus_mask(
            nucleus_raw,
            min_size_px=config.min_nucleus_area_px,
            hole_size_px=config.nucleus_hole_size_px,
            opening_radius=config.nucleus_opening_radius_px,
            min_circularity=config.min_nucleus_circularity,
            max_foreground_fraction=config.max_nucleus_foreground_fraction,
        )
        # ── v16.2 Stage-2 gate (NPC shell + membrane co-localisation) ────
        droplet_labels = measure.label(droplet_mask, connectivity=2).astype(np.uint16)
        nucleus_mask, nucleus_labels, _gm, _n0, _n1 = apply_stage2_gate_to_plane(
            nucleus_mask,
            img_5d[t_idx, z, config.npc_channel_index],
            img_5d[t_idx, z, config.membrane_channel_index],
            droplet_labels, config, t_idx=t_idx)
        class_label_map = collapse_label_map(droplet_mask, npc_mask, nucleus_mask, config)
        results[z] = (class_prob_map, class_label_map, nucleus_mask, nucleus_labels)

    return results


def segment_single_plane_with_overlap(
    plane_yxc: np.ndarray,
    model,
    config: PipelineConfig,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Single-plane wrapper — used by the debug cell (Section 26) only."""
    patches, coords = extract_overlapping_patches(plane_yxc, patch_size, stride)
    patch_arr = np.stack([_preprocess_patch(p) for p in patches], axis=0)
    probs_arr = _run_batched_inference(patch_arr, model, batch_size=max(1, int(config.batch_size)))
    n_classes = probs_arr.shape[-1]
    h, w, _ = plane_yxc.shape
    class_prob_map = stitch_probability_patches(probs_arr, coords, (h, w), n_classes, patch_size)
    _, droplet_mask, npc_mask, nucleus_raw, _ = extract_class_masks(class_prob_map, config)
    nucleus_mask = clean_nucleus_mask(
        nucleus_raw,
        min_size_px=config.min_nucleus_area_px,
        hole_size_px=config.nucleus_hole_size_px,
        opening_radius=config.nucleus_opening_radius_px,
        min_circularity=config.min_nucleus_circularity,
        max_foreground_fraction=config.max_nucleus_foreground_fraction,
    )
    # ── v16.2 Stage-2 gate ───────────────────────────────────────────────
    # plane_yxc is (Y, X, C) in TRAINING order [NLS, NPC, Membrane] — see
    # get_full_plane_yxc — so NPC is index 1 and Membrane index 2 HERE, which
    # is not the hyperstack's channel order. Values are raw (normalisation
    # happens in _preprocess_patch, on a copy), which the gate requires.
    droplet_labels = measure.label(droplet_mask, connectivity=2).astype(np.uint16)
    nucleus_mask, nucleus_labels, gate_metrics, n_cand, n_verified = (
        apply_stage2_gate_to_plane(
            nucleus_mask, plane_yxc[..., 1], plane_yxc[..., 2],
            droplet_labels, config))
    print(f"  Stage-2 gate: {n_verified}/{n_cand} candidates verified"
          f"{'' if config.require_stage2_gate else '  (ADVISORY — gate disabled)'}")
    class_label_map = collapse_label_map(droplet_mask, npc_mask, nucleus_mask, config)
    return class_prob_map, class_label_map, nucleus_mask, nucleus_labels


## 8. TIFF save helpers

In [11]:
def _write_hyperstack_tiff(arr: np.ndarray, out_path: Path, axes: str,
                           prefer_imagej: bool = True) -> None:
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(arr)
    # BigTIFF for arrays >= ~2 GB to avoid ImageJ truncation.
    use_imagej = prefer_imagej and arr.nbytes < (2 * 1024**3 - 32 * 1024**2)
    tiff.imwrite(out_path, arr,
                 imagej=use_imagej, bigtiff=not use_imagej,
                 metadata={"axes": axes})


def save_probability_hyperstack_tiff(probability_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(probability_5d, out_path, axes="TZCYX", prefer_imagej=False)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff((class_mask_5d.astype(np.uint8) * 255), out_path,
                           axes="TZCYX", prefer_imagej=True)


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(label_4d.astype(np.uint8), out_path,
                           axes="TZYX", prefer_imagej=True)


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(instance_4d, out_path, axes="TZYX", prefer_imagej=True)


def create_probability_memmap(
    shape: Tuple[int, int, int, int, int],
    out_dir: Path,
    dtype: np.dtype = np.float16,
) -> Tuple[np.memmap, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_path = out_dir / "probability_hyperstack_temp.dat"
    if temp_path.exists():
        temp_path.unlink()
    return np.memmap(temp_path, mode="w+", dtype=dtype, shape=shape), temp_path



def open_hyperstack_memmap(out_path: Path,
                           shape: Tuple[int, ...],
                           dtype,
                           axes: str) -> np.memmap:
    """Open a hyperstack TIFF for in-place plane writes, creating a
    zero-filled one of ``shape`` if it does not already exist.

    Backs the resumable segmentation path: planes are written straight into
    the final TIFF through the returned memmap, so an interrupted run keeps
    every finished plane on disk and a restart only fills the missing
    timepoints. No ``.dat`` scratch copy is used.

    An existing file whose shape/dtype does not match (an older run with
    different dims, or a non-memmappable ImageJ TIFF) is discarded and
    recreated.
    """
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dtype = np.dtype(dtype)
    shape = tuple(int(s) for s in shape)

    if out_path.exists():
        try:
            mm = tiff.memmap(str(out_path), mode="r+")
            if tuple(mm.shape) == shape and np.dtype(mm.dtype) == dtype:
                return mm
            del mm
        except Exception:
            pass
        out_path.unlink()

    return tiff.memmap(str(out_path), shape=shape, dtype=dtype,
                       bigtiff=True, metadata={"axes": axes})


In [12]:
# ============================================================================
# 8b. Segmentation checkpoint — single source of truth for stage state
# ============================================================================
# `seg_dir/_checkpoint.json` records what segmentation has produced and whether
# every artifact a downstream cell needs is still on disk. It supersedes the old
# `_resume/manifest.json` (the per-timepoint `_resume/t{NNN}.pkl` sidecars stay).
#
# The full-run vs resume vs already-complete decision is made by
# `assess_segmentation_state()` BEFORE the model is loaded and BEFORE the focus
# warm-up, so a finished (or half-finished) run costs nothing to detect.
#
#   status "in_progress"  some timepoints done, run not finalised
#   status "complete"     all timepoints done, seg_index + roi_table written,
#                         every artifact fingerprinted
#
# A "complete" checkpoint whose artifacts no longer verify -> mode "inconsistent"
# -> hard fail with recovery text (no implicit GPU work).

_SEG_DTYPES = {"class": "uint8", "label": "uint8",
               "nucleus": "uint16", "droplet": "uint16", "prob": "float16"}


@dataclass
class SegmentationState:
    mode: str                       # "complete" | "resume" | "fresh" | "inconsistent"
    completed_t: List[int]
    todo_t: List[int]
    checkpoint: Optional[dict]
    missing_artifacts: List[str]
    reason: str


def _seg_artifact_targets(config, save_probability_tiff: bool) -> Dict[str, tuple]:
    """artifact key -> ('file', Path) | ('npy_dir', Path). The set of files a
    completed segmentation must leave on disk for the downstream cells."""
    t = {
        "class_hyperstack":            ("file", config.segmentation_class_hyperstack_path),
        "label_hyperstack":            ("file", config.segmentation_label_hyperstack_path),
        "nucleus_instance_hyperstack": ("file", config.nucleus_instance_hyperstack_path),
        "droplet_instance_hyperstack": ("file", config.droplet_instance_hyperstack_path),
        "seg_index":                   ("file", config.segmentation_index_path),
        "roi_table":                   ("file", config.segmentation_roi_table_path),
        "npy_planes":                  ("npy_dir", config.seg_dir),
    }
    if save_probability_tiff:
        t["probability_hyperstack"] = ("file", config.segmentation_probability_hyperstack_path)
    return t


def _verify_seg_artifacts(cp: dict, needs=None) -> List[str]:
    """Return the artifact keys that are missing or changed since the checkpoint
    was written. `needs` limits the check to a subset of keys."""
    arts = cp.get("artifacts", {})
    keys = list(needs) if needs else list(arts.keys())
    missing: List[str] = []
    for k in keys:
        ent = arts.get(k)
        if not ent:
            missing.append(k); continue
        if "dir" in ent:                                   # npy plane directory
            d = Path(ent["dir"])
            n = len(list(d.glob("nuclear_mask_t*_z*.npy"))) if d.is_dir() else 0
            if n < int(ent.get("count", 0)):
                missing.append(k)
        else:                                              # single file
            p = Path(ent["path"])
            if not p.exists():
                missing.append(k)
            elif ent.get("bytes") is not None and p.stat().st_size != ent["bytes"]:
                missing.append(k)
    return missing


def _mask_stores_ok(config, T: int, Z: int, Y: int, X: int) -> tuple:
    """Are the 4 mask hyperstacks present and memmappable at the expected shape?
    A resume cannot write into a deleted store."""
    checks = [
        (config.segmentation_class_hyperstack_path, (T, Z, config.num_classes, Y, X)),
        (config.segmentation_label_hyperstack_path, (T, Z, Y, X)),
        (config.nucleus_instance_hyperstack_path,   (T, Z, Y, X)),
        (config.droplet_instance_hyperstack_path,   (T, Z, Y, X)),
    ]
    for p, want in checks:
        p = Path(p)
        if not p.exists():
            return False, f"{p.name} missing"
        try:
            mm = tiff.memmap(str(p), mode="r")
            shp = tuple(mm.shape); del mm
        except Exception as exc:
            return False, f"{p.name} not memmappable ({exc})"
        if shp != want:
            return False, f"{p.name} shape {shp} != {want}"
    return True, "ok"


def _recovery_hint(config) -> str:
    prob = Path(config.segmentation_probability_hyperstack_path)
    if prob.exists():
        return ("Recovery: a probability hyperstack is present — set "
                "cfg.probability_hyperstack_input to it and re-run §27 to rebuild the "
                "masks with no GPU; or run reset_segmentation(cfg, scope='all') and "
                "re-run §27 for a full rebuild.")
    return ("Recovery: run reset_segmentation(cfg, scope='all') in the §27 reset cell, "
            "then re-run §27 — no probability hyperstack remains, so this re-runs full "
            "GPU inference for every timepoint.")


def _inconsistent_reason(cp: dict, missing: List[str], config) -> str:
    lines = ["Segmentation checkpoint says status=complete, but these artifacts are "
             "missing or changed on disk:"]
    for k in missing:
        ent = cp.get("artifacts", {}).get(k, {})
        lines.append(f"    - {k}: {ent.get('path', ent.get('dir', '?'))}")
    lines.append(f"  checkpoint finalised: {cp.get('finalized_at', '?')}")
    lines.append(_recovery_hint(config))
    return "\n".join(lines)


def assess_segmentation_state(config, dims_tzyx: tuple,
                              save_probability_tiff: bool = True) -> SegmentationState:
    """Decide fresh / resume / complete / inconsistent from on-disk state only.
    No image load beyond the dims tuple, no TF, no GPU, no focus scoring."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    all_t = list(range(T))
    cp_path = config.seg_dir / "_checkpoint.json"
    resume_dir = config.seg_dir / "_resume"
    legacy_manifest = resume_dir / "manifest.json"

    cp = None
    if cp_path.exists():
        try:
            cp = json.loads(cp_path.read_text())
        except Exception:
            cp = None

    if cp is None and legacy_manifest.exists():          # migrate old resume state
        try:
            _m = json.loads(legacy_manifest.read_text())
            cp = {"schema": 1, "stage": "segmentation", "status": "in_progress",
                  "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
                  "completed_t": sorted(int(t) for t in _m.get("completed_t", [])),
                  "artifacts": {}, "note": "migrated from legacy _resume/manifest.json"}
        except Exception:
            cp = None

    if cp is None:
        return SegmentationState("fresh", [], all_t, None, [],
            f"No segmentation checkpoint ({cp_path.name}) — fresh run, "
            f"all {T} timepoints to segment.")

    want = {"T": T, "Z": Z, "Y": Y, "X": X}
    cp_dims = cp.get("dims", {})
    if any(cp_dims.get(k) != v for k, v in want.items()):
        return SegmentationState("fresh", [], all_t, cp, [],
            f"Checkpoint dims {cp_dims} != current {want} — stale checkpoint ignored, "
            f"re-running all {T} timepoints.")

    completed = sorted(int(t) for t in cp.get("completed_t", []) if 0 <= int(t) < T)
    status = cp.get("status")

    if status == "complete":
        missing = _verify_seg_artifacts(cp)
        if not missing:
            return SegmentationState("complete", all_t, [], cp, [],
                f"Segmentation checkpoint complete and all {len(cp.get('artifacts', {}))} "
                f"artifacts verified (finalised {cp.get('finalized_at', '?')}). "
                f"Loading saved tables — no model load, no focus scoring.")
        return SegmentationState("inconsistent", completed, [], cp, missing,
            _inconsistent_reason(cp, missing, config))

    # status == "in_progress" (or a partial checkpoint with completed_t)
    ok, msg = _mask_stores_ok(config, T, Z, Y, X)
    if completed and not ok:
        return SegmentationState("inconsistent", completed, [], cp, ["mask_hyperstacks"],
            f"Checkpoint says timepoints {completed} were segmented, but the mask "
            f"hyperstacks are gone or the wrong shape ({msg}). Cannot resume into a "
            f"deleted store.\n" + _recovery_hint(config))

    survived = [t for t in completed if (resume_dir / f"t{t:03d}.pkl").exists()]
    lost = sorted(set(completed) - set(survived))
    todo = [t for t in all_t if t not in survived]

    if not todo:
        return SegmentationState("resume", survived, [], cp, [],
            f"Checkpoint lists all {T} timepoints done but was never finalised "
            f"(status={status!r}). Re-running finalisation only.")

    reason = (f"Resuming segmentation: {len(survived)}/{T} timepoints already done "
              f"(t={survived}); {len(todo)} to segment (t={todo}).")
    if lost:
        reason += f"\n  sidecars missing for t={lost} — those will be re-segmented."
    return SegmentationState("resume", survived, todo, cp, [], reason)


def write_segmentation_checkpoint(config, *, status: str, dims_tzyx: tuple,
                                  dtypes: dict, save_probability_tiff: bool,
                                  completed_t) -> dict:
    """Write `seg_dir/_checkpoint.json` atomically, fingerprinting every artifact
    that exists right now (st_size for files, plane count for the npy dir)."""
    T, Z, Y, X = (int(v) for v in dims_tzyx)
    cp_path = config.seg_dir / "_checkpoint.json"
    prev = {}
    if cp_path.exists():
        try:
            prev = json.loads(cp_path.read_text())
        except Exception:
            prev = {}

    artifacts = {}
    for key, (kind, p) in _seg_artifact_targets(config, save_probability_tiff).items():
        p = Path(p)
        if kind == "npy_dir":
            artifacts[key] = {"dir": str(p),
                              "count": len(list(p.glob("nuclear_mask_t*_z*.npy")))}
        else:
            artifacts[key] = {"path": str(p),
                              "bytes": p.stat().st_size if p.exists() else None}

    now = datetime.now().isoformat(timespec="seconds")
    cp = {
        "schema": 1,
        "stage": "segmentation",
        "run_id": getattr(getattr(config, "paths", None), "run_id", None),
        "code_version": "v18.1",
        "status": status,
        "dims": {"T": T, "Z": Z, "Y": Y, "X": X},
        "dtypes": dtypes,
        "save_probability_tiff": bool(save_probability_tiff),
        "completed_t": sorted(int(t) for t in completed_t),
        "artifacts": artifacts,
        "started_at": prev.get("started_at") or now,
        "updated_at": now,
    }
    if status == "complete":
        cp["finalized_at"] = now

    tmp = cp_path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(cp, indent=2, default=str))
    os.replace(tmp, cp_path)
    return cp


def require_segmentation(config, needs=()) -> dict:
    """Guard for downstream cells. Raise unless the segmentation checkpoint is
    'complete' and every artifact key in `needs` still verifies on disk."""
    cp_path = config.seg_dir / "_checkpoint.json"
    if not cp_path.exists():
        raise RuntimeError(
            f"{cp_path} not found — segmentation (§27) has not completed in this run. "
            f"Run §27 first.")
    cp = json.loads(cp_path.read_text())
    if cp.get("status") != "complete":
        raise RuntimeError(
            f"Segmentation checkpoint status={cp.get('status')!r}, not 'complete'. "
            f"Re-run §27 to finish it.")
    missing = _verify_seg_artifacts(cp, needs=needs or None)
    if missing:
        raise RuntimeError(_inconsistent_reason(cp, missing, config))
    return cp


def reset_segmentation(config, scope: str) -> None:
    """Wipe segmentation state so the next §27 run rebuilds.

    scope="resume"  delete only the checkpoint + _resume/ (keep the TIFFs).
                    Forces a full recompute; use after editing a seg function.
    scope="all"     also delete the 5 hyperstack TIFFs, the per-plane .npy
                    masks, the seg index / ROI pickles, AND the stale downstream
                    object pickles. Nothing half-deleted.
    """
    if scope not in ("resume", "all"):
        raise ValueError("scope must be 'resume' or 'all'")
    removed: List[str] = []

    def _rm(p):
        p = Path(p)
        if p.is_dir():
            shutil.rmtree(p); removed.append(str(p) + "/")
        elif p.exists():
            p.unlink(); removed.append(str(p))

    _rm(config.seg_dir / "_checkpoint.json")
    _rm(config.seg_dir / "_checkpoint.json.tmp")
    _rm(config.seg_dir / "_resume")
    _rm(config.seg_dir / "segmentation_checkpoint.json")          # legacy

    if scope == "all":
        for p in (config.segmentation_class_hyperstack_path,
                  config.segmentation_label_hyperstack_path,
                  config.nucleus_instance_hyperstack_path,
                  config.droplet_instance_hyperstack_path,
                  config.segmentation_probability_hyperstack_path,
                  config.segmentation_index_path,
                  config.segmentation_roi_table_path):
            _rm(p)
        for name in ("class_mask_5d.dat", "class_label_4d.dat",
                     "nucleus_instance_4d.dat", "droplet_instance_4d.dat"):
            _rm(config.mask_tif_dir / name)
        _npy = list(config.seg_dir.glob("nuclear_mask_t*_z*.npy"))
        for p in _npy:
            p.unlink()
        if _npy:
            removed.append(f"{config.seg_dir}/nuclear_mask_t*_z*.npy  ({len(_npy)} files)")
        for name in ("plane_objects.pkl", "grouped_z_objects.pkl",
                     "best_z_nuclei.pkl", "best_z_nuclei_repaired.pkl",
                     "best_z_nuclei_with_exclusion.pkl"):
            _rm(config.obj_dir / name)
        _rm(config.obj_dir / "repaired_masks")

    print(f"reset_segmentation(scope={scope!r}) removed {len(removed)} path(s):")
    for r in removed:
        print("   ", r)
    if not removed:
        print("    (nothing to remove)")
    if scope == "all":
        print("Re-run §27 to rebuild. Stale downstream object pickles were also cleared.")

## 9. Region-props helper

In [13]:
def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
    min_area_px: int = 0,
    extra_by_label: Optional[Dict[int, Dict]] = None,
) -> List[Dict]:
    """
    Convert a labelled mask to a list of row dicts.

    Uses regionprops_table for a single vectorised C-level pass over all
    objects.  Perimeter is NOT computed here — it requires binary_erosion
    on every object which is catastrophically slow when the model produces
    large merged regions (e.g. droplet lattice at mid-stack Z planes).
    Circularity filtering happens earlier in clean_nucleus_mask; by the time
    we reach this function it is not needed again.

    extra_by_label (v16.2) merges per-object measurements keyed by label. Used
    to carry the Stage-2 gate metrics into the ROI table so a run can be audited
    — and thresholds re-tuned — without re-running segmentation.
    """
    if labeled_mask.max() == 0:
        return []

    tbl = measure.regionprops_table(
        labeled_mask,
        properties=("label", "area", "centroid", "bbox"),
    )

    labels   = tbl["label"]
    areas    = tbl["area"]
    cy_arr   = tbl["centroid-0"]
    cx_arr   = tbl["centroid-1"]
    rmin     = tbl["bbox-0"]
    cmin     = tbl["bbox-1"]
    rmax     = tbl["bbox-2"]
    cmax     = tbl["bbox-3"]

    rows = []
    for i in range(len(labels)):
        if areas[i] < min_area_px:
            continue
        row = {
            "t": t_idx, "z": z_idx,
            "class_id": class_id, "class_name": class_name,
            "label":          int(labels[i]),
            "centroid_x_px":  float(cx_arr[i]),
            "centroid_y_px":  float(cy_arr[i]),
            "area_px":        int(areas[i]),
            "bbox_min_row":   int(rmin[i]),
            "bbox_min_col":   int(cmin[i]),
            "bbox_max_row":   int(rmax[i]),
            "bbox_max_col":   int(cmax[i]),
        }
        if extra_by_label:
            row.update(extra_by_label.get(int(labels[i]), {}))
        rows.append(row)
    return rows

## 10. Full segmentation loop

In [14]:
def run_segmentation_for_all_planes(
    img_5d: np.ndarray,
    model,
    config: PipelineConfig,
    state: "SegmentationState" = None,
    save_binary_masks: bool = True,
    save_probability_tiff: bool = True,
) -> pd.DataFrame:
    """
    Run U-Net segmentation over the full hyperstack.

    Memory architecture
    -------------------
    WRITE-AS-YOU-GO: each output plane is written straight into its final
    memory-mapped hyperstack TIFF right after inference and the local arrays
    are deleted. Peak extra RAM per plane is ~one plane, not the full stack.

    Checkpoint / resume
    -------------------
    The five hyperstack TIFFs ARE the persistent store, opened in place
    (``tiff.memmap`` mode "r+"). After every timepoint a sidecar
    ``<seg_dir>/_resume/t{NNN}.pkl`` holds that timepoint's index + ROI rows and
    ``<seg_dir>/_checkpoint.json`` is refreshed (status "in_progress"). On
    finalisation the seg index / ROI table are written and the checkpoint flips
    to status "complete" with a size fingerprint for every artifact.

    ``state`` (a SegmentationState from ``assess_segmentation_state``) decides
    which timepoints to run: ``state.completed_t`` are reloaded from their
    sidecars and skipped, ``state.todo_t`` are (re)inferred. The focus warm-up
    scores only ``todo`` timepoints. If ``state`` is None a full fresh run over
    every timepoint is assumed.

    For a clean rebuild use ``reset_segmentation(config, scope=...)``.
    """
    T, Z = img_5d.shape[0], img_5d.shape[1]
    Y, X = img_5d.shape[-2], img_5d.shape[-1]
    C = config.num_classes

    if state is None:
        state = SegmentationState(
            mode="fresh", completed_t=[], todo_t=list(range(T)),
            checkpoint=None, missing_artifacts=[], reason="(no state passed)")

    completed_t: set = set(int(t) for t in state.completed_t)
    todo = sorted(set(range(T)) - completed_t)

    records:  List[dict] = []
    roi_rows: List[dict] = []

    # -- Open the five output hyperstacks in place (create if absent) --------
    mm_class = open_hyperstack_memmap(config.segmentation_class_hyperstack_path,
                                      (T, Z, C, Y, X), np.uint8,  "TZCYX")
    mm_label = open_hyperstack_memmap(config.segmentation_label_hyperstack_path,
                                      (T, Z, Y, X),    np.uint8,  "TZYX")
    mm_nuc   = open_hyperstack_memmap(config.nucleus_instance_hyperstack_path,
                                      (T, Z, Y, X),    np.uint16, "TZYX")
    mm_drop  = open_hyperstack_memmap(config.droplet_instance_hyperstack_path,
                                      (T, Z, Y, X),    np.uint16, "TZYX")

    probability_5d = None
    if save_probability_tiff:
        probability_5d = open_hyperstack_memmap(
            config.segmentation_probability_hyperstack_path,
            (T, Z, C, Y, X), np.float16, "TZCYX")

    resume_dir = config.seg_dir / "_resume"

    def _flush_all():
        for mm in (mm_class, mm_label, mm_nuc, mm_drop):
            mm.flush()
        if probability_5d is not None:
            probability_5d.flush()

    def _persist_timepoint(t_idx, rec_start, roi_start, completed):
        """Flush the TIFF planes, write this timepoint's sidecar, refresh the
        checkpoint. TIFF-first ordering keeps a crash idempotent: a timepoint
        not in the checkpoint's completed_t is simply re-run."""
        _flush_all()
        resume_dir.mkdir(parents=True, exist_ok=True)
        pd.to_pickle({"records":  records[rec_start:],
                      "roi_rows": roi_rows[roi_start:]},
                     resume_dir / f"t{t_idx:03d}.pkl")
        completed.add(t_idx)
        write_segmentation_checkpoint(
            config, status="in_progress", dims_tzyx=(T, Z, Y, X),
            dtypes=_SEG_DTYPES, save_probability_tiff=save_probability_tiff,
            completed_t=sorted(completed))

    try:
        # -- Reload finished timepoints from their sidecars -----------------
        for t_done in sorted(completed_t):
            _sc = resume_dir / f"t{t_done:03d}.pkl"
            if not _sc.exists():
                raise RuntimeError(
                    f"resume sidecar {_sc} vanished mid-run — run "
                    f"reset_segmentation(cfg, scope='all') and re-run §27.")
            _d = pd.read_pickle(_sc)
            records.extend(_d["records"])
            roi_rows.extend(_d["roi_rows"])
        if completed_t:
            print(f"Resuming: {len(completed_t)}/{T} timepoints reloaded from "
                  f"sidecars (t={sorted(completed_t)}); {len(todo)} to segment.")

        if not todo:
            print("No timepoints to segment — running finalisation only.")

        # -- Pre-compute focus scores for the TODO timepoints only ---------
        focus_data: Dict[int, Tuple[List[int], np.ndarray, Optional[int], str]] = {}
        if todo:
            print("Pre-computing focus scores (v13 adaptive) for "
                  f"t={todo} ...")
            for t_pre in todo:
                if config.use_focus_z_selection:
                    kz, fs, bz, ch = get_best_focus_z_indices_adaptive(
                        img_5d=img_5d, t=t_pre, config=config)
                    focus_data[t_pre] = (kz, fs, bz, ch)
                    print(f"  t={t_pre}: channel={ch!r}  best_z={bz}  keep_z={kz}")
                else:
                    focus_data[t_pre] = (list(range(Z)),
                                         np.full(Z, np.nan, dtype=np.float32),
                                         None, 'disabled')
            print("Focus scoring complete. Starting GPU segmentation...\n")

            # Warm up GPU kernels once before the main loop
            n_img_channels = img_5d.shape[2]
            warm_up_model(model, config.patch_size, n_img_channels,
                          max(1, int(config.batch_size)))

        error_log_path = config.seg_dir / "segmentation_errors.txt"

        # -- Segmentation loop --------------------------------------------
        for t_idx in range(T):
            if t_idx in completed_t:
                import sys
                print(f"t={t_idx}: resume skip (already segmented)"); sys.stdout.flush()
                continue

            rec_start = len(records)
            roi_start = len(roi_rows)

            keep_z, focus_scores, best_z, focus_channel = focus_data[t_idx]
            keep_z_set = set(keep_z)

            import resource, sys
            rss_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
            gpu_info = ""
            if tf is not None:
                try:
                    gpus = tf.config.list_physical_devices("GPU")
                    if gpus:
                        mem = tf.config.experimental.get_memory_info("GPU:0")
                        gpu_info = (f"  GPU current={mem['current']/1e9:.1f}GB "
                                    f"peak={mem['peak']/1e9:.1f}GB")
                except Exception:
                    pass
            print(f"t={t_idx}: channel={focus_channel!r}  best_z={best_z}  "
                  f"segmenting z={keep_z}  [RSS={rss_gb:.1f}GB{gpu_info}]")
            sys.stdout.flush()
            t_start = time.perf_counter()

            # -- Record skipped planes (no inference needed) --------------
            for z_idx in range(Z):
                if z_idx in keep_z_set:
                    continue
                fscore = (float(focus_scores[z_idx])
                          if z_idx < len(focus_scores)
                          and np.isfinite(focus_scores[z_idx]) else np.nan)
                out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
                reason = ("edge_z_excluded"
                          if z_idx < config.focus_edge_z_exclusion
                             or z_idx >= Z - config.focus_edge_z_exclusion
                          else "outside_best_focus_window"
                          if config.use_focus_z_selection else "skipped")
                if save_binary_masks:
                    np.save(out_path, np.zeros((Y, X), dtype=np.uint8))
                records.append({"t": t_idx, "z": z_idx, "included": False,
                                "reason": reason, "focus_score": fscore,
                                "best_focus_z": best_z, "focus_channel": focus_channel,
                                "mask_path": str(out_path)})

            patch_size = config.patch_size
            stride     = config.patch_stride
            batch_size = max(1, int(config.batch_size))

            for z_idx in keep_z:
                fscore = (float(focus_scores[z_idx])
                          if z_idx < len(focus_scores)
                          and np.isfinite(focus_scores[z_idx]) else np.nan)
                out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
                import sys

                try:
                    plane_yxc = get_full_plane_yxc(img_5d, t_idx, z_idx, config=config)
                    patches, coords = extract_overlapping_patches(
                        plane_yxc, patch_size, stride)
                    patch_arr = np.stack(
                        [_preprocess_patch(p) for p in patches], axis=0)
                    del plane_yxc, patches

                    probs_arr = _run_batched_inference(patch_arr, model, batch_size)
                    del patch_arr

                    n_classes = probs_arr.shape[-1]
                    class_prob_map = stitch_probability_patches(
                        probs_arr, coords,
                        (img_5d.shape[-2], img_5d.shape[-1]),
                        n_classes, patch_size)
                    del probs_arr

                    background_mask, droplet_mask, npc_mask, nucleus_raw, _ = \
                        extract_class_masks(class_prob_map, config)
                    nucleus_mask = clean_nucleus_mask(
                        nucleus_raw,
                        min_size_px=config.min_nucleus_area_px,
                        hole_size_px=config.nucleus_hole_size_px,
                        opening_radius=config.nucleus_opening_radius_px,
                        min_circularity=config.min_nucleus_circularity,
                        max_foreground_fraction=config.max_nucleus_foreground_fraction,
                    )
                    droplet_instances = measure.label(
                        droplet_mask, connectivity=2).astype(np.uint16)

                    (nucleus_mask, nucleus_instance_labels, gate_metrics,
                     n_cand, n_verified) = apply_stage2_gate_to_plane(
                        nucleus_mask,
                        img_5d[t_idx, z_idx, config.npc_channel_index],
                        img_5d[t_idx, z_idx, config.membrane_channel_index],
                        droplet_instances, config, t_idx=t_idx)

                    class_label_map = collapse_label_map(
                        droplet_mask, npc_mask, nucleus_mask, config)

                except Exception as exc:
                    import traceback
                    msg = (f"\n[ERROR] t={t_idx} z={z_idx} — "
                           f"{type(exc).__name__}: {exc}\n"
                           + traceback.format_exc())
                    print(msg); sys.stdout.flush()
                    with open(error_log_path, "a") as ef:
                        ef.write(msg)
                    Y, X = img_5d.shape[-2], img_5d.shape[-1]
                    class_prob_map        = np.zeros((Y, X, config.num_classes), np.float32)
                    class_label_map       = np.zeros((Y, X),     np.uint8)
                    nucleus_mask          = np.zeros((Y, X),     bool)
                    nucleus_raw           = np.zeros((Y, X),     bool)
                    nucleus_instance_labels = np.zeros((Y, X),   np.uint16)
                    background_mask       = np.ones((Y, X),      bool)
                    droplet_mask          = np.zeros((Y, X),     bool)
                    droplet_instances     = np.zeros((Y, X),     np.uint16)
                    npc_mask              = np.zeros((Y, X),     bool)
                    gate_metrics          = {}
                    n_cand = n_verified   = 0

                if save_probability_tiff and probability_5d is not None:
                    probability_5d[t_idx, z_idx] = np.moveaxis(
                        class_prob_map.astype(np.float16), -1, 0)

                mm_class[t_idx, z_idx, 0] = background_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 1] = droplet_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 2] = npc_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 3] = nucleus_mask.astype(np.uint8)
                mm_label[t_idx, z_idx]    = class_label_map
                mm_nuc  [t_idx, z_idx]    = nucleus_instance_labels
                mm_drop [t_idx, z_idx]    = droplet_instances

                roi_rows.extend(regionprops_to_rows(
                    nucleus_instance_labels, t_idx, z_idx,
                    config.nucleus_class_index, "nucleus",
                    extra_by_label=gate_metrics))
                roi_rows.extend(regionprops_to_rows(
                    droplet_instances, t_idx, z_idx,
                    config.droplet_class_index, "droplet"))

                if save_binary_masks:
                    np.save(out_path, nucleus_mask.astype(np.uint8))

                records.append({
                    "t": t_idx, "z": z_idx, "included": True,
                    "reason": "segmented", "focus_score": fscore,
                    "best_focus_z": best_z, "focus_channel": focus_channel,
                    "mask_path": str(out_path),
                    "nucleus_pixels": int(nucleus_mask.sum()),
                    "droplet_pixels": int(droplet_mask.sum()),
                    "gate_candidates": n_cand,
                    "gate_verified":   n_verified,
                })
                gate_note = (f"  gate={n_verified}/{n_cand}"
                             if config.require_stage2_gate
                             else f"  gate=OFF ({n_cand} cand)")
                print(f"    z={z_idx}: nuc_px={int(nucleus_mask.sum()):,}  "
                      f"fg_frac={float(nucleus_mask.mean()):.3f}{gate_note}")
                sys.stdout.flush()

                del class_prob_map, class_label_map, nucleus_mask, nucleus_raw
                del nucleus_instance_labels, background_mask, gate_metrics
                del droplet_mask, droplet_instances, npc_mask

            gc.collect()
            elapsed = time.perf_counter() - t_start

            _persist_timepoint(t_idx, rec_start, roi_start, completed_t)

            import sys
            print(f"  t={t_idx} complete in {elapsed:.1f}s"); sys.stdout.flush()

        # -- Guard: refuse to finalise an empty run --------------------
        if not records:
            raise RuntimeError(
                "Segmentation produced no plane records — refusing to write an "
                f"empty {config.segmentation_index_path.name}. Run "
                f"reset_segmentation(cfg, scope='all') and re-run §27.")

        print("Flushing hyperstack TIFFs...")
        _flush_all()

        # -- Finalise: integrity-check the index BEFORE overwriting ------
        seg_index_df = (pd.DataFrame(records)
                        .sort_values(["t", "z"]).reset_index(drop=True))
        _dups = int(seg_index_df.duplicated(["t", "z"]).sum())
        if _dups:
            raise RuntimeError(
                f"segmentation index has {_dups} duplicate (t,z) rows — refusing "
                f"to overwrite {config.segmentation_index_path.name}. Inspect "
                f"{resume_dir}.")
        if len(seg_index_df) != T * Z:
            raise RuntimeError(
                f"segmentation index has {len(seg_index_df)} rows, expected "
                f"T*Z={T * Z} — refusing to overwrite "
                f"{config.segmentation_index_path.name}. A timepoint sidecar in "
                f"{resume_dir} is probably missing; run "
                f"reset_segmentation(cfg, scope='all') and re-run §27.")
        seg_index_df.to_pickle(config.segmentation_index_path)

        roi_df = pd.DataFrame(roi_rows)
        roi_df.to_pickle(config.segmentation_roi_table_path)

        write_segmentation_checkpoint(
            config, status="complete", dims_tzyx=(T, Z, Y, X),
            dtypes=_SEG_DTYPES, save_probability_tiff=save_probability_tiff,
            completed_t=list(range(T)))

        print(f"Segmentation complete. {len(seg_index_df)} plane records, "
              f"{len(roi_df)} ROI rows. Checkpoint status=complete.")
        return seg_index_df

    finally:
        for mm in (mm_class, mm_label, mm_nuc, mm_drop):
            try:
                mm.flush(); del mm
            except Exception:
                pass
        if isinstance(probability_5d, np.memmap):
            try:
                probability_5d.flush()
            except Exception:
                pass

## 10b. Re-extract from a saved probability hyperstack (layer 3)

`run_reextraction_from_probability` regenerates the class / label / instance
hyperstacks and the ROI table **from an existing probability hyperstack**, with
no model and no inference. Set `cfg.probability_hyperstack_input` to any saved
probability map and the run cell (Section 27) takes this path automatically.

Use it to sweep `droplet_threshold` / `npc_threshold` / `nucleus_threshold`
across datasets in seconds instead of re-running the ~15-min inference. The
extraction contract is identical to the inference path — both call
`extract_class_masks`, so masks are guaranteed consistent.

In [15]:
def run_reextraction_from_probability(
    config: PipelineConfig,
    save_binary_masks: bool = True,
) -> pd.DataFrame:
    """
    Re-extract masks from a saved probability hyperstack (no model / no inference).

    Reads config.probability_input_path (TZCYX, class-index channel order, written
    by run_segmentation_for_all_planes) plane-by-plane via memmap, applies the
    shared v9 extraction contract (extract_class_masks -> clean -> collapse), and
    writes the class / label / instance hyperstacks + ROI table + segmentation
    index exactly as the inference path does. The probability hyperstack itself is
    the INPUT and is not rewritten.

    v16.2: the Stage-2 gate needs the RAW NPC and Membrane channels, which a
    probability hyperstack does not contain. The raw stack at
    config.input_image_path is opened as a memmap and only the planes actually
    touched are read, so this costs no extra RAM. If the raw stack is missing
    (re-extracting a probability map copied from elsewhere) this RAISES unless
    require_stage2_gate is False — failing loudly is deliberate, because
    silently skipping the gate would produce a run whose snapshotted config
    claims it was verified when it was not.
    """
    prob_path = config.probability_input_path
    if prob_path is None or not Path(prob_path).exists():
        raise FileNotFoundError(f"probability hyperstack not found: {prob_path}")

    prob_mm = tiff.memmap(str(prob_path))          # (T, Z, C, Y, X)
    T, Z, C, Y, X = prob_mm.shape
    if C != config.num_classes:
        raise ValueError(f"probability map has {C} classes, config.num_classes={config.num_classes}")
    print(f"Re-extraction source: {prob_path}  shape={prob_mm.shape} dtype={prob_mm.dtype}")

    # ── Raw hyperstack for the Stage-2 gate (memmap; lazy, per plane) ─────
    raw_mm = None
    try:
        raw_mm = tiff.memmap(str(config.input_image_path), mode="r")
        if raw_mm.shape[0] != T or raw_mm.shape[1] != Z:
            raise ValueError(
                f"raw stack {raw_mm.shape[:2]} does not match the probability "
                f"map (T={T}, Z={Z}) — these are not the same acquisition")
        print(f"Raw stack for Stage-2 gate: {config.input_image_path}  "
              f"shape={raw_mm.shape}")
    except Exception as exc:
        if config.require_stage2_gate:
            raise RuntimeError(
                f"The Stage-2 gate requires the raw hyperstack at "
                f"{config.input_image_path}, which could not be opened: {exc}\n"
                f"Either place the raw stack there, or set "
                f"cfg.require_stage2_gate = False to re-extract without it."
            ) from exc
        print(f"[warn] raw stack unavailable ({exc}); gate metrics not recorded.")
        raw_mm = None

    records:  List[dict] = []
    roi_rows: List[dict] = []

    def _open_memmap(path: Path, dtype, shape):
        path.parent.mkdir(parents=True, exist_ok=True)
        return np.memmap(path, dtype=dtype, mode="w+", shape=shape)

    mm_class = _open_memmap(config.mask_tif_dir / "class_mask_5d.dat",
                            np.uint8,  (T, Z, config.num_classes, Y, X))
    mm_label = _open_memmap(config.mask_tif_dir / "class_label_4d.dat",
                            np.uint8,  (T, Z, Y, X))
    mm_nuc   = _open_memmap(config.mask_tif_dir / "nucleus_instance_4d.dat",
                            np.uint16, (T, Z, Y, X))
    mm_drop  = _open_memmap(config.mask_tif_dir / "droplet_instance_4d.dat",
                            np.uint16, (T, Z, Y, X))
    try:
        import sys
        for t_idx in range(T):
            t_start = time.perf_counter()
            for z_idx in range(Z):
                # (C, Y, X) -> (Y, X, C) to match extract_class_masks channel access
                class_prob_map = np.moveaxis(
                    np.asarray(prob_mm[t_idx, z_idx], dtype=np.float32), 0, -1)

                if float(class_prob_map.max()) == 0.0:
                    # plane was never segmented (outside focus window) -> empty
                    mm_class[t_idx, z_idx] = 0
                    mm_label[t_idx, z_idx] = 0
                    mm_nuc  [t_idx, z_idx] = 0
                    mm_drop [t_idx, z_idx] = 0
                    records.append({"t": t_idx, "z": z_idx, "included": False,
                                    "reason": "empty_in_probability_map",
                                    "focus_score": np.nan, "best_focus_z": None,
                                    "focus_channel": "reextract", "mask_path": ""})
                    continue

                # ── shared v9 extraction contract ─────────────────────────
                background_mask, droplet_mask, npc_mask, nucleus_raw, _ = \
                    extract_class_masks(class_prob_map, config)
                nucleus_mask = clean_nucleus_mask(
                    nucleus_raw,
                    min_size_px=config.min_nucleus_area_px,
                    hole_size_px=config.nucleus_hole_size_px,
                    opening_radius=config.nucleus_opening_radius_px,
                    min_circularity=config.min_nucleus_circularity,
                    max_foreground_fraction=config.max_nucleus_foreground_fraction,
                )
                droplet_instances = measure.label(
                    droplet_mask, connectivity=2).astype(np.uint16)

                # ── v16.2 Stage-2 gate (identical call to the inference path) ─
                npc_raw = (raw_mm[t_idx, z_idx, config.npc_channel_index]
                           if raw_mm is not None else None)
                mem_raw = (raw_mm[t_idx, z_idx, config.membrane_channel_index]
                           if raw_mm is not None else None)
                (nucleus_mask, nucleus_instance_labels, gate_metrics,
                 n_cand, n_verified) = apply_stage2_gate_to_plane(
                    nucleus_mask, npc_raw, mem_raw, droplet_instances,
                    config, t_idx=t_idx)

                class_label_map = collapse_label_map(
                    droplet_mask, npc_mask, nucleus_mask, config)

                mm_class[t_idx, z_idx, 0] = background_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 1] = droplet_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 2] = npc_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 3] = nucleus_mask.astype(np.uint8)
                mm_label[t_idx, z_idx]    = class_label_map
                mm_nuc  [t_idx, z_idx]    = nucleus_instance_labels
                mm_drop [t_idx, z_idx]    = droplet_instances

                roi_rows.extend(regionprops_to_rows(
                    nucleus_instance_labels, t_idx, z_idx,
                    config.nucleus_class_index, "nucleus",
                    extra_by_label=gate_metrics))
                roi_rows.extend(regionprops_to_rows(
                    droplet_instances, t_idx, z_idx,
                    config.droplet_class_index, "droplet"))

                if save_binary_masks:
                    out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
                    np.save(out_path, nucleus_mask.astype(np.uint8))
                else:
                    out_path = ""
                records.append({"t": t_idx, "z": z_idx, "included": True,
                                "reason": "reextracted", "focus_score": np.nan,
                                "best_focus_z": None, "focus_channel": "reextract",
                                "mask_path": str(out_path),
                                "nucleus_pixels": int(nucleus_mask.sum()),
                                "droplet_pixels": int(droplet_mask.sum()),
                                "gate_candidates": n_cand,
                                "gate_verified":   n_verified})

                del class_prob_map, class_label_map, nucleus_mask, nucleus_raw
                del nucleus_instance_labels, background_mask, gate_metrics
                del droplet_mask, droplet_instances, npc_mask, npc_raw, mem_raw
            gc.collect()
            print(f"  t={t_idx} re-extracted in {time.perf_counter()-t_start:.1f}s")
            sys.stdout.flush()

        if not records:
            raise RuntimeError(
                "Re-extraction produced no plane records — refusing to "
                f"overwrite {config.segmentation_index_path} and the "
                "hyperstack TIFFs."
            )
        print("Flushing memmaps to hyperstack TIFFs...")
        for mm in (mm_class, mm_label, mm_nuc, mm_drop):
            mm.flush()
        _write_hyperstack_tiff(mm_class, config.segmentation_class_hyperstack_path,
                               axes="TZCYX", prefer_imagej=False)
        _write_hyperstack_tiff(mm_label, config.segmentation_label_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)
        _write_hyperstack_tiff(mm_nuc,   config.nucleus_instance_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)
        _write_hyperstack_tiff(mm_drop,  config.droplet_instance_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)

        roi_df = pd.DataFrame(roi_rows)
        roi_df.to_pickle(config.segmentation_roi_table_path)
        seg_index_df = pd.DataFrame(records)
        seg_index_df.to_pickle(config.segmentation_index_path)
        write_segmentation_checkpoint(
            config, status="complete", dims_tzyx=(T, Z, Y, X),
            dtypes=_SEG_DTYPES, save_probability_tiff=False,
            completed_t=list(range(T)))
        print(f"Re-extraction complete. {len(seg_index_df)} plane records, "
              f"{len(roi_df)} ROI rows.")
        return seg_index_df
    finally:
        for mm, name in [(mm_class, "class_mask_5d.dat"), (mm_label, "class_label_4d.dat"),
                         (mm_nuc, "nucleus_instance_4d.dat"), (mm_drop, "droplet_instance_4d.dat")]:
            try:
                mm.flush(); del mm
            except Exception:
                pass
        for dat_name in ["class_mask_5d.dat", "class_label_4d.dat",
                         "nucleus_instance_4d.dat", "droplet_instance_4d.dat"]:
            dat_path = config.mask_tif_dir / dat_name
            if dat_path.exists():
                try:
                    dat_path.unlink()
                except Exception:
                    pass

## 11. Object extraction from masks

In [16]:
def extract_objects_from_saved_masks(
    seg_index_df: pd.DataFrame,
    config: PipelineConfig,
) -> pd.DataFrame:
    """
    Load saved binary mask files and extract region properties.

    Rows with included=False are skipped — they hold empty masks and
    contribute no objects.
    """
    included = seg_index_df[seg_index_df["included"].astype(bool)]
    all_rows: List[dict] = []
    for row in included.itertuples(index=False):
        mask = np.load(row.mask_path).astype(bool)
        rows = regionprops_to_rows(
            measure.label(mask, connectivity=2),
            t_idx=row.t, z_idx=row.z,
            class_id=config.nucleus_class_index, class_name="nucleus",
            min_area_px=config.min_nucleus_area_px,
        )
        all_rows.extend(rows)

    objects_df = pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
    objects_df.to_pickle(config.obj_dir / "plane_objects.pkl")
    return objects_df


## 12. Distance-based helpers

In [17]:
def nearest_neighbor_matches(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_dist_px: float,
) -> List[Tuple[int, int, float]]:
    if source_df.empty or target_df.empty:
        return []
    src_xy = source_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tgt_xy = target_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tree = cKDTree(tgt_xy)
    dists, idxs = tree.query(src_xy, distance_upper_bound=max_dist_px)

    matches, used = [], set()
    for i in np.argsort(dists):
        dist, j = dists[i], idxs[i]
        if np.isinf(dist) or j in used:
            continue
        matches.append((int(i), int(j), float(dist)))
        used.add(int(j))
    return matches


## 13. Group nuclei across Z

In [18]:
def group_nuclei_across_z(objects_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if objects_df.empty:
        return pd.DataFrame()

    grouped_rows = []
    next_group_id = 1

    for t_idx, df_t in objects_df.groupby("t", sort=True):
        df_t = df_t.sort_values(["z", "label"]).reset_index(drop=True).copy()
        df_t["nucleus_3d_id"] = -1
        z_values = sorted(df_t["z"].unique())

        for idx in df_t.index[df_t["z"] == z_values[0]]:
            df_t.at[idx, "nucleus_3d_id"] = next_group_id
            next_group_id += 1

        for z_prev, z_curr in zip(z_values[:-1], z_values[1:]):
            prev_df = df_t[df_t["z"] == z_prev].reset_index()
            curr_df = df_t[df_t["z"] == z_curr].reset_index()
            matched_curr = set()

            for i_prev, i_curr, _ in nearest_neighbor_matches(
                    prev_df, curr_df, config.z_group_tolerance_px):
                prev_global = int(prev_df.loc[i_prev, "index"])
                curr_global = int(curr_df.loc[i_curr, "index"])
                matched_curr.add(curr_global)

                grp = df_t.at[prev_global, "nucleus_3d_id"]
                if grp == -1:
                    grp = next_group_id
                    df_t.at[prev_global, "nucleus_3d_id"] = grp
                    next_group_id += 1
                df_t.at[curr_global, "nucleus_3d_id"] = grp

            for curr_global in curr_df["index"].tolist():
                if curr_global not in matched_curr and df_t.at[curr_global, "nucleus_3d_id"] == -1:
                    df_t.at[curr_global, "nucleus_3d_id"] = next_group_id
                    next_group_id += 1

        grouped_rows.append(df_t)

    grouped_z_df = pd.concat(grouped_rows, ignore_index=True)
    grouped_z_df.to_pickle(config.obj_dir / "grouped_z_objects.pkl")
    return grouped_z_df


## 14. Select best Z per nucleus

In [19]:
def select_best_z_per_nucleus(grouped_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if grouped_z_df.empty:
        return pd.DataFrame()
    best_z_df = (
        grouped_z_df
        .sort_values(["t", "nucleus_3d_id", "area_px"], ascending=[True, True, False])
        .groupby(["t", "nucleus_3d_id"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    best_z_df.to_pickle(config.obj_dir / "best_z_nuclei.pkl")
    return best_z_df


## 15. Exclude likely multi-nucleus droplets

In [20]:
def flag_close_nuclei(best_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if best_z_df.empty:
        return pd.DataFrame()
    out = []
    for _, df_t in best_z_df.groupby("t", sort=True):
        df_t = df_t.copy().reset_index(drop=True)
        coords = df_t[["centroid_x_px", "centroid_y_px"]].to_numpy()
        tree = cKDTree(coords)
        close_flags = np.zeros(len(df_t), dtype=bool)
        for i, xy in enumerate(coords):
            neighbors = [j for j in tree.query_ball_point(xy, r=config.multi_nucleus_exclusion_px) if j != i]
            if neighbors:
                close_flags[i] = True
                for j in neighbors:
                    close_flags[j] = True
        df_t["valid_single_nucleus"] = ~close_flags
        out.append(df_t)
    filtered_df = pd.concat(out, ignore_index=True)
    filtered_df.to_pickle(config.obj_dir / "best_z_nuclei_with_exclusion.pkl")
    return filtered_df


## 16. Tile assignment and true acquisition time

In [21]:
def assign_tile_index_from_xy(
    x_px: float, y_px: float,
    image_width_px: int, image_height_px: int,
    config: PipelineConfig,
) -> Tuple[int, int, int]:
    col = min(int(x_px // (image_width_px  / config.tile_cols)), config.tile_cols - 1)
    row = min(int(y_px // (image_height_px / config.tile_rows)), config.tile_rows - 1)
    if config.serpentine_scan and row % 2 != 0:
        tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col
    return row, col, tile_index


def add_tile_timing_metadata(
    nuclei_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if nuclei_df.empty:
        return pd.DataFrame()
    image_height_px, image_width_px = img_5d.shape[-2], img_5d.shape[-1]
    out_rows = []
    tiles_per_frame = config.tile_rows * config.tile_cols * config.minutes_per_tile
    for row in nuclei_df.itertuples(index=False):
        tile_row, tile_col, tile_index = assign_tile_index_from_xy(
            row.centroid_x_px, row.centroid_y_px,
            image_width_px, image_height_px, config)
        d = row._asdict()
        d["tile_row"]        = tile_row
        d["tile_col"]        = tile_col
        d["tile_index"]      = tile_index
        d["tile_offset_min"] = tile_index * config.minutes_per_tile
        d["true_time_min"]   = row.t * tiles_per_frame + d["tile_offset_min"]
        out_rows.append(d)
    timed_df = pd.DataFrame(out_rows)
    timed_df.to_pickle(config.track_dir / "best_z_nuclei_timed.pkl")
    return timed_df


## 17. Track nuclei across time — hybrid DBSCAN tracker

1. Build local spatial neighbourhoods with DBSCAN from consecutive frames.
2. Score candidate links by distance + area consistency.
3. One-to-one Hungarian assignment inside each local neighbourhood.
4. Reject crowded or low-confidence links rather than forcing a track.


In [22]:
def _safe_area_ratio(area_a: float, area_b: float) -> float:
    a, b = max(float(area_a), 1.0), max(float(area_b), 1.0)
    return max(a, b) / min(a, b)


def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)
    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        for col in ("dbscan_cluster_id", "dbscan_cluster_size", "dbscan_is_crowded"):
            pooled[col] = pd.Series(dtype=int if col != "dbscan_is_crowded" else bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(eps=config.track_dbscan_eps_px,
                    min_samples=config.track_dbscan_min_samples).fit_predict(coords)
    pooled["dbscan_cluster_id"] = labels

    crowded_flags = []
    cluster_sizes = []
    for _, df_c in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_c["frame_role"] == "prev").sum())
        n_curr = int((df_c["frame_role"] == "curr").sum())
        is_crowded = n_prev > 1 or n_curr > 1
        crowded_flags.extend([is_crowded] * len(df_c))
        cluster_sizes.extend([len(df_c)] * len(df_c))
    pooled["dbscan_cluster_size"] = cluster_sizes
    pooled["dbscan_is_crowded"]   = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[dict]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    n_prev, n_curr = len(prev_cluster), len(curr_cluster)
    cost_matrix = np.full((n_prev, n_curr), np.nan, dtype=float)
    max_dist_base = float(config.time_track_tolerance_px)

    for i, prev_row in prev_cluster.iterrows():
        for j, curr_row in curr_cluster.iterrows():
            dx  = float(curr_row["centroid_x_px"]) - float(prev_row["centroid_x_px"])
            dy  = float(curr_row["centroid_y_px"]) - float(prev_row["centroid_y_px"])
            dist_px = math.hypot(dx, dy)

            max_dist = max_dist_base
            if prev_row.get("dbscan_is_crowded", False) or curr_row.get("dbscan_is_crowded", False):
                max_dist *= config.track_crowded_distance_scale

            prev_area = max(float(prev_row["area_px"]), 1.0)
            curr_area = max(float(curr_row["area_px"]), 1.0)
            area_ratio = max(prev_area, curr_area) / min(prev_area, curr_area)

            if dist_px <= max_dist and area_ratio <= config.track_area_ratio_max:
                area_cost    = abs(math.log(curr_area / prev_area))
                dist_cost    = dist_px / max(max_dist, 1e-6)
                cost_matrix[i, j] = dist_cost + config.track_area_log_weight * area_cost

    finite_mask = np.isfinite(cost_matrix)
    valid_rows  = np.where(finite_mask.any(axis=1))[0]
    valid_cols  = np.where(finite_mask.any(axis=0))[0]
    if len(valid_rows) == 0 or len(valid_cols) == 0:
        return []

    large_penalty = 1e9
    reduced = np.where(np.isfinite(cost_matrix[np.ix_(valid_rows, valid_cols)]),
                       cost_matrix[np.ix_(valid_rows, valid_cols)], large_penalty)

    bad_rows = np.all(reduced >= large_penalty, axis=1)
    bad_cols = np.all(reduced >= large_penalty, axis=0)
    if bad_rows.any() or bad_cols.any():
        keep_r = np.where(~bad_rows)[0]
        keep_c = np.where(~bad_cols)[0]
        if len(keep_r) == 0 or len(keep_c) == 0:
            return []
        reduced    = reduced   [np.ix_(keep_r, keep_c)]
        valid_rows = valid_rows[keep_r]
        valid_cols = valid_cols[keep_c]

    row_ind, col_ind = linear_sum_assignment(reduced)

    matches = []
    for r, c in zip(row_ind, col_ind):
        orig_i = valid_rows[r]
        orig_j = valid_cols[c]
        link_cost = cost_matrix[orig_i, orig_j]
        if not np.isfinite(link_cost):
            continue
        pr = prev_cluster.iloc[orig_i]
        cr = curr_cluster.iloc[orig_j]
        dist_px = math.hypot(
            float(cr["centroid_x_px"]) - float(pr["centroid_x_px"]),
            float(cr["centroid_y_px"]) - float(pr["centroid_y_px"]),
        )
        matches.append({
            "prev_index":        int(pr["global_index"]),
            "curr_index":        int(cr["global_index"]),
            "link_cost":         float(link_cost),
            "link_distance_px":  float(dist_px),
            "dbscan_cluster_id":   int(pr.get("dbscan_cluster_id", -1)),
            "dbscan_cluster_size": int(pr.get("dbscan_cluster_size", 0)),
            "dbscan_is_crowded":   bool(pr.get("dbscan_is_crowded", False)),
        })
    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy().reset_index(drop=True)
    df["global_index"] = np.arange(len(df), dtype=int)
    df = df.sort_values(["t", "nucleus_3d_id"]).reset_index(drop=True)

    df["track_id"]                = -1
    df["track_link_distance_px"]  = np.nan
    df["track_link_cost"]         = np.nan
    df["track_dbscan_cluster_id"] = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"] = False
    df["track_link_method"]       = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    for idx in df.index[df["t"] == time_values[0]]:
        df.at[idx, "track_id"]          = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(
            prev_df, curr_df, config)

        all_matches: List[dict] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            pc = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            cc = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            all_matches.extend(match_cluster_with_assignment(pc, cc, config))

        matched_curr_global: set = set()

        # FIX: loop body was previously unreachable due to a misindented `continue`.
        for match in sorted(all_matches, key=lambda x: x["link_cost"]):
            prev_global = int(match["prev_index"])
            curr_global = int(match["curr_index"])

            if curr_global in matched_curr_global:
                continue

            track_id = int(df.at[prev_global, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global, "track_id"]          = track_id
                df.at[prev_global, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global)
            df.at[curr_global, "track_id"]                  = track_id
            df.at[curr_global, "track_link_distance_px"]    = float(match["link_distance_px"])
            df.at[curr_global, "track_link_cost"]           = float(match["link_cost"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(match["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(match["dbscan_is_crowded"])
            df.at[curr_global, "track_link_method"]         = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global = int(row["index"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(row["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(row["dbscan_is_crowded"])
            if df.at[curr_global, "track_id"] == -1:
                df.at[curr_global, "track_id"]          = next_track_id
                df.at[curr_global, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()
    track_len = tracked_df.groupby("track_id").size()
    return pd.DataFrame([{
        "n_rows":                int(len(tracked_df)),
        "n_tracks":              int(tracked_df["track_id"].nunique()),
        "median_track_length":   float(track_len.median()),
        "mean_track_length":     float(track_len.mean()),
        "fraction_crowded_links":float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks":   float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }])


## 18. Fiji-equivalent cumulative halos

In [23]:
def build_cumulative_halo_masks(nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, np.ndarray]:
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    masks = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        masks[f"halo_{i}_cum"] = dist_outside <= i * config.halo_step_px

    for i in range(1, config.n_halos + 1):
        prev = nucleus_mask if i == 1 else masks[f"halo_{i-1}_cum"]
        masks[f"ring_{i}"] = masks[f"halo_{i}_cum"] & ~prev

    masks["nucleus_eroded"] = (
        ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        if config.erosion_px > 0 else nucleus_mask.copy()
    )
    masks["cytoplasm_mask"] = masks[f"halo_{config.n_halos}_cum"] & ~nucleus_mask
    return masks


def _mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {"area_px": area_px, "intden": float(vals.sum()), "mean_intensity": float(vals.mean())}


def measure_fiji_equivalent_halos(
    intensity_image: np.ndarray,
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, float]:
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out: Dict[str, float] = {}

    for key in ("nucleus_mask", "nucleus_eroded", "cytoplasm_mask"):
        prefix = key.replace("_mask", "").replace("_cum", "")
        for k, v in _mask_stats(intensity_image, masks[key]).items():
            out[f"{prefix}_{k}"] = v

    for i in range(1, config.n_halos + 1):
        for k, v in _mask_stats(intensity_image, masks[f"halo_{i}_cum"]).items():
            out[f"halo_{i}_cum_{k}"] = v
        for k, v in _mask_stats(intensity_image, masks[f"ring_{i}"]).items():
            out[f"ring_{i}_{k}"] = v

    nm = out.get("nucleus_mean_intensity", np.nan)
    cm = out.get("cytoplasm_mean_intensity", np.nan)
    out["nc_ratio"]          = float(nm / cm)          if np.isfinite(nm) and np.isfinite(cm) and cm != 0         else np.nan
    out["nc_ratio_fraction"] = float(nm / (nm + cm))   if np.isfinite(nm) and np.isfinite(cm) and (nm + cm) != 0  else np.nan
    return out


def recover_nucleus_mask_from_plane(
    t_idx: int, z_idx: int,
    centroid_x_px: float, centroid_y_px: float,
    config: PipelineConfig,
) -> np.ndarray:
    mask_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
    mask = np.load(mask_path).astype(bool)
    labeled = measure.label(mask, connectivity=2)
    best_label, best_dist = None, np.inf
    for prop in measure.regionprops(labeled):
        cy, cx = prop.centroid
        dist = math.hypot(cx - centroid_x_px, cy - centroid_y_px)
        if dist < best_dist:
            best_dist, best_label = dist, prop.label
    return (labeled == best_label) if best_label is not None else np.zeros_like(mask, dtype=bool)


def _repaired_mask_path(config: PipelineConfig, t: int, nucleus_3d_id: int) -> Path:
    """Shared naming convention between repair_fragmented_nuclei (writer,
    Section 30b) and recover_nucleus_mask_for_row (reader, here)."""
    return config.obj_dir / "repaired_masks" / f"t{int(t):03d}_nuc{int(nucleus_3d_id):03d}.npy"


def recover_nucleus_mask_for_row(row, config: PipelineConfig) -> np.ndarray:
    """
    Prefer the repaired mask (Section 30b) when this row was repaired. The
    original per-plane .npy predates the repair, and on the Z the repair
    picked it may hold only the fragment (or nothing) -- loading it directly
    would silently undo the repair. Falls back to recover_nucleus_mask_from_plane
    for every row the repair didn't touch.
    """
    nucleus_3d_id = getattr(row, "nucleus_3d_id", None)
    if nucleus_3d_id is not None:
        p = _repaired_mask_path(config, row.t, nucleus_3d_id)
        if p.exists():
            return np.load(p).astype(bool)
    return recover_nucleus_mask_from_plane(
        int(row.t), int(row.z), float(row.centroid_x_px), float(row.centroid_y_px), config)


def _process_single_tracked_nucleus(row, img_5d: np.ndarray, config: PipelineConfig) -> dict:
    nucleus_mask  = recover_nucleus_mask_for_row(row, config)
    nuclear_plane = get_nuclear_plane(img_5d, int(row.t), int(row.z), config)
    halo_metrics  = measure_fiji_equivalent_halos(nuclear_plane, nucleus_mask, config)
    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"]   = rec.get("nucleus_area_px", 0) * config.pixel_size_um ** 2
    rec["cytoplasm_area_um2"] = rec.get("cytoplasm_area_px", 0) * config.pixel_size_um ** 2
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df
    rows = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )
    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df


## 19. Radial sweep (droplet-bounded)

Replaces the v15 placeholder. Casts rays from each tracked nucleus centroid outward to the edge of its assigned droplet, sampling the membrane / NE channel. Runs on v15's already-tracked nuclei at their selected best-Z, using v15's own droplet instance masks — so `Nucleus_ID` (`track_id`) matches automatically with no reconciliation.

Two methodological choices are carried over unchanged from the standalone `Halo_Radial_Sweep` notebook (both previously flagged as future work):
- nearest-pixel sampling (`int(round(...))`), not sub-pixel interpolation;
- only *this* nucleus's interior is excluded — rays are not clipped at the territory boundary with a neighbouring nucleus sharing the same droplet.


In [24]:
# =============================================================================
# 19. Radial sweep (droplet-bounded)  [v16]
# =============================================================================
# Reuses v15 artefacts:
#   - recover_nucleus_mask_from_plane(...) for the per-nucleus mask (Section 18)
#   - droplet_instance_hyperstack.tif (TZYX) written by segmentation (Section 27)
# Iterates the SAME tracked_df the halo analysis consumes, so every row already
# carries a persistent track_id == schema Nucleus_ID.

_DROPLET_STACK_CACHE: Dict[str, np.ndarray] = {}


def load_droplet_instance_stack(config: PipelineConfig) -> np.ndarray:
    """Load (and cache) the TZYX droplet instance hyperstack from segmentation."""
    key = str(config.droplet_instance_hyperstack_path)
    if key not in _DROPLET_STACK_CACHE:
        if not config.droplet_instance_hyperstack_path.exists():
            raise FileNotFoundError(
                f"Droplet instance hyperstack not found: "
                f"{config.droplet_instance_hyperstack_path}. Run segmentation "
                f"(Section 27) first, or point cfg at an existing run."
            )
        _DROPLET_STACK_CACHE[key] = tiff.imread(config.droplet_instance_hyperstack_path)
    return _DROPLET_STACK_CACHE[key]


def assign_nucleus_to_droplet(nucleus_mask: np.ndarray,
                              droplet_labeled_2d: np.ndarray) -> int:
    """Droplet label with greatest overlap with this nucleus (0 if no overlap)."""
    overlapping = droplet_labeled_2d[nucleus_mask]
    overlapping = overlapping[overlapping > 0]
    if overlapping.size == 0:
        return 0
    labels, counts = np.unique(overlapping, return_counts=True)
    return int(labels[np.argmax(counts)])


def sample_ray_to_droplet_edge(
    image_2d, droplet_mask, nucleus_mask, center_y, center_x, theta,
    step_size=1.0, exclude_nucleus=True, max_steps=4000,
):
    """Sample one radial ray from the centroid outward until it leaves the droplet.
    All distances in pixels. Nearest-pixel sampling (see Section 19 notes)."""
    ys, xs, vals, dists, inside_flags = [], [], [], [], []
    for step in range(max_steps):
        r = step * step_size
        y = center_y + r * np.sin(theta)
        x = center_x + r * np.cos(theta)
        yi = int(round(y)); xi = int(round(x))
        if yi < 0 or yi >= image_2d.shape[0] or xi < 0 or xi >= image_2d.shape[1]:
            break
        if not droplet_mask[yi, xi]:
            break
        is_nucleus = bool(nucleus_mask[yi, xi])
        if exclude_nucleus and is_nucleus:
            continue
        ys.append(yi); xs.append(xi); vals.append(image_2d[yi, xi])
        dists.append(r); inside_flags.append(is_nucleus)
    return {
        "y": np.asarray(ys, dtype=int),
        "x": np.asarray(xs, dtype=int),
        "values": np.asarray(vals, dtype=float),
        "distance_px": np.asarray(dists, dtype=float),
        "inside_nucleus": np.asarray(inside_flags, dtype=bool),
    }


def radial_profile_for_tracked_nucleus(
    radial_plane, nucleus_mask, droplet_mask, track_id, droplet_id, t, z,
    center_y, center_x, n_angles=180, step_size=1.0,
    exclude_nucleus=True, max_steps=4000,
) -> pd.DataFrame:
    """Long-form radial profile for one nucleus: one row per sampled ray point.
    distance_norm is the per-ray fraction of the distance to the droplet edge
    (schema Rho_Normalized)."""
    rows = []
    thetas = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    for theta_idx, theta in enumerate(thetas):
        ray = sample_ray_to_droplet_edge(
            radial_plane, droplet_mask, nucleus_mask, center_y, center_x, theta,
            step_size=step_size, exclude_nucleus=exclude_nucleus, max_steps=max_steps)
        if ray["distance_px"].size == 0:
            continue
        ray_len = float(ray["distance_px"][-1])
        norm = (np.zeros_like(ray["distance_px"]) if np.isclose(ray_len, 0)
                else ray["distance_px"] / ray_len)
        for i in range(ray["distance_px"].size):
            rows.append({
                "track_id": int(track_id), "t": int(t), "z": int(z),
                "droplet_id": int(droplet_id),
                "theta_index": theta_idx, "theta_rad": float(theta),
                "y": int(ray["y"][i]), "x": int(ray["x"][i]),
                "distance_px": float(ray["distance_px"][i]),
                "distance_norm": float(norm[i]),
                "intensity": float(ray["values"][i]),
                "ray_length_px": ray_len,
                "inside_nucleus": bool(ray["inside_nucleus"][i]),
            })
    return pd.DataFrame(rows)


def _process_single_tracked_nucleus_radial(row, img_5d, droplet_stack,
                                           config: PipelineConfig) -> pd.DataFrame:
    t, z = int(row.t), int(row.z)
    cx, cy = float(row.centroid_x_px), float(row.centroid_y_px)
    nucleus_mask = recover_nucleus_mask_for_row(row, config)
    droplet_labeled_2d = droplet_stack[t, z]
    droplet_id = assign_nucleus_to_droplet(nucleus_mask, droplet_labeled_2d)
    if droplet_id == 0:
        return pd.DataFrame()   # no assigned droplet -> no bounded sweep
    droplet_mask = droplet_labeled_2d == droplet_id
    radial_plane = img_5d[t, z, config.radial_channel_index]
    return radial_profile_for_tracked_nucleus(
        radial_plane, nucleus_mask, droplet_mask,
        track_id=int(row.track_id), droplet_id=droplet_id, t=t, z=z,
        center_y=cy, center_x=cx,
        n_angles=config.radial_n_angles,
        step_size=config.radial_step_size_px,
        exclude_nucleus=config.radial_exclude_nucleus,
        max_steps=config.radial_max_ray_steps)


def run_radial_sweep_for_tracked_nuclei(
    tracked_df: pd.DataFrame, img_5d: np.ndarray, config: PipelineConfig,
) -> pd.DataFrame:
    """Radial sweep over every tracked nucleus at its selected best-Z.
    Returns a long-form DataFrame (one row per ray sample point) and pickles it."""
    out_path = config.analysis_dir / "radial_sweep.pkl"
    if tracked_df.empty:
        radial_df = pd.DataFrame()
        radial_df.to_pickle(out_path)
        return radial_df

    droplet_stack = load_droplet_instance_stack(config)
    assert droplet_stack.shape[0] == img_5d.shape[0], "Droplet stack T mismatch vs image"
    assert droplet_stack.shape[1] == img_5d.shape[1], "Droplet stack Z mismatch vs image"
    assert droplet_stack.shape[-2:] == img_5d.shape[-2:], "Droplet stack Y/X mismatch vs image"

    dfs = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus_radial)(row, img_5d, droplet_stack, config)
        for row in tracked_df.itertuples(index=False)
    )
    dfs = [d for d in dfs if not d.empty]
    radial_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    radial_df.to_pickle(out_path)
    return radial_df


## 20. Portable export helper

In [25]:
def _save_portable_table(df: pd.DataFrame, out_stem: Path) -> dict:
    """Save a table as CSV (human-readable) + pickle (lossless, fast).
    Parquet removed — requires pyarrow/fastparquet which are not available."""
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    saved: dict = {}
    if df is None:
        return saved
    csv_path = out_stem.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    saved["csv"] = str(csv_path)
    pkl_path = out_stem.with_suffix(".pkl")
    df.to_pickle(pkl_path)
    saved["pickle"] = str(pkl_path)
    return saved


def export_portable_analysis_bundle(
    config: PipelineConfig,
    export_dir: Union[str, Path],
    seg_index_df: Optional[pd.DataFrame] = None,
    objects_df: Optional[pd.DataFrame] = None,
    grouped_z_df: Optional[pd.DataFrame] = None,
    best_z_df: Optional[pd.DataFrame] = None,
    filtered_df: Optional[pd.DataFrame] = None,
    timed_df: Optional[pd.DataFrame] = None,
    tracked_df: Optional[pd.DataFrame] = None,
    halo_df: Optional[pd.DataFrame] = None,
    copy_mask_npy_files: bool = False,
) -> Path:
    export_dir = Path(export_dir)
    dirs = {k: export_dir / k for k in ("tables", "masks", "config", "notes")}
    for p in [export_dir] + list(dirs.values()):
        p.mkdir(parents=True, exist_ok=True)

    manifest = {
        "created_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_image_name": config.input_image_name,
        "input_image_path_on_cluster": str(config.input_image_path),
        "pixel_size_um": float(config.pixel_size_um),
        "z_step_um": float(config.z_step_um),
        "nuclear_channel_index": int(config.nuclear_channel_index),
        "membrane_channel_index": int(config.membrane_channel_index),
        "copied_files": {}, "tables": {},
    }

    config_path = dirs["config"] / "pipeline_config.json"
    config_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))
    manifest["copied_files"]["pipeline_config"] = str(config_path)

    for key, src in {
        "segmentation_class_hyperstack":       config.segmentation_class_hyperstack_path,
        "segmentation_probability_hyperstack": config.segmentation_probability_hyperstack_path,
        "segmentation_label_hyperstack":       config.segmentation_label_hyperstack_path,
        "nucleus_instance_hyperstack":         config.nucleus_instance_hyperstack_path,
        "droplet_instance_hyperstack":         config.droplet_instance_hyperstack_path,
    }.items():
        src = Path(src)
        if src.exists():
            dst = dirs["masks"] / src.name
            shutil.copy2(src, dst)
            manifest["copied_files"][key] = str(dst)

    for name, df in {
        "segmentation_index":           seg_index_df,
        "plane_objects":                objects_df,
        "grouped_z_objects":            grouped_z_df,
        "best_z_nuclei":                best_z_df,
        "best_z_nuclei_with_exclusion": filtered_df,
        "best_z_nuclei_timed":          timed_df,
        "tracked_nuclei":               tracked_df,
        "halo_analysis":                halo_df,
    }.items():
        if df is not None:
            manifest["tables"][name] = _save_portable_table(df, dirs["tables"] / name)

    if copy_mask_npy_files and seg_index_df is not None and "mask_path" in seg_index_df.columns:
        plane_mask_dir = dirs["masks"] / "plane_npy_masks"
        plane_mask_dir.mkdir(parents=True, exist_ok=True)
        portable_idx = seg_index_df.copy()
        new_paths = []
        for row in portable_idx.itertuples(index=False):
            src = Path(row.mask_path)
            if src.exists():
                dst = plane_mask_dir / src.name
                shutil.copy2(src, dst)
                new_paths.append(str(dst))
            else:
                new_paths.append(None)
        portable_idx["portable_mask_path"] = new_paths
        manifest["tables"]["segmentation_index_portable"] = _save_portable_table(
            portable_idx, dirs["tables"] / "segmentation_index_portable")

    readme = (
        "Portable laptop analysis bundle\n\n"
        "Contents\n--------\n"
        "masks/  — Hyperstack TIFFs for segmentation QC, instance labels, and napari overlays.\n"
        "tables/ — Analysis tables (CSV + pickle).\n"
        "config/ — Pipeline configuration snapshot.\n\n"
        "Recommended workflow\n--------------------\n"
        "1. Open TIFF files in napari for segmentation debugging.\n"
        "2. Load halo_analysis.pkl or .csv for plotting and filtering.\n"
        "3. Use tracked_nuclei / best_z_nuclei tables for re-analysis without rerunning the model.\n"
    )
    (dirs["notes"] / "README.txt").write_text(readme)
    (export_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return export_dir


## 21. QC plotting helpers

In [26]:
def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config: PipelineConfig,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Overlay raw nuclear channel with the binary nucleus mask."""
    T, Z, C, Y, X = img_5d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, t_idx in zip(axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        raw = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        if raw.max() > 0:
            raw /= raw.max()
        ax.imshow(raw, cmap="gray")
        ax.imshow(nucleus_mask_4d[t_idx, z_idx].astype(bool), alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")

    for ax in axes[len(timepoints):]:
        ax.axis("off")

    fig.suptitle("QC: Nuclear channel + nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Mask snapshots with contours + total nuclear area over time."""
    from skimage.measure import find_contours
    T, Z, Y, X = nucleus_mask_4d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs  = fig.add_gridspec(n_rows + 1, n_cols)

    panel_axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]

    for ax, t_idx in zip(panel_axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled    = measure.label(mask_plane)
        ax.imshow(mask_plane, cmap="gray")
        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)
        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")

    for ax in panel_axes[len(timepoints):]:
        ax.axis("off")

    area_ax = fig.add_subplot(gs[n_rows, :])
    area_ax.plot(np.arange(T), nucleus_mask_4d.reshape(T, -1).sum(axis=1), marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """Count connected nucleus objects across all Z slices per time point."""
    T, Z = nucleus_mask_4d.shape[:2]
    counts = [
        sum(measure.label(nucleus_mask_4d[t, z].astype(bool)).max() for z in range(Z))
        for t in range(T)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(T), counts, marker="o")
    ax.set_xlabel("Time index")
    ax.set_ylabel("Connected nucleus objects")
    ax.set_title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()


def plot_best_focus_nuclear_planes(
    img_5d: np.ndarray,
    config: PipelineConfig,
    figsize_per_panel: float = 4.0,
) -> pd.DataFrame:
    """Plot the single best-focus nuclear-channel Z plane for each time point."""
    n_t = img_5d.shape[0]
    n_cols = min(5, n_t)
    n_rows = math.ceil(n_t / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_panel * n_cols, figsize_per_panel * n_rows))
    axes = np.array(axes).reshape(-1)
    focus_rows = []

    for t in range(n_t):
        keep_z, scores, best_z = get_best_focus_z_indices(
            img_5d, t,
            nuc_channel_idx=config.effective_focus_channel_index,
            exclude_edge_z=config.focus_edge_z_exclusion,
            window_radius=0,
            metric=config.focus_metric,
            pixel_size_um=config.pixel_size_um,
            config=config,
        )
        ax = axes[t]
        if best_z is None:
            ax.set_title(f"t={t}: no valid focus plane")
            ax.axis("off")
            focus_rows.append({"t": t, "best_z": np.nan, "focus_score": np.nan,
                                "focus_metric": config.focus_metric,
                                "nuclear_channel_index": config.effective_focus_channel_index})
            continue
        nuc2d      = img_5d[t, best_z, config.effective_focus_channel_index]
        best_score = float(scores[best_z])
        ax.imshow(nuc2d, cmap="gray")
        ax.set_title(f"t={t}, best z={best_z}\nfocus={best_score:.4g}")
        ax.axis("off")
        focus_rows.append({"t": t, "best_z": int(best_z), "focus_score": best_score,
                           "focus_metric": config.focus_metric,
                           "nuclear_channel_index": config.effective_focus_channel_index})

    for ax in axes[n_t:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(focus_rows)


def plot_nucleus_probability_qc(class_prob_map: np.ndarray, raw_plane: np.ndarray,
                                title: str = "") -> None:
    """Show raw plane, nucleus probability map, and its histogram."""
    nucleus_prob = class_prob_map[..., 2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_plane, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    im = axes[1].imshow(nucleus_prob, cmap="inferno")
    axes[1].set_title("Nucleus probability"); plt.colorbar(im, ax=axes[1])
    axes[2].hist(nucleus_prob.ravel(), bins=100); axes[2].set_title("Probability distribution")
    fig.suptitle(title); plt.tight_layout(); plt.show()


def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("N / (N + C)")
    ax.set_title("N/C ratio fraction by track")
    plt.tight_layout(); plt.show()


def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear area (µm²)")
    ax.set_title("Largest nuclear cross-sectional area by track")
    plt.tight_layout(); plt.show()


def plot_largest_cross_sectional_area_vs_time(
    df: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """
    Scatter all individual nuclear areas plus population mean vs true acquisition time.

    Works with timed_df, tracked_df, or halo_df.  Requires 'true_time_min'.
    """
    if df.empty:
        print("No data to plot."); return
    plot_df = df.copy()
    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("DataFrame must contain 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * config.pixel_size_um ** 2
    if "true_time_min" not in plot_df.columns:
        raise ValueError("DataFrame must contain 'true_time_min'.")

    plot_df = plot_df.sort_values("true_time_min")
    mean_df = (plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
               .mean().rename(columns={"nucleus_area_um2": "mean_area"}))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["true_time_min"], plot_df["nucleus_area_um2"], alpha=0.35, s=20,
               label="Individual nuclei")
    ax.plot(mean_df["true_time_min"], mean_df["mean_area"], linewidth=2, label="Population mean")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear cross-sectional area (µm²)")
    ax.set_title("Nuclear area vs time")
    ax.legend(); plt.tight_layout(); plt.show()


## 22. Stage execution flags

Set a flag to `True` to recompute that stage; `False` loads the saved artefact from disk.

In [27]:
# -- Stage execution flags -----------------------------------------------
# Segmentation (§27) has NO RUN_/RESUME_ flag: it reads its on-disk checkpoint
# (seg_dir/_checkpoint.json) and decides fresh / resume / complete automatically.
# To force a rebuild, call reset_segmentation(cfg, scope=...) in the §27 reset
# cell, then re-run §27.
SAVE_PROBABILITY_TIFF  = True    # write the ~35 GB probability hyperstack (enables no-GPU re-extraction)
RUN_SINGLE_PLANE_DEBUG = False   # §26 / cell-after-§27 single-plane inspection

RUN_OBJECT_EXTRACTION = True
RUN_GROUP_ACROSS_Z    = True
RUN_SELECT_BEST_Z     = True
RUN_FLAG_CLOSE_NUCLEI = True
RUN_ADD_TIMING        = True
RUN_TRACKING          = True
RUN_HALO_ANALYSIS     = True
RUN_RADIAL_SWEEP      = True
RUN_EXPORT_PORTABLE   = True

DEBUG_T_IDX = 9
DEBUG_Z_IDX = 16

## 23. Initialise runtime and confirm paths

In [28]:
configure_tensorflow_for_gpu(cfg)

print(f"Model path:      {cfg.model_path}")
print(f"Input image:     {cfg.input_image_path}")
print(f"Output dir:      {cfg.output_dir}")

assert cfg.model_path.exists(),       f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "debug_pipeline_config.json")
print("Configuration saved.")


GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Model path:      /data/user/tdeibert/Nuclear_Scaling/Inputs/Models/Vulcan_1.1_best.keras
Input image:     /data/user/tdeibert/Nuclear_Scaling/Inputs/Raw_Images/control_extract_1.1.tif
Output dir:      /data/user/tdeibert/Nuclear_Scaling/Runs/control_extract_1.1__cfg-a6725cc8
Configuration saved.


## 24. Load image

In [29]:
t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Shape: {img_5d.shape}  |  dtype: {img_5d.dtype}  |  load time: {time.perf_counter() - t0:.2f}s")


Loaded image shape: (10, 20, 3, 3889, 5732)
Shape: (10, 20, 3, 3889, 5732)  |  dtype: uint16  |  load time: 10.47s


## 25. Optional focus-score QC

In [30]:
y_starts = _generate_patch_starts(3889, cfg.patch_size, cfg.patch_stride)
x_starts = _generate_patch_starts(5732, cfg.patch_size, cfg.patch_stride)
print(f"Y starts: {len(y_starts)} patches, first={y_starts[0]}, last={y_starts[-1]}")
print(f"X starts: {len(x_starts)} patches, first={x_starts[0]}, last={x_starts[-1]}")
print(f"Total patches: {len(y_starts) * len(x_starts)}")

Y starts: 15 patches, first=0, last=3377
X starts: 22 patches, first=0, last=5220
Total patches: 330


## 26. Single-plane segmentation debug

In [31]:
# Single-plane segmentation debug (§26). Opt-in via RUN_SINGLE_PLANE_DEBUG
# (§22). Not part of the pipeline path — a full run (§27) loads its own model.
model = None

if RUN_SINGLE_PLANE_DEBUG:
    model = load_unet_model(cfg.model_path)
    print("Model loaded for single-plane debug.")

    plane_yxc = get_full_plane_yxc(img_5d, DEBUG_T_IDX, DEBUG_Z_IDX, config=cfg)
    print(f"Debug plane shape: {plane_yxc.shape}")

    _debug_batch_size = cfg.batch_size
    cfg.batch_size = 1  # single-plane debug: limit VRAM
    debug_prob_map, debug_class_map, debug_nucleus_mask, debug_instance_labels = (
        segment_single_plane_with_overlap(
            plane_yxc, model, cfg,
            patch_size=cfg.patch_size, stride=cfg.patch_stride,
        )
    )
    print("Single-plane debug segmentation complete.")
    print(f"  Probability map shape: {debug_prob_map.shape}")
    print(f"  Unique class labels:   {np.unique(debug_class_map)}")
    print(f"  Nucleus pixels:        {int(debug_nucleus_mask.sum())}")
    for c in range(4):
        print(f"  Class {c} — min {debug_prob_map[...,c].min():.4f}  "
              f"max {debug_prob_map[...,c].max():.4f}  "
              f"mean {debug_prob_map[...,c].mean():.4f}")
    cfg.batch_size = _debug_batch_size
    del debug_prob_map, debug_class_map, debug_nucleus_mask
    del debug_instance_labels, plane_yxc
    gc.collect()
    print("Debug arrays deleted from kernel memory.")
else:
    print("RUN_SINGLE_PLANE_DEBUG is False — skipping §26 single-plane debug.")

RUN_SINGLE_PLANE_DEBUG is False — skipping §26 single-plane debug.


## 27. Run or load full segmentation

### ⚠️ Before running this cell

If you have edited any function cells above (Sections 6–10), you **must restart the kernel and re-run all cells from the top** before running segmentation.

The kernel caches function definitions in memory. Running only the edited cell updates the source but does **not** replace the compiled function object the segmentation loop calls. A stale cached definition is the cause of `NameError: class_mask_5d` and similar errors even when the notebook source looks correct.

**Restart kernel → Run All** (or Kernel → Restart & Run All Cells).

In [ ]:
# State is decided from the on-disk checkpoint BEFORE any model load or focus
# warm-up. reset_segmentation(cfg, scope=...) in the next cell forces a rebuild.
_T, _Z = img_5d.shape[0], img_5d.shape[1]
_Y, _X = img_5d.shape[-2], img_5d.shape[-1]

seg_state = assess_segmentation_state(cfg, (_T, _Z, _Y, _X),
                                      save_probability_tiff=SAVE_PROBABILITY_TIFF)
print(f"[segmentation state: {seg_state.mode}]")
print(seg_state.reason)

if cfg.probability_input_path is not None:
    # Layer 3: re-extract masks from a saved probability hyperstack (no model,
    # no inference). Point cfg.probability_hyperstack_input at any saved map.
    print(f"\nRe-extracting from probability hyperstack: {cfg.probability_input_path}")
    print(f"  thresholds  droplet={cfg.droplet_threshold}  "
          f"npc={cfg.npc_threshold}  nucleus={cfg.nucleus_threshold}")
    t0 = time.perf_counter()
    seg_index_df = run_reextraction_from_probability(config=cfg)
    print(f"Re-extraction finished in {time.perf_counter() - t0:.2f}s")

elif seg_state.mode == "complete":
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Segmentation already complete — skipped model load and focus scoring.")

elif seg_state.mode == "inconsistent":
    raise RuntimeError(seg_state.reason)

elif seg_state.mode in ("fresh", "resume"):
    if seg_state.todo_t and ('model' not in globals() or model is None):
        model = load_unet_model(cfg.model_path)
    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(
        img_5d=img_5d,
        model=(model if seg_state.todo_t else None),
        config=cfg, state=seg_state,
        save_probability_tiff=SAVE_PROBABILITY_TIFF)
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f}s")

else:
    raise RuntimeError(f"unhandled seg_state.mode={seg_state.mode!r}")

print(seg_index_df.shape)
display(seg_index_df.head())

[segmentation state: fresh]
No segmentation checkpoint (_checkpoint.json) — fresh run, all 10 timepoints to segment.


2026-09-02 13:48:28.507971: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79078 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:81:00.0, compute capability: 8.0


Pre-computing focus scores (v13 adaptive) for t=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...
    ref_z=12 (0 droplets)
  t=0: channel='npc_ring'  best_z=16  keep_z=[10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=9 (1 droplets)
  t=1: channel='npc_ring'  best_z=9  keep_z=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    ref_z=12 (9 droplets)
  t=2: channel='npc_ring'  best_z=12  keep_z=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=9 (28 droplets)
  t=3: channel='npc_ring'  best_z=9  keep_z=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    ref_z=12 (47 droplets)
  t=4: channel='npc_ring'  best_z=12  keep_z=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=13 (23 droplets)
  t=5: channel='npc_ring'  best_z=14  keep_z=[8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=13 (25 droplets)
  t=6: channel='npc_ring'  best_z=13  keep_z=[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=14 (19 droplets)
  t=7: channel='npc_ring'  best_z=15  keep_z=[9, 10, 11, 12, 13, 14, 15, 16, 17]
    ref_z=15 (26 droplets)
  t=8: 

2026-09-02 13:58:32.959545: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904


  Model warmed up (batch=8, patch=512, channels=3)
t=0: channel='npc_ring'  best_z=16  segmenting z=[10, 11, 12, 13, 14, 15, 16, 17]  [RSS=28.2GB  GPU current=0.0GB peak=6.3GB]
    z=10: nuc_px=0  fg_frac=0.000  gate=0/3
    z=11: nuc_px=0  fg_frac=0.000  gate=0/2
    z=12: nuc_px=1,203  fg_frac=0.000  gate=1/1
    z=13: nuc_px=0  fg_frac=0.000  gate=0/1
    z=14: nuc_px=0  fg_frac=0.000  gate=0/0
    z=15: nuc_px=0  fg_frac=0.000  gate=0/0
    z=16: nuc_px=0  fg_frac=0.000  gate=0/0
    z=17: nuc_px=0  fg_frac=0.000  gate=0/0
  t=0 complete in 505.5s
t=1: channel='npc_ring'  best_z=9  segmenting z=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  [RSS=137.5GB  GPU current=0.0GB peak=6.3GB]
    z=6: nuc_px=1,323  fg_frac=0.000  gate=1/42
    z=7: nuc_px=2,139  fg_frac=0.000  gate=1/37
    z=8: nuc_px=0  fg_frac=0.000  gate=0/33
    z=9: nuc_px=3,077  fg_frac=0.000  gate=2/32
    z=10: nuc_px=0  fg_frac=0.000  gate=0/26
    z=11: nuc_px=7,526  fg_frac=0.000  gate=2/19
    z=12: nuc_px=9,184  fg_fra

In [ ]:
# ============================================================
# Debug: Visualize single plane — raw channels + model predictions
# Opt-in: needs RUN_SINGLE_PLANE_DEBUG=True (§22) so `model` is loaded.
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if not RUN_SINGLE_PLANE_DEBUG or ('model' not in globals()) or model is None:
    print("RUN_SINGLE_PLANE_DEBUG is False (or no model loaded) — "
          "skipping single-plane visualization.")
else:
    mem_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.membrane_channel_index].astype(np.float32)
    nls_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.nuclear_channel_index].astype(np.float32)
    npc_raw  = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.npc_channel_index].astype(np.float32)

    def norm(arr):
        lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
        return np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)

    plane_yxc = get_full_plane_yxc(img_5d, DEBUG_T_IDX, DEBUG_Z_IDX, config=cfg)
    patches, coords = extract_overlapping_patches(plane_yxc, cfg.patch_size, cfg.patch_stride)
    patch_arr = np.stack([_preprocess_patch(p) for p in patches], axis=0)
    probs_arr = _run_batched_inference(patch_arr, model, batch_size=cfg.batch_size)
    h, w, _ = plane_yxc.shape
    prob_map = stitch_probability_patches(probs_arr, coords, (h, w), cfg.num_classes, cfg.patch_size)
    _, _dbg_drop, _dbg_npc, _dbg_nuc, label_map = extract_class_masks(prob_map, cfg)

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.suptitle(f"t={DEBUG_T_IDX}  z={DEBUG_Z_IDX}  |  thr nuc={cfg.nucleus_threshold} npc={cfg.npc_threshold} drop={cfg.droplet_threshold}", fontsize=12)

    axes[0, 0].imshow(norm(mem_raw),  cmap="gray");    axes[0, 0].set_title("Ch0: Membrane (raw)")
    axes[0, 1].imshow(norm(nls_raw),  cmap="gray");    axes[0, 1].set_title("Ch1: NLS / Nucleus (raw)")
    axes[0, 2].imshow(norm(npc_raw),  cmap="gray");    axes[0, 2].set_title("Ch2: NPC (raw)")
    axes[0, 3].imshow(label_map, cmap="viridis", vmin=0, vmax=3)
    axes[0, 3].set_title("Predicted class label\n0=bg 1=droplet 2=NPC 3=nucleus")

    class_info = [
        (0, "Background prob",  "gray"),
        (1, "Droplet prob",     "Blues"),
        (2, "NPC prob",         "Purples"),
        (3, "Nucleus prob",     "Greens"),
    ]
    for c, title, cmap in class_info:
        im = axes[1, c].imshow(prob_map[..., c], cmap=cmap, vmin=0, vmax=1)
        axes[1, c].set_title(f"{title}\nmax={prob_map[...,c].max():.3f}  mean={prob_map[...,c].mean():.4f}")
        plt.colorbar(im, ax=axes[1, c], fraction=0.03)

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\nClass pixel counts in predicted label map:")
    for c, title, _ in class_info:
        px = int((label_map == c).sum())
        frac = px / label_map.size
        print(f"  Class {c} ({title.split()[0]:10s}): {px:>10,} px  ({frac:.3%})")

tm = pd.read_pickle(cfg.track_dir / "best_z_nuclei_timed.pkl")
print((tm.groupby(["t","tile_index"])["area_px"].median()
         .unstack() * cfg.pixel_size_um**2).round(0))

In [ ]:
# -- Reset segmentation state --------------------------------------------
# Uncomment ONE call, run it, then re-run §27.
#
#   scope="all"     wipe the checkpoint, _resume/, the 5 hyperstack TIFFs, the
#                   per-plane .npy masks, the seg index / ROI pickles, AND the
#                   stale downstream object pickles (plane / grouped / best_z /
#                   repaired, repaired_masks/). Nothing left half-deleted.
#   scope="resume"  wipe only the checkpoint + _resume/ (keep the TIFFs). Forces
#                   a full recompute; use after editing a segmentation function.
#
# reset_segmentation(cfg, scope="all")
# reset_segmentation(cfg, scope="resume")

print("Reset cell — edit to uncomment a reset_segmentation(...) call.")

## 28. Run or load object extraction

In [ ]:
# -- Manual recovery utility (NOT run automatically) --------------------
# Rebuilds segmentation_index.pkl from the per-plane nuclear_mask_*.npy files
# alone, for the case where those planes survived but the index pickle was lost.
# focus_score / gate_* cannot be recovered from binary masks and are padded NaN.
# Prefer reset_segmentation(cfg, scope="all") + a clean §27 re-run; use this only
# when a full re-run is not affordable.
def rebuild_seg_index_from_npy(config) -> "pd.DataFrame":
    import re
    mask_files = sorted(config.seg_dir.glob("nuclear_mask_t*_z*.npy"))
    if not mask_files:
        raise FileNotFoundError(f"no nuclear_mask_t*_z*.npy under {config.seg_dir}")
    pat = re.compile(r"nuclear_mask_t(\d+)_z(\d+)\.npy$")
    rows = []
    for p in mask_files:
        m = pat.search(p.name)
        t_idx, z_idx = int(m.group(1)), int(m.group(2))
        npix = int(np.load(p).astype(bool).sum())
        rows.append({
            "t": t_idx, "z": z_idx, "included": npix > 0,
            "reason": "segmented" if npix > 0 else "empty_or_skipped",
            "focus_score": np.nan, "best_focus_z": np.nan,
            "focus_channel": "reconstructed", "mask_path": str(p),
            "nucleus_pixels": float(npix), "droplet_pixels": np.nan,
            "gate_candidates": np.nan, "gate_verified": np.nan,
        })
    df = pd.DataFrame(rows).sort_values(["t", "z"]).reset_index(drop=True)
    df.to_pickle(config.segmentation_index_path)
    print(f"Rebuilt {config.segmentation_index_path}  {df.shape}")
    return df

# rebuild_seg_index_from_npy(cfg)   # <- uncomment to run

In [ ]:
require_segmentation(cfg, needs=("npy_planes",))

objects_path = cfg.obj_dir / "plane_objects.pkl"

if RUN_OBJECT_EXTRACTION:
    t0 = time.perf_counter()
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
    print(f"Object extraction finished in {time.perf_counter() - t0:.2f}s")
else:
    objects_df = pd.read_pickle(objects_path)
    print("Loaded saved object table.")

print(objects_df.shape)
display(objects_df.head())

## 29. Run or load Z grouping

In [ ]:
grouped_z_path = cfg.obj_dir / "grouped_z_objects.pkl"

if RUN_GROUP_ACROSS_Z:
    t0 = time.perf_counter()
    grouped_z_df = group_nuclei_across_z(objects_df, cfg)
    print(f"Z grouping finished in {time.perf_counter() - t0:.2f}s")
else:
    grouped_z_df = pd.read_pickle(grouped_z_path)
    print("Loaded grouped-Z table.")

print(grouped_z_df.shape)
display(grouped_z_df.head())


## 30. Run or load best-Z selection

In [ ]:
best_z_path = cfg.obj_dir / "best_z_nuclei.pkl"

if RUN_SELECT_BEST_Z:
    t0 = time.perf_counter()
    best_z_df = select_best_z_per_nucleus(grouped_z_df, cfg)
    print(f"Best-Z selection finished in {time.perf_counter() - t0:.2f}s")
else:
    best_z_df = pd.read_pickle(best_z_path)
    print("Loaded best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())


In [ ]:
# QC: Z-grouping / best-Z-selection diagnostics (moved here from earlier in the
# notebook — they depend on grouped_z_df and best_z_df, which do not exist yet
# the first time Sections 27-30 run top to bottom).
edge = grouped_z_df.groupby(["t", "nucleus_3d_id"])["z"].agg(["min", "max"])
m = best_z_df.set_index(["t", "nucleus_3d_id"]).join(edge)
m["at_edge"] = (m.z == m["min"]) | (m.z == m["max"])
print("fraction selected at window edge, by t:")
print(m.groupby(level="t")["at_edge"].mean())
print("\nplanes per nucleus (how many Z actually grouped):")
print(grouped_z_df.groupby(["t", "nucleus_3d_id"]).size().value_counts())
print("\nselected z by t:")
print(best_z_df.groupby("t")["z"].value_counts().unstack(fill_value=0))

three = grouped_z_df.groupby(["t", "nucleus_3d_id"]).filter(lambda x: len(x) == 3)
prof = (three.sort_values("z")
             .groupby(["t", "nucleus_3d_id"])["area_px"]
             .agg(list).apply(lambda a: (max(a) - min(a)) / max(a)))
print("\nmedian fractional area spread across the 3 planes, by t:")
print(prof.groupby(level="t").median().round(3))

# is the max at the middle plane or an edge?
pos = (three.sort_values("z").groupby(["t", "nucleus_3d_id"])["area_px"]
            .agg(lambda a: int(np.argmax(list(a)))))
print("\nargmax position (0=low edge, 1=center, 2=high edge):")
print(pos.groupby(level="t").value_counts().unstack(fill_value=0))


## 30b. Fragmentation repair  *(new in v17)*

### What this fixes

The nucleus class mask frequently covers only a bright interior region of the
nucleus — a chromatin-dense core — rather than the whole nucleoplasm. Measured
on `control_extract_1.1`: **31 % of detections**, with median solidity 0.894
against 0.963 for the rest, and areas roughly a third of true.

The model is not at fault. At one inspected nucleus the probability map gave
456 µm² at p>0.7 and 486 µm² at p>0.5 while the reported area was 56 µm². The
full nucleus is present in the probabilities; the mask keeps the core.

### Why it also destroyed z-coverage

The bright core spans far fewer planes than the nucleus, so most planes
produced no detection at all — `NucleusZStack` held 1.44 planes per nucleus
where a 400 µm² nucleus should span ~11 at 2 µm. And `select_best_z_per_nucleus`
picks the plane where the **fragment** is largest, which is the core's own
z-centre, not the nuclear equator. So the reported area was a non-equatorial
slice of an already-truncated mask.

### The repair

1. **Detect** — solidity < 0.93, or area far below the timepoint's 75th
   percentile. Both are needed: a small round core is convex, so solidity
   alone misses it.
2. **Bound by the NPC ring** — cast 72 rays from the seed, take the median
   radius of peak NPC intensity. The ring is a closed structure around the
   nucleus, so it constrains every ray at once. Without this the basin
   crossed the envelope and leaked into fused neighbouring droplets.
3. **Flatten, then watershed** — opening-then-closing by reconstruction
   removes the bright core without displacing the envelope (a Gaussian strong
   enough to erase the core also softens the boundary), then a marker
   watershed on the flattened gradient grows the seed to the envelope.
4. **Gate on signed contrast** — the interior must exceed the droplet median.
   Plain `ne_contrast` passed masks that had grabbed a *dark* region beside
   the nucleus, because dark-inside over bright-outside still reads high.
5. **Re-select z** — repair every plane in a window, propagating the mask
   inward as the seed so planes that never produced a detection are still
   measured, and take the true maximum.

### Validation

Scored against the NPC ring, which is independent of the NLS channel and so
cannot inherit the fragmentation. Expect **0.88–0.95**: the NLS boundary is
the inner face of the envelope and the NPC peak is its centre-line, so the
ratio should sit slightly below 1. Above 1.0 means the mask has crossed the
envelope; below 0.85 means it is still short.

`area_px` is **overwritten** with the repaired value so every downstream stage
picks it up unchanged. The original is preserved in `area_px_original`.

In [ ]:
from scipy import ndimage as ndi
from skimage import filters, measure, morphology, segmentation


def _resolve_instance(inst_plane, cx, cy, bbox=None):
    """
    objects_df.label is per-plane regionprops numbering; the instance
    hyperstack uses sparse global ids. They are different namespaces, so the
    label column cannot index the instance map — resolve by centroid instead.
    """
    H, W = inst_plane.shape
    yi, xi = int(round(cy)), int(round(cx))
    if 0 <= yi < H and 0 <= xi < W:
        v = int(inst_plane[yi, xi])
        if v > 0:
            return v
    if bbox is not None:
        r0, c0, r1, c1 = [int(b) for b in bbox]
        sub = inst_plane[max(r0, 0):min(r1, H), max(c0, 0):min(c1, W)]
        if sub.size:
            cts = np.bincount(sub.ravel()); cts[0] = 0
            if cts.max() > 0:
                return int(cts.argmax())
    return 0


def flatten_interior(img_norm, se_radius_px):
    """
    Opening- then closing-by-reconstruction. Removes bright structures smaller
    than the SE while leaving the position and shape of everything larger
    untouched — the property a Gaussian does not have.

    Without this the watershed does not move: the core's own edge is a stronger
    gradient than the envelope, so the basin stops at the core.
    """
    se = morphology.disk(max(int(se_radius_px), 1))
    opened = morphology.reconstruction(morphology.erosion(img_norm, se),
                                       img_norm, method="dilation")
    closed = morphology.reconstruction(morphology.dilation(opened, se),
                                       opened, method="erosion")
    return closed.astype(np.float32)


def seeded_watershed(seed, nls, bound, config, flatten_um=8.0,
                     smooth_um=0.8, wall_um=1.5):
    """Grow `seed` to the envelope, confined to `bound`."""
    if not seed.any() or not bound.any():
        return np.zeros(nls.shape, bool)
    lo, hi = np.percentile(nls[bound], [1, 99.5])
    if hi <= lo:
        return np.zeros(nls.shape, bool)
    norm = np.clip((nls - lo) / (hi - lo), 0, 1)
    sm = filters.gaussian(norm, sigma=max(smooth_um / config.pixel_size_um, 0.5),
                          preserve_range=True).astype(np.float32)
    sm = flatten_interior(sm, int(round(flatten_um / config.pixel_size_um)))

    er = max(int(round(wall_um / config.pixel_size_um)), 1)
    wall = bound & ~morphology.binary_erosion(bound, morphology.disk(er))
    if not wall.any():
        wall = bound & ~morphology.binary_erosion(bound, morphology.disk(1))
    markers = np.zeros(seed.shape, np.int32)
    markers[wall] = 1
    markers[seed & bound] = 2
    if not (markers == 2).any():
        return np.zeros(nls.shape, bool)
    ws = segmentation.watershed(filters.sobel(sm), markers, mask=bound)
    return ndi.binary_fill_holes(ws == 2)


def npc_ring_radius(npc_crop, bound, ly, lx, config, n_rays=72, k_std=1.5):
    """Median radius of peak NPC intensity over rays from (ly, lx)."""
    r_max = int(np.sqrt(max(bound.sum(), 1) / np.pi))
    rr = np.arange(2, max(r_max, 3))
    peaks = []
    for th in np.linspace(0, 2 * np.pi, n_rays, endpoint=False):
        ys = (ly + rr * np.sin(th)).astype(int)
        xs = (lx + rr * np.cos(th)).astype(int)
        ok = ((ys >= 0) & (ys < npc_crop.shape[0])
              & (xs >= 0) & (xs < npc_crop.shape[1]))
        if ok.sum() < 5:
            continue
        prof = npc_crop[ys[ok], xs[ok]]
        if prof.max() < prof.mean() + k_std * prof.std():
            continue
        peaks.append(rr[ok][int(np.argmax(prof))])
    if len(peaks) < n_rays // 8:
        return np.nan, len(peaks)
    return float(np.median(peaks)) * config.pixel_size_um, len(peaks)


def signed_contrast_ok(mask, nls, droplet, config):
    """Interior must beat the droplet median — kills dark-region grabs."""
    if not mask.any():
        return False, "empty"
    if droplet.any() and mask.sum() / droplet.sum() > config.repair_max_frac_droplet:
        return False, f"flooded ({mask.sum()/droplet.sum():.2f})"
    inside = float(nls[mask].mean())
    if droplet.any() and inside < float(np.median(nls[droplet])):
        return False, "interior darker than droplet"
    shell = (morphology.binary_dilation(
        mask, morphology.disk(max(int(1.3 / config.pixel_size_um), 1)))
        & ~mask & droplet)
    if shell.any():
        out = float(nls[shell].mean())
        if out > 0 and inside / out < config.repair_min_signed_margin:
            return False, f"low signed contrast ({inside/out:.2f})"
    return True, ""


def repair_plane(t, z, cx, cy, image_5d, nucleus_instance_4d,
                 droplet_instance_4d, config, seed_override=None):
    """Repair one nucleus on one plane. Returns (mask, metrics, offsets)."""
    half = int(round((config.repair_win_um / config.pixel_size_um) / 2))
    H, W = image_5d.shape[-2], image_5d.shape[-1]
    r0, c0 = max(int(cy) - half, 0), max(int(cx) - half, 0)
    r1, c1 = min(int(cy) + half, H), min(int(cx) + half, W)
    sl = (slice(r0, r1), slice(c0, c1))

    nls = np.asarray(image_5d[t, z, config.nuclear_channel_index], np.float32)[sl]
    npc = np.asarray(image_5d[t, z, config.npc_channel_index], np.float32)[sl]
    dpl = np.asarray(droplet_instance_4d[t, z])
    dl = _resolve_instance(dpl, cx, cy)
    droplet = (dpl == dl)[sl] if dl else np.ones(nls.shape, bool)

    if seed_override is not None:
        seed = seed_override & droplet
    else:
        inst = np.asarray(nucleus_instance_4d[t, z])
        nl = _resolve_instance(inst, cx, cy)
        seed = (inst == nl)[sl] if nl else np.zeros(nls.shape, bool)
    if not seed.any():
        return np.zeros(nls.shape, bool), {"reject": "no seed"}, (r0, c0)

    ly, lx = cy - r0, cx - c0
    r_um, n_rays = npc_ring_radius(npc, droplet, ly, lx, config)

    bound = droplet
    if np.isfinite(r_um) and n_rays >= config.repair_npc_min_rays:
        sy, sx = ndi.center_of_mass(seed)
        yy, xx = np.ogrid[:nls.shape[0], :nls.shape[1]]
        rp = (r_um * config.repair_npc_slack) / config.pixel_size_um
        cand = droplet & ((yy - sy) ** 2 + (xx - sx) ** 2 <= rp ** 2)
        if (seed & cand).any():
            bound = cand

    mask = seeded_watershed(seed, nls, bound, config,
                            flatten_um=config.repair_flatten_um)
    ok, why = signed_contrast_ok(mask, nls, droplet, config)
    px2 = config.pixel_size_um ** 2
    m = {"reject": "" if ok else why, "npc_r_um": r_um, "npc_n_rays": n_rays,
         "npc_area_um2": np.pi * r_um ** 2 if np.isfinite(r_um) else np.nan,
         "area_px": int(mask.sum()), "area_um2": float(mask.sum()) * px2}
    if ok and mask.any():
        rp_ = measure.regionprops(mask.astype(np.uint8))[0]
        m["solidity"] = float(rp_.solidity)
        m["centroid_y_px"] = r0 + rp_.centroid[0]
        m["centroid_x_px"] = c0 + rp_.centroid[1]
    else:
        mask = np.zeros(nls.shape, bool)
    return mask, m, (r0, c0)


def repair_fragmented_nuclei(best_z_df, grouped_z_df, image_5d,
                             nucleus_instance_4d, droplet_instance_4d,
                             config, verbose=True):
    """
    Detect fragmented nuclei, repair them, re-select z, return the corrected
    best-Z table.

    Overwrites `area_px`, `z`, and the centroids with the repaired values so
    every downstream stage consumes them unchanged. Originals are preserved as
    `*_original`, and provenance goes in `repair_status` / `repair_gain` /
    `ratio_vs_npc`.
    """
    px2 = config.pixel_size_um ** 2
    H, W = image_5d.shape[-2], image_5d.shape[-1]
    Z = image_5d.shape[1]
    em = max(int(round(config.repair_edge_margin_um / config.pixel_size_um)), 1)

    d = best_z_df.copy().reset_index(drop=True)
    d["area_px_original"] = d.area_px
    d["z_original"] = d.z
    d["nucleus_area_um2"] = d.area_px * px2

    # ---- edge status: droplet OR nucleus touching the frame ----
    def touches(arr, lab):
        if not lab:
            return True
        ys, xs = np.where(arr == lab)
        return bool(ys.size == 0 or ys.min() <= em or xs.min() <= em
                    or ys.max() >= H - 1 - em or xs.max() >= W - 1 - em)

    edge, sol = [], []
    for r in d.itertuples():
        inst = np.asarray(nucleus_instance_4d[int(r.t), int(r.z)])
        dpl = np.asarray(droplet_instance_4d[int(r.t), int(r.z)])
        nl = _resolve_instance(inst, r.centroid_x_px, r.centroid_y_px)
        dl = _resolve_instance(dpl, r.centroid_x_px, r.centroid_y_px)
        edge.append(touches(inst, nl) or touches(dpl, dl))
        if nl:
            rp = measure.regionprops((inst == nl).astype(np.uint8))[0]
            sol.append(float(rp.solidity))
        else:
            sol.append(np.nan)
    d["exclude_edge"] = edge
    d["solidity_original"] = sol

    pool = d[~d.exclude_edge]
    ref = pool.groupby("t").nucleus_area_um2.quantile(0.75).rename("_ref")
    d = d.join(ref, on="t")
    d["area_deficit"] = 1.0 - d.nucleus_area_um2 / d._ref
    d["is_fragmented"] = ((d.solidity_original < config.repair_min_solidity)
                          | (d.area_deficit > config.repair_area_deficit))
    d = d.drop(columns=["_ref"])

    for c in ("repair_gain", "ratio_vs_npc", "npc_r_um", "npc_area_um2",
              "npc_n_rays", "z_shift", "n_planes_repaired"):
        d[c] = np.nan
    d["repair_status"] = "not fragmented"

    if verbose:
        print(f"{len(d)} nuclei | {int(d.exclude_edge.sum())} at frame edge "
              f"| {int(d.is_fragmented.sum())} fragmented "
              f"({100*d.is_fragmented.mean():.1f}%)")

    todo = d.index[d.is_fragmented & ~d.exclude_edge]
    n_ok = n_fail = 0
    for i in todo:
        r = d.loc[i]
        t, z0 = int(r.t), int(r.z)
        cx, cy = float(r.centroid_x_px), float(r.centroid_y_px)
        mask0, m0, off0 = repair_plane(t, z0, cx, cy, image_5d,
                                       nucleus_instance_4d, droplet_instance_4d,
                                       config)
        if not mask0.any():
            d.at[i, "repair_status"] = f"rejected: {m0['reject']}"
            n_fail += 1
            continue

        best, best_z, best_mask, best_off, seen = m0, z0, mask0, off0, 1
        for direction in (-1, +1):
            seed = mask0
            for step in range(1, config.repair_z_pad + 1):
                z = z0 + direction * step
                if not (0 <= z < Z):
                    break
                sd = morphology.binary_erosion(seed, morphology.disk(3))
                if not sd.any():
                    break
                mk, mm, off = repair_plane(t, z, cx, cy, image_5d,
                                           nucleus_instance_4d,
                                           droplet_instance_4d, config,
                                           seed_override=sd)
                if not mk.any() or mm["area_px"] < 0.15 * best["area_px"]:
                    break
                seen += 1
                if mm["area_px"] > best["area_px"]:
                    best, best_z, best_mask, best_off = mm, z, mk, off
                seed = mk

        d.at[i, "area_px"] = best["area_px"]
        d.at[i, "nucleus_area_um2"] = best["area_um2"]
        d.at[i, "z"] = best_z
        d.at[i, "z_shift"] = best_z - z0
        d.at[i, "n_planes_repaired"] = seen
        if "centroid_x_px" in best:
            d.at[i, "centroid_x_px"] = best["centroid_x_px"]
            d.at[i, "centroid_y_px"] = best["centroid_y_px"]
        d.at[i, "repair_gain"] = best["area_px"] / max(r.area_px, 1)
        d.at[i, "npc_r_um"] = best["npc_r_um"]
        d.at[i, "npc_area_um2"] = best["npc_area_um2"]
        d.at[i, "npc_n_rays"] = best["npc_n_rays"]
        if np.isfinite(best["npc_area_um2"]) and best["npc_area_um2"] > 0:
            d.at[i, "ratio_vs_npc"] = best["area_um2"] / best["npc_area_um2"]
        d.at[i, "repair_status"] = "repaired"

        # ── persist the winning mask (full-frame) so halo/radial-sweep find it ──
        # recover_nucleus_mask_from_plane would otherwise reload the pre-repair
        # per-plane .npy and silently undo the repair (see Section 18/19).
        r0, c0 = best_off
        canvas = np.zeros((H, W), dtype=bool)
        bh, bw = best_mask.shape
        canvas[r0:r0 + bh, c0:c0 + bw] = best_mask
        out_path = _repaired_mask_path(config, t, r.nucleus_3d_id)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(out_path, canvas)

        n_ok += 1

    if verbose:
        rep = d[d.repair_status == "repaired"]
        print(f"  repaired {n_ok} | rejected {n_fail}")
        if n_fail:
            print(d.loc[todo, "repair_status"][
                d.loc[todo, "repair_status"].str.startswith("rejected")]
                .value_counts().head(5).to_string())
        if len(rep):
            print(f"  median gain {rep.repair_gain.median():.2f}x | "
                  f"z moved {int((rep.z_shift != 0).sum())}/{len(rep)} | "
                  f"median planes {rep.n_planes_repaired.median():.0f}")
            print(f"  ratio vs NPC ring {rep.ratio_vs_npc.median():.3f} "
                  f"(target 0.88-0.95)")
            if rep.ratio_vs_npc.median() > 1.0:
                print("  WARNING: above 1.0 — the mask is crossing the "
                      "envelope. Lower cfg.repair_npc_slack.")
            elif rep.ratio_vs_npc.median() < 0.85:
                print("  NOTE: below 0.85 — still under-recovering. Consider "
                      "raising cfg.repair_flatten_um.")
    return d

### 30c. Run the repair

Sits between best-Z selection (§30) and close-nuclei exclusion (§32), and
overwrites `best_z_df`, so §31 onward run unchanged.

In [ ]:
repaired_path = cfg.obj_dir / "best_z_nuclei_repaired.pkl"
RUN_FRAGMENTATION_REPAIR = True

if cfg.repair_enabled and RUN_FRAGMENTATION_REPAIR:
    require_segmentation(cfg, needs=("nucleus_instance_hyperstack",
                                    "droplet_instance_hyperstack"))
    t0 = time.perf_counter()
    nucleus_instance_4d = tiff.memmap(cfg.nucleus_instance_hyperstack_path, mode="r")
    droplet_instance_4d = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
    best_z_df = repair_fragmented_nuclei(best_z_df, grouped_z_df, img_5d,
                                         nucleus_instance_4d,
                                         droplet_instance_4d, cfg)
    best_z_df.to_pickle(repaired_path)
    print(f"Fragmentation repair finished in {time.perf_counter() - t0:.2f}s")
elif cfg.repair_enabled:
    best_z_df = pd.read_pickle(repaired_path)
    print("Loaded repaired best-Z table.")
else:
    print("cfg.repair_enabled is False — using unrepaired best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())

### 30d. Repair QC

Read `ratio_vs_npc` first — it is the only reference independent of the NLS
channel. The scatter should straddle 0.9, not sit at 1.2 (crossing the
envelope) or 0.6 (still fragmented).

In [ ]:
def plot_repair_qc(d, config, time_col=None, figsize=(16, 4.6)):
    dd = d[~d.exclude_edge].copy()
    if time_col is None:
        time_col = "true_time_min" if "true_time_min" in dd.columns else "t"
    frag = dd.is_fragmented
    if "nucleus_area_um2_original" not in dd:
        dd["nucleus_area_um2_original"] = dd.area_px_original * config.pixel_size_um ** 2

    fig, ax = plt.subplots(1, 3, figsize=figsize)
    for a, col, ttl in ((ax[0], "nucleus_area_um2_original", "BEFORE"),
                        (ax[1], "nucleus_area_um2", "AFTER — repaired")):
        a.scatter(dd.loc[~frag, time_col], dd.loc[~frag, col], s=11, alpha=0.45,
                  c="tab:blue", label="never fragmented")
        a.scatter(dd.loc[frag, time_col], dd.loc[frag, col], s=15, alpha=0.75,
                  c="tab:red", label="fragmented")
        a.set_title(ttl); a.set_xlabel(time_col); a.grid(alpha=0.25)
        a.legend(fontsize=8)
    ax[0].set_ylabel("nuclear area (µm²)")
    ax[1].set_ylim(ax[0].get_ylim())

    rep = dd[dd.repair_status == "repaired"]
    a = ax[2]
    if len(rep):
        a.scatter(rep.npc_area_um2, rep.nucleus_area_um2, s=14, alpha=0.6)
        lim = [0, float(rep.npc_area_um2.quantile(0.99))]
        a.plot(lim, lim, "k--", lw=1, label="equal")
        a.plot(lim, [0.9 * x for x in lim], "g--", lw=1, label="0.90 (expected)")
        a.set_xlabel("NPC ring area (µm²)")
        a.set_ylabel("repaired NLS area (µm²)")
        a.set_title(f"vs the independent reference "
                    f"(median {rep.ratio_vs_npc.median():.2f})")
        a.legend(fontsize=8); a.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    print(f"{'':<28}{'before':>10}{'after':>10}")
    print(f"{'median area, fragmented':<28}"
          f"{dd[frag].nucleus_area_um2_original.median():>10.0f}"
          f"{dd[frag].nucleus_area_um2.median():>10.0f}")
    print(f"{'median area, all':<28}"
          f"{dd.nucleus_area_um2_original.median():>10.0f}"
          f"{dd.nucleus_area_um2.median():>10.0f}")
    print(f"{'never-fragmented median':<28}"
          f"{dd[~frag].nucleus_area_um2_original.median():>10.0f}")
    return dd


if cfg.repair_enabled:
    _ = plot_repair_qc(best_z_df, cfg)

## 31. Optional area-based cleanup

In [ ]:
def filter_nuclei_by_area(
    df: pd.DataFrame,
    config: PipelineConfig,
    min_area_um2: float = 100.0,
    max_area_um2: float = 600.0,
) -> pd.DataFrame:
    """Remove objects that are too small or too large to be real nuclei."""
    df = df.copy()
    if "nucleus_area_um2" not in df.columns:
        df["nucleus_area_um2"] = df["area_px"] * config.pixel_size_um ** 2
    filtered = df[(df["nucleus_area_um2"] >= min_area_um2) & (df["nucleus_area_um2"] <= max_area_um2)]
    print(f"Filtered: {len(df)} → {len(filtered)} nuclei")
    return filtered


# Optional debug usage:
# v17: was (100, 300) while the function default is (100, 600) --
# the debug call silently used a different ceiling than the docstring.
best_z_area_filtered_df = filter_nuclei_by_area(best_z_df, cfg, 100, 600)
display(best_z_area_filtered_df.head())


## 32. Run or load close-nuclei exclusion

In [ ]:
filtered_path = cfg.obj_dir / "best_z_nuclei_with_exclusion.pkl"

if RUN_FLAG_CLOSE_NUCLEI:
    t0 = time.perf_counter()
    filtered_df = flag_close_nuclei(best_z_df, cfg)
    print(f"Close-nuclei flagging finished in {time.perf_counter() - t0:.2f}s")
else:
    filtered_df = pd.read_pickle(filtered_path)
    print("Loaded exclusion table.")

print(filtered_df.shape)
display(filtered_df.head())


## 33. Run or load timing metadata

In [ ]:
timed_path = cfg.track_dir / "best_z_nuclei_timed.pkl"

if RUN_ADD_TIMING:
    t0 = time.perf_counter()
    timed_df = add_tile_timing_metadata(filtered_df, img_5d, cfg)
    print(f"Timing metadata finished in {time.perf_counter() - t0:.2f}s")
else:
    timed_df = pd.read_pickle(timed_path)
    print("Loaded timed nuclei table.")

print(timed_df.shape)
display(timed_df.head())


## 34. Run or load tracking

In [ ]:
tracked_path = cfg.track_dir / "tracked_nuclei.pkl"

if RUN_TRACKING:
    t0 = time.perf_counter()
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg, valid_only=False)
    print(f"Hybrid DBSCAN tracking finished in {time.perf_counter() - t0:.2f}s")
else:
    tracked_df = pd.read_pickle(tracked_path)
    print("Loaded tracked nuclei table.")

print(tracked_df.shape)
display(tracked_df.head())
display(summarize_tracking_debug(tracked_df))


## 35. Run or load halo analysis

In [ ]:
halo_path = cfg.analysis_dir / "halo_analysis.pkl"

if RUN_HALO_ANALYSIS:
    t0 = time.perf_counter()
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Halo analysis finished in {time.perf_counter() - t0:.2f}s")
else:
    halo_df = pd.read_pickle(halo_path)
    print("Loaded halo analysis table.")

print(halo_df.shape)
display(halo_df.head())


## 35b. Run or load radial sweep (droplet-bounded)

Runs on the same `tracked_df` as the halo analysis (best-Z, one row per nucleus per timepoint). Requires the droplet instance hyperstack from segmentation. Output is long-form (`radial_df`), pickled to `analysis_dir/radial_sweep.pkl`.

In [ ]:
radial_sweep_path = cfg.analysis_dir / "radial_sweep.pkl"

if RUN_RADIAL_SWEEP:
    t0 = time.perf_counter()
    radial_df = run_radial_sweep_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Radial sweep finished in {time.perf_counter() - t0:.2f}s")
else:
    radial_df = pd.read_pickle(radial_sweep_path)
    print("Loaded radial sweep table.")

print(radial_df.shape)
display(radial_df.head())


In [ ]:
radial_sweep_path = cfg.analysis_dir / "radial_sweep.pkl"

if RUN_RADIAL_SWEEP:
    t0 = time.perf_counter()
    radial_df = run_radial_sweep_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Radial sweep finished in {time.perf_counter() - t0:.2f}s")
else:
    radial_df = pd.read_pickle(radial_sweep_path)
    print("Loaded radial sweep table.")

print(radial_df.shape)
display(radial_df.head())


In [ ]:
radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")
print(radial_df.shape)
radial_df.head()

In [ ]:
import numpy as np, pandas as pd

radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")
print(radial_df.shape)
print(radial_df.columns.tolist())
print(radial_df["t"].value_counts().sort_index())

px = cfg.pixel_size_um
out = cfg.exports_dir   # created by the resolver patch

# (1) Mean radial intensity profile: median intensity vs absolute radius, per t
prof = (radial_df
        .assign(r_um=(radial_df.distance_px * px).round(1))
        .groupby(["t", "r_um"])["intensity"]
        .agg(median="median", mean="mean", n="size")
        .reset_index())
prof.to_csv(out / "radial_profile_binned.csv", index=False)

# (2) Per-nucleus envelope peak: radius of max membrane intensity per ray,
#     then median across rays for each nucleus. This is the mask-independent
#     nuclear radius estimate.
peak = (radial_df.loc[radial_df.groupby(["t","track_id","theta_index"])["intensity"].idxmax()]
        .groupby(["t","track_id"])
        .agg(peak_r_um=("distance_px", lambda s: float(np.median(s)) * px),
             peak_intensity=("intensity", "median"),
             n_rays=("theta_index", "nunique"))
        .reset_index())
peak["implied_area_um2"] = np.pi * peak.peak_r_um**2
peak.to_csv(out / "radial_envelope_peaks.csv", index=False)

print(peak.groupby("t")[["peak_r_um","implied_area_um2"]].median().round(1))
print([f"{p.name}: {p.stat().st_size/1e6:.1f}MB" for p in out.glob("radial_*.csv")])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

for t, d in prof.groupby("t"):
    ax[0].plot(d.r_um, d["median"], label=f"t={t}", lw=1.4)
ax[0].axvline(6.2, ls="--", c="k", lw=1, label="current mask R")
ax[0].set_xlabel("distance from centroid (µm)")
ax[0].set_ylabel("median membrane intensity")
ax[0].set_title("Radial profile by timepoint")
ax[0].legend(fontsize=7, ncol=2)

late = peak[peak.t >= 7]
ax[1].hist(late.peak_r_um, bins=40)
ax[1].axvline(6.2, ls="--", c="k", lw=1, label="current mask R")
ax[1].axvline(11.3, ls="--", c="r", lw=1, label="expected R (400 µm²)")
ax[1].set_xlabel("envelope peak radius (µm)")
ax[1].set_ylabel("nuclei")
ax[1].set_title("Peak radius, t≥7")
ax[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
plane = np.asarray(drop[9, 15])          # late t, in-focus z
d = pd.DataFrame(measure.regionprops_table(plane, properties=("area","equivalent_diameter")))
d = d[d.area > 1000]
med_px = d.equivalent_diameter.median()
print("droplet diameter (px):", round(med_px, 1))
print("droplet diameter (µm) @0.108:", round(med_px * cfg.pixel_size_um, 1))
print("implied px size if droplets are 65 µm:", round(65 / med_px, 4))


In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
b = pd.read_pickle(cfg.obj_dir / "best_z_nuclei.pkl")

rows = []
for (t, z), grp in b[b.t >= 7].groupby(["t", "z"]):
    plane = np.asarray(drop[t, z])
    props = {r.label: r.area for r in measure.regionprops(plane)}
    for r in grp.itertuples():
        did = plane[int(r.centroid_y_px), int(r.centroid_x_px)]
        if did > 0 and props.get(did, 0) > 0:
            rows.append({"t": t, "nuc_px": r.area_px, "drop_px": props[did],
                         "ratio": r.area_px / props[did]})

d = pd.DataFrame(rows)
print("n nuclei matched:", len(d))
print("median droplet area (px):", round(float(d.drop_px.median())))
print("median droplet diameter (px):", round(float(2*np.sqrt(d.drop_px.median()/np.pi)), 1))
print("median nucleus/droplet area ratio:", round(float(d.ratio.median()), 4))

D_UM = 65.0   # <-- set to your actual RAN008 droplet diameter
diam_px = float(2*np.sqrt(d.drop_px.median()/np.pi))
print(f"\nif droplets are {D_UM} µm diameter -> nucleus area:",
      round(float(d.ratio.median()) * np.pi * (D_UM/2)**2, 1), "µm²")
print("implied pixel size:", round(D_UM / diam_px, 4), "µm/px")

In [ ]:
class_mask_5d       = tiff.imread(cfg.segmentation_class_hyperstack_path)
label_mask_4d       = tiff.imread(cfg.segmentation_label_hyperstack_path)
nucleus_instance_4d = tiff.imread(cfg.nucleus_instance_hyperstack_path)

print(f"class_mask_5d shape:       {class_mask_5d.shape}")
print(f"label_mask_4d shape:       {label_mask_4d.shape}")
print(f"nucleus_instance_4d shape: {nucleus_instance_4d.shape}")

nucleus_mask_4d = class_mask_5d[:, :, cfg.nucleus_class_index] > 0
print(f"nucleus_mask_4d shape:     {nucleus_mask_4d.shape}")


In [ ]:
plot_nucleus_mask_overlay_qc(img_5d, nucleus_mask_4d, cfg, z_mode="max_mask")
plot_nucleus_mask_qc_summary(nucleus_mask_4d, z_mode="max_mask")
plot_nucleus_count_over_time(nucleus_mask_4d)


## 36. Figures — analysis plot set

Everything below runs off the tables the pipeline already writes; nothing here
re-reads the image stack. Each figure is saved to `<run>/exports/figures/` as
PNG + PDF, so a run is reproducible from its own directory.

| figure | function | source table |
|---|---|---|
| N/C ratio, area vs time | `plot_nc_vs_time`, `plot_area_vs_time` | `halo_df` |
| rose — envelope radius by angle | `plot_rose_individual`, `plot_rose_pooled` | `radial_df` |
| membrane intensity by angle × distance | `plot_membrane_angle_distance` | `radial_df` |
| membrane asymmetry vs time | `plot_membrane_asymmetry_vs_time` | `radial_df` + `tracked_df` |
| cross-sectional area, 6-min boxes | `plot_area_boxplot_binned` | `halo_df` / `timed_df` |

**Three things changed relative to the scratch versions further down the
notebook**, each flagged in the section it affects:

1. `distance_norm` in `radial_sweep.pkl` is *not* the nucleus-surface-to-wall
   coordinate it was being plotted as (§36a).
2. An unrestricted per-ray `argmax` for the envelope peak finds the droplet
   interface, not the nuclear envelope (§36b).
3. Polar axes are now drawn in the image frame, so a rose can be laid against
   a raw crop without mirroring (§36b).

The scratch cells in §37+ are superseded by this section and can be deleted
once you're happy with the output.

In [ ]:
# ── Section 36 parameters ─────────────────────────────────────────────────
# Peak search is capped at this fraction of the nucleus->wall gap. See 36b for
# why: the membrane channel also images the droplet interface, which outshines
# the NE. Raise toward 1.0 only after checking frac_at_search_edge.
ENVELOPE_SEARCH_RHO_MAX = 0.6

# Perinuclear shell used for the asymmetry metrics, in microns outward from the
# nuclear mask edge.
ASYMMETRY_BAND_UM = (0.0, 3.0)

# Width of the acquisition-time bins for the box plots, in minutes.
# 2x3 tiles at 1 min/tile => 6 min per frame, so 6.0 == one frame per box.
AREA_BIN_MIN = 6.0

In [ ]:
# ── Bind the tables this section plots, loading from the current run if the
# kernel doesn't already hold them. Lets Section 36 run against a finished
# run without re-executing the pipeline above.
for _name, _path in [
    ("halo_df",    cfg.analysis_dir / "halo_analysis.pkl"),
    ("timed_df",   cfg.track_dir    / "best_z_nuclei_timed.pkl"),
    ("tracked_df", cfg.track_dir    / "tracked_nuclei.pkl"),
    ("radial_df",  cfg.analysis_dir / "radial_sweep.pkl"),
]:
    if _name not in dir():
        globals()[_name] = pd.read_pickle(_path)
        print(f"Loaded {_name} from disk.")

print({k: globals()[k].shape for k in ("halo_df", "timed_df", "tracked_df", "radial_df")})

In [ ]:
from typing import Sequence  # not in the Section 1 import block

SAVE_FIGURES = True
FIG_FORMATS = ("png", "pdf")
FIG_DPI = 300


def figures_dir(config) -> Path:
    d = config.exports_dir / "figures"
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_figure(fig, name: str, config, formats=FIG_FORMATS, dpi: int = FIG_DPI) -> List[Path]:
    """Write a figure to <run>/exports/figures/<name>.<ext> for each format."""
    if not SAVE_FIGURES:
        return []
    out = []
    d = figures_dir(config)
    for ext in formats:
        p = d / f"{name}.{ext}"
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        out.append(p)
    return out


def resolve_area_um2(df: pd.DataFrame, config, out_col: str = "nucleus_area_um2") -> pd.DataFrame:
    """Guarantee an area column in um^2, recomputed from pixels when possible.

    Recomputing from *_area_px means the figure always reflects the live
    pixel_size_um rather than whatever calibration was in force when the
    table was pickled.
    """
    df = df.copy()
    px2 = config.pixel_size_um ** 2
    if "nucleus_area_px" in df.columns:
        df[out_col] = df["nucleus_area_px"] * px2
    elif "area_px" in df.columns:
        df[out_col] = df["area_px"] * px2
    elif out_col in df.columns:
        pass
    else:
        raise ValueError("need one of nucleus_area_px / area_px / " + out_col)
    return df


def attach_true_time(df: pd.DataFrame, tracked_df: pd.DataFrame) -> pd.DataFrame:
    """Merge true_time_min onto a table keyed by (t, track_id).

    radial_df carries only (t, track_id, ...) -- acquisition time lives on
    tracked_df / timed_df.
    """
    if "true_time_min" in df.columns:
        return df
    if tracked_df is None or "true_time_min" not in tracked_df.columns:
        out = df.copy()
        out["true_time_min"] = out["t"].astype(float)
        return out
    lut = (tracked_df[["t", "track_id", "true_time_min"]]
           .drop_duplicates(subset=["t", "track_id"]))
    return df.merge(lut, on=["t", "track_id"], how="left")


def _time_bins(time_min: pd.Series, bin_min: float) -> Tuple[pd.Series, np.ndarray]:
    """Left edge of each observation's bin, plus the sorted unique edges."""
    lo = np.floor(np.asarray(time_min, dtype=float) / bin_min) * bin_min
    return pd.Series(lo, index=time_min.index), np.unique(lo[np.isfinite(lo)])


def _close_polar(theta: np.ndarray, r: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Repeat the first point at theta+2pi so a polar curve closes."""
    theta = np.asarray(theta, dtype=float)
    r = np.asarray(r, dtype=float)
    if theta.size == 0:
        return theta, r
    return np.append(theta, theta[0] + 2 * np.pi), np.append(r, r[0])


def _orient_polar(ax) -> None:
    """Put a polar axis in the same frame as the displayed image.

    The sweep casts rays as y = cy + r*sin(theta), x = cx + r*cos(theta). Image
    y increases DOWNWARD, so theta increases clockwise on screen and theta=0
    points right (+x). Matplotlib's polar default is zero at east but
    counter-clockwise, which mirrors the nucleus about the horizontal axis --
    fine for a shape summary, wrong the moment you compare a rose against a
    raw crop or against phi_deg from the asymmetry table.
    """
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(-1)


def _t_colors(all_t: Sequence) -> Dict:
    cmap = plt.get_cmap("viridis")
    n = max(len(all_t) - 1, 1)
    return {t: cmap(i / n) for i, t in enumerate(sorted(all_t))}

### 36a. Radius coordinate — `rho_wall`

`radial_profile_for_tracked_nucleus` writes `distance_norm = distance_px /
ray_length_px`, which is a fraction of the **centroid**-to-wall distance. Since
`radial_exclude_nucleus=True` drops interior samples, the smallest value on a
ray is `R_nucleus / R_droplet` — around 0.17 on this dataset, not 0. Plotting it
on an axis labelled *nucleus edge → droplet edge* therefore mislabels the origin,
and worse, the same `distance_norm` value means a different physical distance
from the envelope for a small nucleus than for a large one, so binning it pools
non-comparable points as nuclei grow.

`add_wall_normalised_radius` recomputes the schema's `Rho_Normalized` per ray —
0 at the nuclear surface, 1 at the wall — and also adds `r_um` and
`r_from_edge_um` for work in absolute microns. It does not modify
`radial_sweep.pkl`; it returns an augmented copy.

In [ ]:
def add_wall_normalised_radius(radial_df: pd.DataFrame, config,
                               out_col: str = "rho_wall") -> pd.DataFrame:
    """Add a radius coordinate that runs 0 at the nuclear surface -> 1 at the wall.

    `distance_norm` as written by the sweep is distance_px / ray_length_px, i.e.
    a fraction of the CENTROID-to-wall distance. Because the sweep skips samples
    inside the nucleus, the smallest value on a ray is R_nucleus / R_droplet
    (~0.2-0.5), not 0 -- so a plot with `distance_norm` on the x axis labelled
    "nucleus edge -> droplet edge" is mis-labelled, and the effective bin width
    differs between a big nucleus and a small one.

    This recomputes the schema's Rho_Normalized per ray:

        rho_wall = (r - r_edge) / (r_wall - r_edge)

    r_edge is the last sample inside the nucleus when the sweep kept interior
    points, otherwise the first retained sample (which sits just outside the
    mask boundary). Also adds r_um and r_from_edge_um for absolute-distance work.
    """
    if radial_df is None or radial_df.empty:
        return radial_df

    df = radial_df.copy()
    keys = ["t", "track_id", "theta_index"]
    g = df.groupby(keys)["distance_px"]

    r_first = g.transform("min")
    if "inside_nucleus" in df.columns and bool(df["inside_nucleus"].any()):
        inside_d = df["distance_px"].where(df["inside_nucleus"])
        r_edge = inside_d.groupby([df[k] for k in keys]).transform("max")
        r_edge = r_edge.fillna(r_first)
    else:
        r_edge = r_first

    r_wall = (df["ray_length_px"] if "ray_length_px" in df.columns
              else g.transform("max"))

    gap = (r_wall - r_edge).replace(0, np.nan)
    df[out_col] = ((df["distance_px"] - r_edge) / gap).clip(0.0, 1.0)
    df["r_um"] = df["distance_px"] * config.pixel_size_um
    df["r_edge_um"] = r_edge * config.pixel_size_um
    df["r_from_edge_um"] = df["r_um"] - df["r_edge_um"]
    df["theta_deg"] = np.degrees(df["theta_rad"]) % 360.0
    return df

### 36b. Rose plots — envelope radius by angle

For each ray, the radius at which the NE channel peaks — a mask-independent
estimate of nuclear shape, and the one that shows whether growth is isotropic.

**The peak search is capped** at `rho_wall <= search_rho_max` (default 0.6).
The radial channel is Ch0, which images the droplet oil–water interface as well
as the envelope, and on this rig the interface is usually the brightest thing on
the ray. An uncapped `argmax` therefore returns the droplet boundary for any
nucleus whose NE signal is weaker than its own droplet edge, and the rose plot
becomes a picture of the droplet — which is what the earlier scratch cell was
showing when its peak-radius histogram sat near the droplet radius instead of
the expected nuclear radius.

Watch `frac_at_search_edge` in the printout. If it's high, the cap rather than
the data is picking the peak and you should look at a radial profile directly
before trusting the rose.

Polar axes use `_orient_polar`: zero at east, clockwise. The sweep casts rays
with `y = cy + r·sin θ` and image y increases downward, so θ runs clockwise on
screen — matplotlib's counter-clockwise default silently mirrors the nucleus.

In [ ]:
def compute_envelope_radius_by_angle(
    radial_df: pd.DataFrame,
    config,
    search_rho_max: float = 0.6,
    verbose: bool = True,
) -> pd.DataFrame:
    """Per (t, track_id, ray): the radius at which the NE channel peaks.

    The search is restricted to rho_wall <= search_rho_max. The radial channel
    is Ch0, which images the droplet oil-water interface as well as the nuclear
    envelope, and the interface is usually the brightest thing on the ray -- an
    unrestricted argmax returns the droplet wall for any nucleus whose NE signal
    is weaker than its own droplet boundary, which turns the rose plot into a
    picture of the droplet. Pass search_rho_max=1.0 for the unrestricted
    behaviour, and watch `frac_at_search_edge` in the printout: a high value
    means the cap is doing the deciding, not the data.
    """
    cols = ["t", "track_id", "theta_index", "theta_rad", "peak_r_um",
            "peak_intensity", "rho_at_peak", "at_search_edge"]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    search = df[df["rho_wall"] <= float(search_rho_max)]
    if search.empty:
        return pd.DataFrame(columns=cols)

    idx = search.groupby(["t", "track_id", "theta_index"])["intensity"].idxmax()
    peaks = search.loc[idx, ["t", "track_id", "theta_index", "theta_rad",
                             "distance_px", "intensity", "rho_wall"]].copy()
    peaks = peaks.rename(columns={"intensity": "peak_intensity",
                                  "rho_wall": "rho_at_peak"})
    peaks["peak_r_um"] = peaks["distance_px"] * config.pixel_size_um
    peaks["at_search_edge"] = peaks["rho_at_peak"] >= 0.98 * float(search_rho_max)
    peaks = peaks.drop(columns="distance_px").reset_index(drop=True)[cols]

    if verbose:
        frac = float(peaks["at_search_edge"].mean())
        print(f"envelope peaks: {len(peaks)} rays, "
              f"search_rho_max={search_rho_max}, "
              f"frac_at_search_edge={frac:.3f}")
        print(peaks.groupby("t")["peak_r_um"].median().round(2).to_string())
    return peaks


def plot_rose_individual(peak_df: pd.DataFrame, tracked_df: pd.DataFrame, config,
                         n: int = 5, selection: str = "longest",
                         share_r: bool = True, save_as: Optional[str] = "rose_individual"):
    """Polar envelope-shape plot for n example nuclei, one curve per timepoint."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return []

    if selection == "longest":
        counts = (tracked_df.groupby("track_id")["t"].nunique()
                  .sort_values(ascending=False))
        track_ids = [tid for tid in counts.index if tid in set(peak_df["track_id"])][:n]
    else:
        track_ids = sorted(peak_df["track_id"].unique())[:n]
    if not track_ids:
        print("No tracks in common between peak_df and tracked_df."); return []

    n_cols = min(5, len(track_ids))
    n_rows = math.ceil(len(track_ids) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.6 * n_cols, 3.9 * n_rows),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    axes = axes.ravel()
    colors = _t_colors(peak_df["t"].unique())
    r_max = float(peak_df[peak_df.track_id.isin(track_ids)]["peak_r_um"].quantile(0.99))

    for ax, tid in zip(axes, track_ids):
        d = peak_df[peak_df.track_id == tid]
        for t in sorted(d["t"].unique()):
            dt = d[d.t == t].sort_values("theta_index")
            th, r = _close_polar(dt.theta_rad.to_numpy(), dt.peak_r_um.to_numpy())
            ax.plot(th, r, color=colors[t], lw=1.2, label=f"t={t}")
        ax.set_title(f"track {tid}", fontsize=10)
        _orient_polar(ax)
        ax.tick_params(labelsize=7)
        if share_r:
            ax.set_ylim(0, r_max)

    for ax in axes[len(track_ids):]:
        ax.axis("off")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=7,
               title="frame", title_fontsize=7)
    fig.suptitle("Envelope radius by angle (\u00b5m) \u2014 individual nuclei", fontsize=13)
    fig.tight_layout(rect=(0, 0, 0.90, 1))
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return track_ids


def plot_rose_pooled(peak_df: pd.DataFrame, config, show_iqr: bool = True,
                     min_rays: int = 5, save_as: Optional[str] = "rose_pooled"):
    """Population-pooled rose: radius = median peak_r_um over nuclei per (t, theta)."""
    if peak_df is None or peak_df.empty:
        print("No data to plot."); return

    pooled = (peak_df.groupby(["t", "theta_index"])
              .agg(theta_rad=("theta_rad", "first"),
                   med=("peak_r_um", "median"),
                   q1=("peak_r_um", lambda s: s.quantile(0.25)),
                   q3=("peak_r_um", lambda s: s.quantile(0.75)),
                   n=("peak_r_um", "size"))
              .reset_index())
    pooled = pooled[pooled["n"] >= min_rays]
    if pooled.empty:
        print(f"No (t, theta) bin reached min_rays={min_rays}."); return

    fig, ax = plt.subplots(figsize=(6.4, 6.4), subplot_kw={"projection": "polar"})
    colors = _t_colors(pooled["t"].unique())
    for t in sorted(pooled["t"].unique()):
        dt = pooled[pooled.t == t].sort_values("theta_index")
        th, r = _close_polar(dt.theta_rad.to_numpy(), dt["med"].to_numpy())
        ax.plot(th, r, color=colors[t], lw=1.6, label=f"t={t}")
        if show_iqr:
            _, lo = _close_polar(dt.theta_rad.to_numpy(), dt["q1"].to_numpy())
            th2, hi = _close_polar(dt.theta_rad.to_numpy(), dt["q3"].to_numpy())
            ax.fill_between(th2, lo, hi, color=colors[t], alpha=0.12, lw=0)
    _orient_polar(ax)
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.05, 1.05), title="frame")
    ax.set_title("Envelope radius by angle (\u00b5m)\nmedian \u00b1 IQR, pooled over nuclei",
                 fontsize=12, pad=18)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return pooled

### 36c. Membrane intensity by angle and distance

The θ × r intensity map, one panel per frame: the raw evidence behind both the
rose and the asymmetry index. A complete envelope is a continuous bright band at
r ≈ 0 spanning all 360°; a partial one is a band with gaps; directional delivery
is a band that is brighter over a contiguous arc.

`radius_col="rho_wall"` spans the full gap and shows the droplet wall as a
second bright band at r = 1. Because the wall usually outshines the envelope, it
compresses the NE contrast — pass `radius_col="r_from_edge_um"` with
`r_max_um≈8` to crop to the perinuclear zone, which is the more readable view
for membrane delivery.

Empty (θ, r) cells are drawn in grey rather than interpolated, and the function
warns when more than 5 % of the grid is empty — that means `rbins` is finer than
the ray sampling supports.

In [ ]:
def build_angle_distance_grid(radial_df: pd.DataFrame, config, rbins: int = 60,
                              radius_col: str = "rho_wall",
                              r_max_um: Optional[float] = None,
                              agg: str = "mean") -> Dict:
    """theta x radius intensity grids, one per timepoint.

    radius_col='rho_wall'        -> 0 at the nuclear surface, 1 at the droplet wall
    radius_col='r_from_edge_um'  -> absolute microns outward from the nuclear surface
    """
    if radial_df is None or radial_df.empty:
        return {}
    df = radial_df
    if radius_col not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if radius_col == "rho_wall":
        edges = np.linspace(0.0, 1.0, rbins + 1)
    else:
        hi = float(r_max_um if r_max_um is not None
                   else np.nanpercentile(df[radius_col], 99))
        edges = np.linspace(0.0, hi, rbins + 1)

    d = df.copy()
    d["rbin"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["rbin"].notna()]

    n_ang = int(config.radial_n_angles)
    grids = {}
    for t, sub in d.groupby("t"):
        grids[int(t)] = (sub.pivot_table(index="theta_index", columns="rbin",
                                         values="intensity", aggfunc=agg)
                         .reindex(index=range(n_ang), columns=range(rbins))
                         .to_numpy())
    return {"grids": grids, "edges": edges, "radius_col": radius_col}


def plot_membrane_angle_distance(radial_df: pd.DataFrame, config,
                                 track_id: Optional[int] = None,
                                 rbins: int = 60,
                                 radius_col: str = "rho_wall",
                                 r_max_um: Optional[float] = None,
                                 clip_pct: Tuple[float, float] = (2, 98),
                                 cmap: str = "inferno",
                                 max_cols: int = 5,
                                 save_as: Optional[str] = None):
    """Membrane intensity as a function of angle and distance, one panel per frame.

    track_id=None pools every nucleus in absolute lab angle. Pooling is only
    meaningful if asymmetry shares a direction across nuclei -- check panel (c)
    of the asymmetry figure first; if the population direction is random, the
    pooled map washes out and the single-nucleus view is the informative one.
    """
    if radial_df is None or radial_df.empty:
        print("No data to plot."); return

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    if track_id is None:
        span = df.groupby("track_id")["t"].nunique().sort_values(ascending=False)
        label = "pooled over all nuclei"
        sub = df
    else:
        sub = df[df.track_id == int(track_id)]
        label = f"track {int(track_id)}"
        if sub.empty:
            print(f"track {track_id} not present in radial_df."); return

    built = build_angle_distance_grid(sub, config, rbins=rbins,
                                      radius_col=radius_col, r_max_um=r_max_um)
    grids, edges = built["grids"], built["edges"]
    if not grids:
        print("No data to plot."); return

    stacked = np.concatenate([g.ravel() for g in grids.values()])
    allv = stacked[np.isfinite(stacked)]
    if allv.size == 0:
        print("Grid is entirely empty."); return
    vmin, vmax = np.percentile(allv, clip_pct)

    empty_frac = 1.0 - allv.size / stacked.size
    if empty_frac > 0.05:
        print(f"WARNING: {empty_frac:.1%} of (theta, r) cells have no sample. "
              f"Lower rbins (currently {rbins}) or raise config.radial_step_size_px "
              f"resolution -- empty cells are drawn in grey, not interpolated.")

    cmap_obj = plt.get_cmap(cmap).copy()
    cmap_obj.set_bad("0.88")

    ts = sorted(grids)
    ncol = min(max_cols, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow),
                             squeeze=False, constrained_layout=True)
    xlabel = ("r (nuclear surface \u2192 droplet wall)" if radius_col == "rho_wall"
              else "distance from nuclear surface (\u00b5m)")
    for ax, t in zip(axes.ravel(), ts):
        im = ax.imshow(np.ma.masked_invalid(grids[t]), aspect="auto", origin="lower",
                       extent=[edges[0], edges[-1], 0, 360],
                       vmin=vmin, vmax=vmax, cmap=cmap_obj)
        ax.set_title(f"t={t}", fontsize=9)
        ax.set_xlabel(xlabel, fontsize=7)
        ax.set_ylabel("\u03b8 (deg)", fontsize=7)
        ax.set_yticks([0, 90, 180, 270, 360])
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.75, label="membrane intensity (a.u.)")
    fig.suptitle(f"Membrane intensity by angle and distance \u2014 {label}", fontsize=12)
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return built

### 36d. Membrane asymmetry vs time

`I(θ)` is the mean membrane intensity in a shell just outside the nuclear
surface (default 0–3 µm from the mask edge). Fitting the first Fourier harmonic

$$I(\theta) \approx A_0 + A_1\cos(\theta - \phi)$$

gives the headline metric **`asym_index` = A₁/A₀** — 0 for a uniform shell,
rising as the signal becomes one-sided — and **`phi_deg`**, the direction of the
bright side. `cv` and `lobe_ratio` are reported alongside because a single
harmonic is blind to two-lobed asymmetry; if `cv` climbs while `asym_index` does
not, the shell is patchy rather than polarised.

**Panel (c) is the control to read first.** It asks whether every nucleus is
bright on the *same* side of the image, via the circular resultant `R` of the
population's `phi` values against a Rayleigh threshold. Stochastic
dynein-mediated delivery should be strong per nucleus and randomly oriented
across them, giving R near zero. R above the threshold means the bright side has
a preferred lab-frame direction — which is the signature of an illumination
gradient, a tilted coverslip, or a systematic z-offset, not of biology. Rule
that out before interpreting panel (a).

A₁/A₀ > 1 is possible and means the harmonic fit implies negative intensity
somewhere — read it as "essentially all the signal is on one side".

In [ ]:
def _first_harmonic(theta: np.ndarray, values: np.ndarray) -> Tuple[float, float, float]:
    """Fit v(theta) ~ a0 + a1*cos(theta - phi); return (a0, a1, phi).

    phi is the angle of maximum signal. Uniform theta sampling assumed.
    """
    theta = np.asarray(theta, dtype=float)
    v = np.asarray(values, dtype=float)
    ok = np.isfinite(v) & np.isfinite(theta)
    if ok.sum() < 8:
        return np.nan, np.nan, np.nan
    theta, v = theta[ok], v[ok]
    a0 = float(v.mean())
    c1 = np.mean(v * np.exp(-1j * theta))
    return a0, float(2 * np.abs(c1)), float(-np.angle(c1))


def compute_membrane_asymmetry(radial_df: pd.DataFrame, config,
                               tracked_df: Optional[pd.DataFrame] = None,
                               band: Tuple[float, float] = (0.0, 3.0),
                               band_units: str = "um",
                               min_rays: int = 30) -> pd.DataFrame:
    """Per (t, track_id) asymmetry of the perinuclear membrane signal.

    The signal is I(theta) = mean intensity in a shell just outside the nuclear
    surface -- by default 0-3 microns from the mask edge, which is where
    delivered membrane accumulates. Set band_units='rho' to use a fraction of
    the nucleus-to-wall gap instead (scale-free but droplet-size dependent).

    Columns
    -------
    a0              mean perinuclear intensity around the nucleus
    a1              amplitude of the first Fourier harmonic
    asym_index      a1 / a0, the headline number: 0 = uniform shell,
                    ~0.3+ = clearly one-sided
    phi_deg         direction of the bright side, degrees (image frame, CCW from +x)
    cv              std/mean across angle -- catches multi-lobed asymmetry that
                    a single harmonic misses
    lobe_ratio      brightest decile of angles / dimmest decile
    n_rays          rays contributing
    """
    empty = pd.DataFrame(columns=["t", "track_id", "a0", "a1", "asym_index",
                                  "phi_rad", "phi_deg", "cv", "lobe_ratio",
                                  "n_rays", "true_time_min"])
    if radial_df is None or radial_df.empty:
        return empty

    df = radial_df
    if "rho_wall" not in df.columns:
        df = add_wall_normalised_radius(df, config)

    lo, hi = float(band[0]), float(band[1])
    col = "r_from_edge_um" if band_units == "um" else "rho_wall"
    shell = df[(df[col] >= lo) & (df[col] <= hi)]
    if shell.empty:
        print(f"No samples in band {band} ({band_units}).")
        return empty

    per_ray = (shell.groupby(["t", "track_id", "theta_index"])
               .agg(theta_rad=("theta_rad", "first"), I=("intensity", "mean"))
               .reset_index())

    rows = []
    for (t, tid), d in per_ray.groupby(["t", "track_id"]):
        if len(d) < min_rays:
            continue
        d = d.sort_values("theta_index")
        th, I = d.theta_rad.to_numpy(), d.I.to_numpy()
        a0, a1, phi = _first_harmonic(th, I)
        k = max(1, len(I) // 10)
        srt = np.sort(I)
        dim = float(srt[:k].mean())
        rows.append({
            "t": int(t), "track_id": int(tid),
            "a0": a0, "a1": a1,
            "asym_index": (a1 / a0) if (np.isfinite(a0) and a0 > 0) else np.nan,
            "phi_rad": phi, "phi_deg": float(np.degrees(phi) % 360.0),
            "cv": float(np.std(I) / a0) if a0 else np.nan,
            "lobe_ratio": float(srt[-k:].mean() / dim) if dim > 0 else np.nan,
            "n_rays": int(len(I)),
        })

    asym_df = pd.DataFrame(rows)
    if asym_df.empty:
        return empty
    return attach_true_time(asym_df, tracked_df).sort_values(["t", "track_id"])


def _circular_stats(phi_rad: np.ndarray) -> Tuple[float, float, float]:
    """Mean direction, resultant length R, and Rayleigh p for a set of angles."""
    phi = np.asarray(phi_rad, dtype=float)
    phi = phi[np.isfinite(phi)]
    n = phi.size
    if n == 0:
        return np.nan, np.nan, np.nan
    C = np.mean(np.exp(1j * phi))
    R = float(np.abs(C))
    p = float(np.exp(-n * R ** 2))          # Rayleigh test, large-n approximation
    return float(np.angle(C)), R, p


def plot_membrane_asymmetry_vs_time(asym_df: pd.DataFrame, config,
                                    metric: str = "asym_index",
                                    bin_min: float = 6.0,
                                    show_tracks: bool = True,
                                    save_as: Optional[str] = "membrane_asymmetry_vs_time"):
    """Three panels: asymmetry vs time, its distribution per time bin, and whether
    the bright side points the same way across the population.

    Panel (c) is the control that matters. If the population resultant R sits
    above the Rayleigh line, every nucleus is bright on the same side of the
    image -- which is what an illumination gradient or a coverslip tilt looks
    like, not what stochastic dynein-driven delivery looks like. Biological
    asymmetry should be strong per nucleus and randomly oriented across them.
    """
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return

    d = asym_df.dropna(subset=[metric, "true_time_min"]).copy()
    if d.empty:
        print("No finite values to plot."); return
    d["bin_lo"], edges = _time_bins(d["true_time_min"], bin_min)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    # (a) per-track trajectories + population median/IQR
    ax = axes[0]
    if show_tracks:
        for tid, dt in d.groupby("track_id"):
            if len(dt) < 2:
                continue
            dt = dt.sort_values("true_time_min")
            ax.plot(dt.true_time_min, dt[metric], color="0.7", lw=0.6, alpha=0.6, zorder=1)
    g = d.groupby("true_time_min")[metric]
    med, q1, q3 = g.median(), g.quantile(0.25), g.quantile(0.75)
    ax.fill_between(med.index, q1, q3, alpha=0.25, color="#4C72B0", lw=0, zorder=2)
    ax.plot(med.index, med, lw=2.2, color="#4C72B0", zorder=3, label="median \u00b1 IQR")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel({"asym_index": "asymmetry index  $A_1/A_0$",
                   "cv": "CV of perinuclear intensity",
                   "lobe_ratio": "bright decile / dim decile"}.get(metric, metric))
    ax.set_title("(a) Membrane asymmetry vs time")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) distribution per time bin
    ax = axes[1]
    groups = [d.loc[d.bin_lo == lo, metric].to_numpy() for lo in edges]
    centers = edges + bin_min / 2
    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True,
               boxprops=dict(facecolor="#DD8452", alpha=0.55, lw=0.8),
               medianprops=dict(color="k", lw=1.4),
               whiskerprops=dict(lw=0.8), capprops=dict(lw=0.8))
    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right", fontsize=8)
    ax.set_xlabel(f"Time bin (min, width {bin_min:g})")
    ax.set_ylabel("asymmetry index")
    ax.set_title(f"(b) Distribution per {bin_min:g}-min bin")
    ax.spines[["top", "right"]].set_visible(False)

    # (c) directional consistency across the population
    ax = axes[2]
    stats = []
    for lo in edges:
        sub = d[d.bin_lo == lo]
        mu, R, p = _circular_stats(sub["phi_rad"].to_numpy())
        stats.append({"bin_lo": lo, "n": len(sub), "mean_dir_deg": np.degrees(mu) % 360,
                      "R": R, "rayleigh_p": p})
    stats = pd.DataFrame(stats)
    ax.plot(stats.bin_lo + bin_min / 2, stats.R, marker="o", lw=1.8, color="#55A868",
            label="population resultant $R$")
    n_ref = stats["n"].replace(0, np.nan)
    ax.plot(stats.bin_lo + bin_min / 2, np.sqrt(-np.log(0.05) / n_ref),
            ls="--", color="k", lw=1, label="Rayleigh p=0.05")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Time bin (min)")
    ax.set_ylabel("directional concentration $R$")
    ax.set_title("(c) Do all nuclei point the same way?")
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Perinuclear membrane asymmetry", fontsize=13)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()
    return stats


def plot_asymmetry_direction_rose(asym_df: pd.DataFrame, config, nbins: int = 24,
                                  save_as: Optional[str] = "asymmetry_direction_rose"):
    """Polar histogram of the bright-side direction, one panel per frame."""
    if asym_df is None or asym_df.empty:
        print("No data to plot."); return
    ts = sorted(asym_df["t"].unique())
    ncol = min(5, len(ts))
    nrow = math.ceil(len(ts) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.2 * nrow),
                             subplot_kw={"projection": "polar"}, squeeze=False)
    edges = np.linspace(0, 2 * np.pi, nbins + 1)
    for ax, t in zip(axes.ravel(), ts):
        phi = asym_df.loc[asym_df.t == t, "phi_rad"].dropna().to_numpy() % (2 * np.pi)
        counts, _ = np.histogram(phi, bins=edges)
        ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge",
               color="#8172B3", alpha=0.8, edgecolor="w", lw=0.4)
        mu, R, p = _circular_stats(asym_df.loc[asym_df.t == t, "phi_rad"].to_numpy())
        if np.isfinite(R) and counts.max() > 0:
            ax.plot([mu, mu], [0, counts.max()], color="k", lw=1.6)
        ax.set_title(f"t={t}  n={len(phi)}\nR={R:.2f}, p={p:.3g}", fontsize=8)
        _orient_polar(ax)
        ax.tick_params(labelsize=6)
        ax.set_yticklabels([])
    for ax in axes.ravel()[len(ts):]:
        ax.axis("off")
    fig.suptitle("Direction of the membrane-bright side (image frame)", fontsize=12)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()

### 36e. Cross-sectional area by time interval

Box plot of nuclear cross-sectional area in fixed acquisition-time bins.

On bin width: `true_time_min = t · (tile_rows · tile_cols · minutes_per_tile) +
tile_offset_min`. With the current 2 × 3 tiling at 1 min/tile that is **6 minutes
per frame**, so a 6-minute bin is exactly one frame with its six tile offsets
folded in. Binning narrower than 6 min splits a frame by tile — that is a
stage-position axis, not a time axis, and any structure it shows is tile order.
Wider bins (12, 18) pool frames and are legitimate.

Pass `halo_df` for repaired, halo-measured areas, or `timed_df` for every
detection before tracking. Area is always recomputed from the pixel count with
the live `cfg.pixel_size_um`, so a stale pickle from a different calibration
can't leak into the figure.

In [ ]:
def plot_area_boxplot_binned(df: pd.DataFrame, config, bin_min: float = 6.0,
                             time_col: str = "true_time_min",
                             area_col: Optional[str] = None,
                             show_points: bool = True,
                             show_median_trend: bool = True,
                             show_n: bool = True,
                             ylim: Optional[Tuple[float, float]] = None,
                             title: Optional[str] = None,
                             save_as: Optional[str] = "area_boxplot_6min"):
    """Nuclear cross-sectional area, boxed into fixed-width acquisition-time bins.

    Note on bin width: true_time_min = t * (tile_rows * tile_cols *
    minutes_per_tile) + tile_offset_min. With the current 2x3 tiling at 1 min
    per tile that is 6 minutes per frame, so a 6-minute bin is exactly one
    frame and the tile offsets fall inside it. Narrower bins split a frame by
    tile, which is a stage-position axis, not a biology axis.
    """
    if df is None or df.empty:
        print("No data to plot."); return

    d = df.copy()
    if area_col is None:
        d = resolve_area_um2(d, config)
        area_col = "nucleus_area_um2"
    if time_col not in d.columns:
        raise ValueError(f"DataFrame must contain '{time_col}'.")
    d = d.dropna(subset=[area_col, time_col])
    if d.empty:
        print("No finite values to plot."); return

    d["bin_lo"], edges = _time_bins(d[time_col], bin_min)
    groups = [d.loc[d.bin_lo == lo, area_col].to_numpy() for lo in edges]
    centers = edges + bin_min / 2

    fig, ax = plt.subplots(figsize=(max(7.5, 0.85 * len(edges) + 3), 5))

    if show_points:
        rng = np.random.default_rng(0)
        for lo, vals in zip(edges, groups):
            if vals.size == 0:
                continue
            x = lo + bin_min / 2 + rng.uniform(-bin_min * 0.22, bin_min * 0.22, vals.size)
            ax.scatter(x, vals, s=7, alpha=0.25, color="0.35", lw=0, zorder=1)

    ax.boxplot(groups, positions=centers, widths=bin_min * 0.7,
               showfliers=False, patch_artist=True, zorder=2,
               boxprops=dict(facecolor="#4C72B0", alpha=0.45, lw=0.9),
               medianprops=dict(color="k", lw=1.6),
               whiskerprops=dict(lw=0.9), capprops=dict(lw=0.9))

    if show_median_trend:
        meds = [np.median(v) if v.size else np.nan for v in groups]
        ax.plot(centers, meds, color="#C44E52", lw=1.8, marker="o", ms=4,
                zorder=3, label="median")
        ax.legend(fontsize=9, frameon=False, loc="upper left", bbox_to_anchor=(0.0, 0.95))

    if show_n:
        # blended transform: x in data units, y in axes fraction, so the labels
        # sit inside the axes and cannot collide with the title
        trans = ax.get_xaxis_transform()
        for c, v in zip(centers, groups):
            ax.text(c, 0.99, f"n={v.size}", transform=trans, ha="center", va="top",
                    fontsize=7, color="0.35")

    ax.set_xlim(edges[0] - bin_min * 0.7, edges[-1] + bin_min * 1.7)
    ax.set_xticks(centers)
    ax.set_xticklabels([f"{lo:g}\u2013{lo + bin_min:g}" for lo in edges],
                       rotation=45, ha="right")
    ax.set_xlabel(f"Acquisition time bin (min, width {bin_min:g})")
    ax.set_ylabel("Nuclear cross-sectional area (\u00b5m\u00b2)")
    ax.set_title(title or f"Nuclear cross-sectional area per {bin_min:g}-min interval")
    if ylim:
        ax.set_ylim(*ylim)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    if save_as:
        save_figure(fig, save_as, config)
    plt.show()

    summary = (d.groupby("bin_lo")[area_col]
               .agg(n="size", median="median",
                    q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75),
                    mean="mean", std="std")
               .reset_index().rename(columns={"bin_lo": "bin_start_min"}))
    summary["bin_end_min"] = summary["bin_start_min"] + bin_min
    return summary.round(2)

### 36f. Run the full figure set

In [ ]:
# ── Full figure set ───────────────────────────────────────────────────────
# Every figure is written to <run>/exports/figures/ as PNG + PDF.
SAVE_FIGURES = True

# 0. carried over from v17 — N/C ratio and area vs time
plot_nc_vs_time(halo_df)
plot_area_vs_time(halo_df)
plot_largest_cross_sectional_area_vs_time(timed_df, cfg)

# 1. one-time prep: proper nucleus-surface -> wall radius on the sweep
radial_n_df = add_wall_normalised_radius(radial_df, cfg)

# 2. rose plots — envelope shape by angle
envelope_peak_df = compute_envelope_radius_by_angle(
    radial_n_df, cfg, search_rho_max=ENVELOPE_SEARCH_RHO_MAX)
_ = plot_rose_individual(envelope_peak_df, tracked_df, cfg, n=5, selection="longest")
pooled_rose_df = plot_rose_pooled(envelope_peak_df, cfg)

# 3. membrane intensity by angle and distance
_longest = tracked_df.groupby("track_id")["t"].nunique().idxmax()
plot_membrane_angle_distance(radial_n_df, cfg, track_id=_longest,
                             save_as=f"angle_distance_track{_longest}")
# rbins kept coarse here on purpose: rays are sampled every
# radial_step_size_px (1 px = 0.1625 um), so bins finer than ~1.5 px come back
# part-empty. 8 um / 32 bins = 0.25 um = 1.5 px.
plot_membrane_angle_distance(radial_n_df, cfg, track_id=None, rbins=32,
                             radius_col="r_from_edge_um", r_max_um=8.0,
                             save_as="angle_distance_pooled")

# 4. membrane asymmetry vs time
asym_df = compute_membrane_asymmetry(radial_n_df, cfg, tracked_df=tracked_df,
                                     band=ASYMMETRY_BAND_UM, band_units="um")
asym_stats = plot_membrane_asymmetry_vs_time(asym_df, cfg, bin_min=AREA_BIN_MIN)
plot_asymmetry_direction_rose(asym_df, cfg)
asym_df.to_csv(cfg.exports_dir / "membrane_asymmetry.csv", index=False)

# 5. cross-sectional area boxed into fixed time intervals
area_summary = plot_area_boxplot_binned(halo_df, cfg, bin_min=AREA_BIN_MIN)
display(area_summary)
area_summary.to_csv(cfg.exports_dir / "area_by_time_bin.csv", index=False)

print("figures ->", figures_dir(cfg))

---
## 37. Shell-resolved membrane asymmetry

The faceted rose — radial shells down the rows, timepoints across the columns —
plus the normalised-score supplement and the direction-agnostic aggregate.

Four things about the original version change what it says, so each is
implemented as an explicit, switchable choice rather than a silent default.

**1. Summed intensity is biased by the droplet wall.** Rays stop at the wall,
so a shell defined in absolute distance from the nuclear surface is clipped by
different amounts at different θ as soon as the nucleus sits off-centre in its
droplet. A sum then counts more pixels on the far side and reports asymmetry
pointing away from the near wall — *for a perfectly uniform field*. With a
uniform intensity field and a nucleus 6 µm off-centre, the outermost shell
returns R = 0.42; at 9 µm it returns 0.60. §37a bins on **mean per sample**,
which is invariant to how many pixels a cell happens to contain, and §37b adds
`R_geom` — the same resultant computed on the sample counts with intensity
discarded — as a per-facet floor.

**2. Radius encoding exaggerates magnitude.** On a polar bar the eye reads
area, and area goes as r². A linear radius shows a 2× difference as a 4× wedge.
§37c defaults to `radius_scale="sqrt"`, so wedge **area** is proportional to the
value. `"linear"` restores the literal encoding.

**3. The resultant needs a baseline that is not its own minimum.** With a
background pedestal *b* and signal *s*, R = |Σs·e^{iθ}| / (N·b + Σs), so a
bright cytoplasm drives R toward zero however one-sided the membrane is.
Subtracting each profile's own minimum overcorrects in a way that is worse:
R then becomes a pure *shape* statistic, returning **0.525 for a cosine of any
amplitude**, so a nucleus twice as one-sided scores identically. §37b subtracts
a single dataset-wide floor, giving R = (A₁/A₀)/2 — bounded on [0, 1], zero for
an even shell, linear in how one-sided it is.

**4. The resultant needs a null.** Even a uniform shell returns R ≈ 1/√n_eff,
about 0.17 for 36 bins, so every facet looks mildly asymmetric. §37b reports
`R_corr` (debiased), a Rayleigh p on the weighted resultant using Kish's
effective sample size, and `R_crit`, drawn on each facet as a dashed ring: an
arrow inside that ring is noise.

In [ ]:
from typing import Sequence                      # not in the Section 1 import block
from matplotlib.colors import LinearSegmentedColormap

# ── Section 37 parameters ─────────────────────────────────────────────────
# Shell edges, microns outward from the nuclear mask surface. These are the
# edges from the original figure; change them freely, but see 37b on why the
# outermost shell needs the geometry control.
DEFAULT_SHELL_EDGES_UM = (0.0, 2.34, 5.36, 9.55, 23.5)

# Angular bins. 36 bins = 10 degrees. Fewer bins raises the per-nucleus
# detection floor (R_crit goes as 1/sqrt(n_eff)); more bins makes each bin
# noisier without adding information once bins are finer than the ray spacing.
N_THETA_BINS = 36

# Fraction of theta below which a shell is flagged as wall-clipped.
MIN_SHELL_COVERAGE = 0.80

### 37a. Binning into (shell, θ) cells

Returns a **complete** θ grid per unit — cells with no sample are kept with
`n_samples = 0` rather than dropped, because their absence is the geometry
signal that §37b tests for.

The per-cell statistic is `I_mean`, not `I_sum`. `I_sum` is still carried so
the original encoding can be reproduced for comparison.

In [ ]:
def bin_shell_angle(radial_df: pd.DataFrame, config,
                    shell_edges_um: Sequence[float] = DEFAULT_SHELL_EDGES_UM,
                    n_theta_bins: int = N_THETA_BINS,
                    radius_col: str = "r_from_edge_um") -> pd.DataFrame:
    """Bin the sweep into (shell, theta) cells per nucleus per timepoint.

    Returns one row per (t, track_id, shell_idx, theta_bin) on a COMPLETE theta
    grid -- cells with no sample are kept with n_samples = 0 and I_mean = NaN,
    because their absence is itself the geometry signal that
    compute_shell_resultants tests for.

    I_mean, not I_sum, is the per-cell statistic. Rays terminate at the droplet
    wall, so a shell defined in absolute distance is clipped by different
    amounts at different theta whenever the nucleus is off-centre. A sum then
    counts more pixels on the far side and reports asymmetry pointing away from
    the near wall, for a perfectly uniform field. A mean is invariant to how
    many pixels a cell happens to contain.
    """
    if radial_df is None or radial_df.empty:
        return pd.DataFrame()

    df = radial_df
    if radius_col not in df.columns:
        raise ValueError(f"{radius_col} missing -- run add_wall_normalised_radius first.")

    edges = np.asarray(shell_edges_um, dtype=float)
    d = df[["t", "track_id", "theta_rad", radius_col, "intensity"]].copy()
    d["shell_idx"] = pd.cut(d[radius_col], edges, labels=False, include_lowest=True)
    d = d[d["shell_idx"].notna()]
    if d.empty:
        return pd.DataFrame()
    d["shell_idx"] = d["shell_idx"].astype(int)

    dtheta = 2 * np.pi / n_theta_bins
    d["theta_bin"] = ((d["theta_rad"] % (2 * np.pi)) // dtheta).astype(int)
    d["theta_bin"] = d["theta_bin"].clip(0, n_theta_bins - 1)

    g = (d.groupby(["t", "track_id", "shell_idx", "theta_bin"])["intensity"]
         .agg(n_samples="size", I_sum="sum", I_mean="mean")
         .reset_index())

    # complete the theta grid so empty cells are explicit
    units = g[["t", "track_id", "shell_idx"]].drop_duplicates()
    grid = units.merge(pd.DataFrame({"theta_bin": np.arange(n_theta_bins)}), how="cross")
    out = grid.merge(g, on=["t", "track_id", "shell_idx", "theta_bin"], how="left")
    out["n_samples"] = out["n_samples"].fillna(0).astype(int)
    out["I_sum"] = out["I_sum"].fillna(0.0)

    out["theta_rad"] = (out["theta_bin"] + 0.5) * dtheta
    out["shell_lo_um"] = edges[out["shell_idx"].to_numpy()]
    out["shell_hi_um"] = edges[out["shell_idx"].to_numpy() + 1]
    out["shell_label"] = [f"({lo:g}, {hi:g}]" for lo, hi
                          in zip(out.shell_lo_um, out.shell_hi_um)]
    out.attrs["n_theta_bins"] = n_theta_bins
    out.attrs["shell_edges_um"] = tuple(edges)
    return out.sort_values(["t", "track_id", "shell_idx", "theta_bin"]).reset_index(drop=True)

### 37b. Resultant vectors, baselines, and the two nulls

Four resultants per unit, each answering a different question:

| column | weights | what it measures |
|---|---|---|
| `R` | `I_mean − background` | **the score.** (A₁/A₀)/2 for a cosine; 0 = even, 1 = all one side |
| `R_raw` | `I_mean` | what an uncorrected rose shows; deflated by the pedestal |
| `R_shape` | `I_mean −` own 10th pct | concentration only — saturates at 0.525, never quote as magnitude |
| `R_geom` | `n_samples` | asymmetry from droplet-wall clipping alone |

`R` must clear both `R_crit` (the noise floor) and `R_geom` (the geometry
floor) to mean anything.

On `background`: it has to be estimated from outside the profile being scored —
`estimate_background` uses a low quantile of every sweep sample in the dataset,
which is the camera offset plus the isotropic cytoplasmic floor. Pass your own
value if you have a dark-frame measurement.

`n_eff` is Kish's effective sample size, `(Σw)²/Σw²`: the number of
equally-weighted bins carrying the same information. It is what the Rayleigh
statistic needs when observations are intensity weights rather than counts.

In [ ]:
def _resultant(theta: np.ndarray, w: np.ndarray) -> Tuple[float, float, float]:
    """Weighted circular resultant. Returns (R, phi, n_eff).

    n_eff is Kish's effective sample size (sum w)^2 / sum(w^2): the number of
    equally-weighted bins that would carry the same information. It is what the
    Rayleigh statistic needs when the observations are intensity weights rather
    than counts.
    """
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(w) & np.isfinite(theta) & (w > 0)
    if ok.sum() < 3:
        return np.nan, np.nan, np.nan
    th, w = theta[ok], w[ok]
    sw = w.sum()
    if sw <= 0:
        return np.nan, np.nan, np.nan
    C = (w * np.exp(1j * th)).sum() / sw
    n_eff = float(sw ** 2 / (w ** 2).sum())
    return float(np.abs(C)), float(np.angle(C)), n_eff


def _debias_R(R: float, n_eff: float) -> float:
    """Bias-corrected resultant length.

    E[R^2] ~= rho^2 + (1 - rho^2)/n_eff, so even a perfectly uniform shell
    returns R ~ 1/sqrt(n_eff) -- about 0.17 for 36 bins. Reporting raw R makes
    every facet look mildly asymmetric. This inverts the relation and clips at
    zero.
    """
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    rho2 = (n_eff * R ** 2 - 1.0) / (n_eff - 1.0)
    return float(np.sqrt(max(rho2, 0.0)))


def _rayleigh_p(R: float, n_eff: float) -> float:
    """Rayleigh test on the weighted resultant (Zar's two-term approximation)."""
    if not (np.isfinite(R) and np.isfinite(n_eff)) or n_eff <= 1:
        return np.nan
    z = n_eff * R ** 2
    p = np.exp(-z) * (1 + (2 * z - z ** 2) / (4 * n_eff)
                      - (24 * z - 132 * z ** 2 + 76 * z ** 3 - 9 * z ** 4)
                      / (288 * n_eff ** 2))
    return float(np.clip(p, 0.0, 1.0))


def R_critical(n_eff: float, alpha: float = 0.05) -> float:
    """Resultant length a uniform shell would exceed with probability alpha."""
    if not np.isfinite(n_eff) or n_eff <= 1:
        return np.nan
    return float(np.sqrt(-np.log(alpha) / n_eff))


def estimate_background(radial_df: pd.DataFrame, q: float = 0.05) -> float:
    """Isotropic intensity floor for the whole dataset: the q-quantile of every
    sweep sample.

    This is the zero of the asymmetry scale, and it has to come from OUTSIDE the
    profile being scored. Subtracting a profile's own minimum instead makes R a
    shape statistic that saturates -- a cosine of ANY amplitude returns
    R = 0.525, so a nucleus twice as one-sided scores the same. Subtracting a
    common floor gives R = (A1/A0)/2, which is what a normalised asymmetry
    score should do.
    """
    if radial_df is None or radial_df.empty:
        return 0.0
    return float(np.quantile(radial_df["intensity"].to_numpy(), q))


def compute_shell_resultants(binned: pd.DataFrame,
                             background: Optional[float] = None,
                             background_q: float = 0.05,
                             min_coverage: float = 0.80,
                             alpha: float = 0.05) -> pd.DataFrame:
    """Per (t, track_id, shell): resultant vector, its null, and a geometry control.

    Four numbers per unit, each answering a different question:

    R        THE SCORE. Resultant of (I_mean - background), background being one
             dataset-wide floor (see estimate_background). For a cosine profile
             this is exactly (A1/A0)/2, so it is bounded on [0, 1], zero for an
             even shell, and linear in how one-sided the shell is.
    R_raw    the same without subtracting anything. Reported because it is what
             an uncorrected rose shows, and it is deflated by the pedestal: a
             bright cytoplasm pushes it toward zero however one-sided the
             membrane is. Compare the two to see how much background was
             masking.
    R_shape  resultant after removing each shell's OWN 10th percentile. This
             discards amplitude entirely and measures only how concentrated the
             bright side is -- a narrow lobe scores above a broad one. Useful,
             but never quote it as an asymmetry magnitude.
    R_geom   resultant of n_samples, intensity discarded. The asymmetry a
             perfectly uniform field would show given only how the droplet wall
             clipped the rays. R has to clear R_geom to mean anything.

    Also: R_corr (R debiased for finite bin count), rayleigh_p, R_crit,
    coverage, and A1_over_A0 = 2R for a cosine, quoted directly for continuity
    with Section 36d.
    """
    cols = ["t", "track_id", "shell_idx", "shell_label", "shell_lo_um", "shell_hi_um",
            "I_bar", "I_bg", "R", "R_corr", "R_raw", "R_shape",
            "phi_rad", "phi_deg", "n_eff", "rayleigh_p", "R_crit", "significant",
            "R_geom", "phi_geom_deg", "R_excess", "coverage", "n_occupied",
            "A1_over_A0", "n_samples_total"]
    if binned is None or binned.empty:
        return pd.DataFrame(columns=cols)

    if background is None:
        occ_vals = binned.loc[binned.n_samples > 0, "I_mean"].to_numpy()
        background = float(np.quantile(occ_vals, background_q)) if occ_vals.size else 0.0
        print(f"background floor estimated from binned data: {background:.1f} "
              f"(q={background_q}). Pass background=... to override.")

    rows = []
    for (t, tid, sh), d in binned.groupby(["t", "track_id", "shell_idx"], sort=True):
        d = d.sort_values("theta_bin")
        th = d["theta_rad"].to_numpy()
        I = d["I_mean"].to_numpy(dtype=float)
        n = d["n_samples"].to_numpy(dtype=float)

        occ = n > 0
        coverage = float(occ.mean())
        if occ.sum() < 3:
            continue

        Ifin = np.where(occ, I, np.nan)
        I_bar = float(np.nanmean(Ifin))

        w_raw = np.where(occ, Ifin, 0.0)
        w_sig = np.clip(w_raw - background, 0.0, None) * occ
        w_shape = np.clip(w_raw - np.nanquantile(Ifin, 0.10), 0.0, None) * occ

        R_raw, _, _ = _resultant(th, w_raw)
        R, phi, n_eff = _resultant(th, w_sig)
        R_shape, _, _ = _resultant(th, w_shape)
        R_geom, phi_g, _ = _resultant(th, n)

        # first harmonic of the background-subtracted profile
        sig = np.where(occ, Ifin - background, np.nan)
        base = np.nanmean(sig)
        if occ.sum() >= 8 and np.isfinite(base) and base > 0:
            c1 = np.nanmean(sig * np.exp(-1j * th))
            A1_A0 = float(2 * np.abs(c1) / base)
        else:
            A1_A0 = np.nan

        R_corr = _debias_R(R, n_eff)
        p = _rayleigh_p(R, n_eff)
        Rc = R_critical(n_eff, alpha)
        rows.append({
            "t": int(t), "track_id": int(tid), "shell_idx": int(sh),
            "shell_label": d["shell_label"].iloc[0],
            "shell_lo_um": float(d["shell_lo_um"].iloc[0]),
            "shell_hi_um": float(d["shell_hi_um"].iloc[0]),
            "I_bar": I_bar, "I_bg": float(background),
            "R": R, "R_corr": R_corr, "R_raw": R_raw, "R_shape": R_shape,
            "phi_rad": phi,
            "phi_deg": float(np.degrees(phi) % 360.0) if np.isfinite(phi) else np.nan,
            "n_eff": n_eff, "rayleigh_p": p, "R_crit": Rc,
            "significant": bool(np.isfinite(p) and p < alpha),
            "R_geom": R_geom,
            "phi_geom_deg": float(np.degrees(phi_g) % 360.0) if np.isfinite(phi_g) else np.nan,
            "R_excess": (R_corr - R_geom) if np.isfinite(R_corr) and np.isfinite(R_geom) else np.nan,
            "coverage": coverage, "n_occupied": int(occ.sum()),
            "A1_over_A0": A1_A0,
            "n_samples_total": float(n.sum()),
        })

    out = pd.DataFrame(rows, columns=cols)
    if not out.empty:
        out["low_coverage"] = out["coverage"] < min_coverage
        n_low = int(out["low_coverage"].sum())
        if n_low:
            print(f"NOTE: {n_low}/{len(out)} shell-units below {min_coverage:.0%} theta "
                  f"coverage -- the droplet wall cuts these shells off. They are kept "
                  f"but flagged; read R_geom for them.")
    return out

### 37c. The faceted rose

Rows = shells, columns = timepoints. Reading the facet:

- **coloured wedges** — mean intensity above the background floor, area-scaled
- **grey wedges** — angles with no sample at all (wall clipping), marked rather
  than left blank
- **red arrow** — the resultant, length = R on the 0–1 radius; solid red when
  p < 0.05, grey otherwise
- **dashed ring** — `R_crit`; an arrow inside it is noise
- **dotted grey arrow** — `R_geom`. Where it rivals the red arrow, the facet is
  showing droplet placement
- **facet label** — `R = score`, `g = R_geom`, `*` = significant

`share_r="row"` puts a shell's timepoints on one radial scale so the columns are
comparable; `"all"` compares shells too; `"none"` autoscales each facet and
should only be used to read shape.

In [ ]:
def _rose_axis_values(d: pd.DataFrame, res_row, value: str) -> np.ndarray:
    occ = d["n_samples"].to_numpy() > 0
    I = d["I_mean"].to_numpy(dtype=float)
    if value == "mean":
        v = np.where(occ, I, 0.0)
    elif value == "mean_bgsub":
        v = np.where(occ, I - float(res_row["I_bg"]), 0.0)
    elif value == "sum":
        v = d["I_sum"].to_numpy(dtype=float)
    else:
        raise ValueError(f"unknown value {value!r}")
    return np.clip(v, 0.0, None)


def plot_shell_rose_facets(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                           track_id: Optional[int] = None,
                           timepoints: Optional[Sequence[int]] = None,
                           value: str = "mean_bgsub",
                           radius_scale: str = "sqrt",
                           share_r: str = "row",
                           show_geom_arrow: bool = True,
                           alpha: float = 0.05,
                           save_as: str = "shell_rose_facets"):
    """Rose facets: rows = radial shells, columns = timepoints.

    Encoding choices, all of which change what the reader concludes:

    radius_scale="sqrt" (default) makes wedge AREA proportional to the value.
        On a polar bar the eye reads area, and area goes as r^2, so a linear
        radius shows a 2x difference as a 4x wedge. Every Nightingale rose has
        this problem; sqrt is the standard correction. Pass "linear" to get the
        literal radius encoding back.

    value="mean_bgsub" (default) plots mean intensity per sample with the
        per-shell baseline removed, so the origin is background rather than
        zero counts and the bars show the part of the signal that can be
        asymmetric. "mean" keeps the pedestal and draws it as a grey ring so
        you can see how much of each bar is background. "sum" reproduces the
        original encoding and is biased wherever coverage < 1.

    share_r="row" puts every timepoint in a shell on one radial scale, so the
        columns are comparable; "all" compares shells too; "none" autoscales
        each facet and should only be used for shape.

    The red arrow is the background-subtracted resultant, length R x r_limit.
    The dashed ring is R_crit, the length a uniform shell clears 5 % of the
    time -- an arrow inside that ring is noise. The open grey arrow is R_geom,
    the resultant of the sample counts alone: the asymmetry produced by the
    droplet wall clipping the rays. Where the grey arrow rivals the red one,
    the facet is showing geometry.
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        print("No data to plot."); return

    b, r = binned, resultants
    if track_id is not None:
        b = b[b.track_id == int(track_id)]
        r = r[r.track_id == int(track_id)]
        subtitle = f"track {int(track_id)}"
        if b.empty:
            print(f"track {track_id} not present."); return
    else:
        subtitle = "per-nucleus facets require track_id; pooling shown"

    ts = sorted(b["t"].unique()) if timepoints is None else list(timepoints)
    shells = sorted(b["shell_idx"].unique())
    if not ts or not shells:
        print("No data to plot."); return

    key = ["t", "track_id", "shell_idx"]
    res_lut = r.set_index(key)

    # precompute values so scales can be shared
    vals: Dict[Tuple[int, int], np.ndarray] = {}
    for sh in shells:
        for t in ts:
            try:
                row = res_lut.loc[(t, int(track_id), sh)]
            except KeyError:
                continue
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            if d.empty:
                continue
            vals[(sh, t)] = _rose_axis_values(d, row, value)

    if not vals:
        print("No data to plot."); return

    def _lim(keys):
        pool = np.concatenate([vals[k] for k in keys if k in vals])
        pool = pool[np.isfinite(pool)]
        return float(np.percentile(pool, 99.5)) if pool.size else 1.0

    if share_r == "all":
        lim_all = _lim(list(vals))
        limits = {sh: lim_all for sh in shells}
    elif share_r == "row":
        limits = {sh: _lim([(sh, t) for t in ts]) for sh in shells}
    else:
        limits = None

    xf = (lambda v, L: np.sqrt(np.clip(v, 0, None) / L) if radius_scale == "sqrt"
          else np.clip(v, 0, None) / L)

    dtheta = 2 * np.pi / b.attrs.get("n_theta_bins", N_THETA_BINS)
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    shell_colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(len(shells), len(ts),
                             figsize=(2.05 * len(ts), 2.25 * len(shells)),
                             subplot_kw={"projection": "polar"}, squeeze=False)

    for i, sh in enumerate(shells):
        for j, t in enumerate(ts):
            ax = axes[i][j]
            ax.set_theta_zero_location("E")
            ax.set_theta_direction(-1)          # image frame: y grows downward
            ax.set_yticklabels([])
            ax.set_xticks(np.radians([0, 90, 180, 270]))
            ax.set_xticklabels([])   # orientation stated once in the subtitle
            ax.tick_params(labelsize=5.5, pad=-2)
            ax.grid(alpha=0.25, lw=0.4)
            ax.set_ylim(0, 1.0)

            if (sh, t) not in vals:
                ax.set_axis_off(); continue
            row = res_lut.loc[(t, int(track_id), sh)]
            d = b[(b.t == t) & (b.shell_idx == sh)].sort_values("theta_bin")
            v = vals[(sh, t)]
            L = limits[sh] if limits else max(float(np.max(v)), 1e-12)

            occ = d["n_samples"].to_numpy() > 0
            rr = xf(v, L)
            colors = [shell_colors[sh] if o else "none" for o in occ]
            ax.bar(d["theta_rad"].to_numpy(), rr, width=dtheta * 0.92,
                   bottom=0.0, color=colors, edgecolor="none", align="center")
            # unsampled angles marked, not silently blank
            if (~occ).any():
                ax.bar(d["theta_rad"].to_numpy()[~occ], np.ones((~occ).sum()),
                       width=dtheta * 0.92, bottom=0.0, color="0.90", zorder=0)

            if value == "mean" and np.isfinite(row["I_bg"]):
                ax.plot(np.linspace(0, 2 * np.pi, 200),
                        np.full(200, xf(row["I_bg"], L)),
                        color="0.45", lw=0.7, ls=":", zorder=4)

            Rv, phi = float(row["R"]), float(row["phi_rad"])
            Rc = float(row["R_crit"])
            if np.isfinite(Rc):
                ax.plot(np.linspace(0, 2 * np.pi, 200), np.full(200, Rc),
                        color="k", lw=0.6, ls="--", alpha=0.5, zorder=4)
            if np.isfinite(Rv) and np.isfinite(phi):
                sig = bool(row["significant"])
                ax.annotate("", xy=(phi, Rv), xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.4,
                                            color="#D7263D" if sig else "0.55",
                                            alpha=1.0 if sig else 0.7), zorder=6)
            if show_geom_arrow and np.isfinite(row["R_geom"]) and row["R_geom"] > 0.02:
                ax.annotate("", xy=(np.radians(row["phi_geom_deg"]), row["R_geom"]),
                            xytext=(0, 0),
                            arrowprops=dict(arrowstyle="-|>", lw=1.1, color="0.35",
                                            ls=":", alpha=0.9), zorder=5)

            star = "*" if row["significant"] else ""
            ax.annotate(f"R={row['R']:.2f}{star}  g={row['R_geom']:.2f}",
                        xy=(0.5, -0.13), xycoords="axes fraction",
                        ha="center", fontsize=6.5,
                        color="0.15" if row["significant"] else "0.5")
            if i == 0:
                ax.annotate(f"t = {t}", xy=(0.5, 1.18), xycoords="axes fraction",
                            ha="center", fontsize=9)
            if j == 0:
                ax.annotate(f"{row['shell_label']} \u00b5m", xy=(-0.38, 0.5),
                            xycoords="axes fraction", rotation=90,
                            va="center", ha="center", fontsize=7.5)

    vlab = {"mean_bgsub": "mean intensity above shell baseline",
            "mean": "mean intensity per sample",
            "sum": "summed intensity (biased where coverage < 1)"}[value]
    rlab = ("radius \u221d \u221avalue, so wedge AREA \u221d value"
            if radius_scale == "sqrt" else "radius \u221d value (area \u221d value\u00b2)")
    fig.suptitle("Membrane intensity by angle and distance", fontsize=13, x=0.02, ha="left", y=0.995)
    fig.text(0.02, 0.963, f"10\u00b0 bins \u00b7 {vlab} \u00b7 {rlab} \u00b7 {subtitle}",
             fontsize=8, ha="left", color="0.25")
    fig.text(0.02, 0.941, "0\u00b0 = image +x (right); angle increases clockwise, "
             "matching the displayed image", fontsize=7.5, ha="left", color="0.45")
    fig.text(0.02, 0.012,
             "red arrow = background-subtracted resultant (length = R); dashed ring = R at p=0.05; "
             "dotted grey arrow = R_geom, the resultant of sample counts alone (droplet-wall clipping). "
             "Facet labels: R = asymmetry score, g = R_geom, * = p < %.2f. "
             "Grey wedges = angles with no sample." % alpha,
             fontsize=6.5, color="0.3")
    fig.tight_layout(rect=(0.02, 0.03, 1, 0.925))
    save_figure(fig, save_as, config)
    plt.show()
    return fig

### 37d. Normalised asymmetry score — table and graph

`build_asymmetry_score_table` collapses the per-nucleus vectors by (shell, t):
median score with IQR, the geometry floor, `R_excess` (the part geometry does
not explain), the fraction of nuclei individually clearing Rayleigh, and
`pop_R` — the coherence of φ **across** nuclei.

`pop_R` is the artefact check carried over from §36d. Stochastic delivery
should be strong per nucleus and randomly oriented across them. A `pop_R` above
its p = 0.05 line means the bright side has a preferred lab-frame direction,
which is what an illumination gradient or a tilted coverslip looks like.

In [ ]:
def build_asymmetry_score_table(resultants: pd.DataFrame,
                                by: Sequence[str] = ("shell_label", "t")) -> pd.DataFrame:
    """Aggregate the per-nucleus vectors into a reportable score table.

    Columns
    -------
    n                 shell-units contributing
    R_corr_median     normalised asymmetry score, 0 = even, 1 = all on one side
    R_corr_q1/q3      IQR across nuclei
    R_geom_median     the same statistic on sample counts alone -- the floor
                      set by droplet geometry
    R_excess_median   R_corr - R_geom, the part not explained by geometry
    frac_significant  fraction of nuclei clearing the per-nucleus Rayleigh test
    pop_R             coherence of phi ACROSS nuclei (lab-frame directionality)
    pop_p             Rayleigh p on pop_R
    coverage_min      worst theta coverage in the group
    """
    if resultants is None or resultants.empty:
        return pd.DataFrame()

    def agg(d):
        phi = d["phi_rad"].dropna().to_numpy()
        n = phi.size
        if n:
            C = np.mean(np.exp(1j * phi))
            pop_R = float(np.abs(C))
            pop_dir = float(np.degrees(np.angle(C)) % 360)
            pop_p = float(np.exp(-n * pop_R ** 2))
        else:
            pop_R = pop_dir = pop_p = np.nan
        return pd.Series({
            "n": len(d),
            "R_median": d["R"].median(),
            "R_q1": d["R"].quantile(0.25),
            "R_q3": d["R"].quantile(0.75),
            "R_corr_median": d["R_corr"].median(),
            "R_raw_median": d["R_raw"].median(),
            "R_geom_median": d["R_geom"].median(),
            "R_excess_median": d["R_excess"].median(),
            "R_shape_median": d["R_shape"].median(),
            "A1_over_A0_median": d["A1_over_A0"].median(),
            "frac_significant": d["significant"].mean(),
            "pop_R": pop_R, "pop_dir_deg": pop_dir, "pop_p": pop_p,
            "coverage_min": d["coverage"].min(),
        })

    out = (resultants.groupby(list(by), sort=True)
           .apply(agg, include_groups=False).reset_index())
    return out.round(4)


def plot_asymmetry_score_summary(resultants: pd.DataFrame, config,
                                 score: str = "R",
                                 save_as: str = "asymmetry_score_summary"):
    """Three panels: score vs time per shell, the geometry floor, and lab-frame
    coherence across nuclei."""
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    tab = build_asymmetry_score_table(resultants)
    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    ax = axes[0]
    for sh in shells:
        d = (resultants[resultants.shell_idx == sh]
             .groupby("t")[score].agg(["median", lambda s: s.quantile(.25),
                                       lambda s: s.quantile(.75)]))
        d.columns = ["med", "q1", "q3"]
        ax.fill_between(d.index, d.q1, d.q3, color=colors[sh], alpha=0.15, lw=0)
        ax.plot(d.index, d.med, color=colors[sh], lw=2, marker="o", ms=4,
                label=f"{labels[sh]} \u00b5m")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="single-nucleus noise floor (p=0.05)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("normalised asymmetry score $R$")
    ax.set_title("(a) Asymmetry score by shell")
    ax.legend(fontsize=7, title="shell", title_fontsize=7)
    ax.set_ylim(0, 1); ax.spines[["top", "right"]].set_visible(False)

    ax = axes[1]
    for sh in shells:
        d = resultants[resultants.shell_idx == sh].groupby("t")[["R", "R_geom"]].median()
        ax.plot(d.index, d.R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.index, d.R_geom, color=colors[sh], lw=1.2, ls=":", marker="^", ms=3)
    ax.plot([], [], color="0.3", lw=2, label="$R$ (signal)")
    ax.plot([], [], color="0.3", lw=1.2, ls=":", label="$R_{geom}$ (sampling)")
    ax.set_xlabel("timepoint"); ax.set_ylabel("$R$")
    ax.set_title("(b) Signal vs the geometry floor")
    ax.legend(fontsize=8); ax.set_ylim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)

    ax = axes[2]
    for sh in shells:
        d = tab[tab.shell_label == labels[sh]].sort_values("t")
        ax.plot(d.t, d.pop_R, color=colors[sh], lw=2, marker="o", ms=4)
        ax.plot(d.t, np.sqrt(-np.log(0.05) / d["n"].replace(0, np.nan)),
                color=colors[sh], lw=0.8, ls="--", alpha=0.6)
    ax.set_xlabel("timepoint"); ax.set_ylabel("population resultant of $\\phi$")
    ax.set_title("(c) Do nuclei agree on a lab direction?")
    ax.set_ylim(0, 1)
    ax.plot([], [], color="0.3", lw=0.8, ls="--", label="p = 0.05")
    ax.legend(fontsize=8); ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Normalised asymmetry score", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()
    return tab

### 37e. Aggregate across the dataset, direction discarded

Three views, none of which assume a common direction:

**(a) and (b)** pool the score over every nucleus × every frame per shell —
magnitude only, direction thrown away — against the geometry floor and the
uniform-shell noise line.

**(c) the alignment-averaged profile.** Each nucleus is rotated so its *own*
asymmetry axis sits at 0°, then all are averaged. A population whose bright
sides point in random directions still produces a lobe here, provided each
nucleus is individually one-sided — that is exactly the direction-agnostic
statement, and it also shows the *shape* of the asymmetry: a broad cosine means
gradual enrichment, a narrow spike means a delivery hotspot.

**The null is not optional here.** Choosing the rotation from the same data you
then average is circular: pure noise, aligned to its own noisiest angle,
produces a lobe. `aligned_mean_profile` repeats the entire procedure on
θ-permuted profiles — same values, spatial structure destroyed — `n_perm` times,
building a distribution of *mean* profiles, and returns its 95th percentile.
Comparing an averaged observation against unaveraged null draws understates the
null by √n_units. The test is read at lag 0 only, since both curves peak there
by construction.

In [ ]:
def aligned_mean_profile(binned: pd.DataFrame, resultants: pd.DataFrame,
                         shell_idx: int, n_perm: int = 200,
                         min_coverage: float = 1.0, t_min: Optional[int] = None,
                         seed: int = 0) -> Optional[Dict]:
    """Rotate every nucleus so its own asymmetry axis is at 0, then average.

    Direction-agnostic by construction: no lab frame is assumed, so a
    population whose bright sides point in random directions still produces a
    lobe here as long as each nucleus is individually one-sided.

    The catch, and why the null matters: the rotation is chosen from the same
    data that is then averaged. Pure noise, aligned to its own noisiest angle,
    produces a lobe. The null repeats the ENTIRE procedure -- permute each
    unit's theta bins (same values, spatial structure destroyed), realign each
    to its own permuted axis, average across units -- and does that n_perm
    times to build a distribution of MEAN profiles. The envelope returned is
    the 95th percentile across those replicate means, so it is on the same
    footing as the observed mean. Comparing an averaged observation against
    unaveraged null draws understates the null by a factor of sqrt(n_units).
    """
    if binned is None or binned.empty or resultants is None or resultants.empty:
        return None
    rng = np.random.default_rng(seed)
    n_bins = binned.attrs.get("n_theta_bins", N_THETA_BINS)
    dtheta = 2 * np.pi / n_bins
    th = (np.arange(n_bins) + 0.5) * dtheta

    res = resultants[(resultants.shell_idx == shell_idx)
                     & (resultants.coverage >= min_coverage)]
    if t_min is not None:
        res = res[res.t >= t_min]
    if res.empty:
        return None
    lut = res.set_index(["t", "track_id"])

    units, prof = [], []
    for (t, tid), d in binned[binned.shell_idx == shell_idx].groupby(["t", "track_id"]):
        if (t, tid) not in lut.index:
            continue
        row = lut.loc[(t, tid)]
        d = d.sort_values("theta_bin")
        if not (d["n_samples"].to_numpy() > 0).all():
            continue
        I = d["I_mean"].to_numpy(dtype=float)
        bg = float(row["I_bg"])
        s = I - bg
        base = s.mean()
        if not np.isfinite(base) or base <= 0 or not np.isfinite(row["phi_rad"]):
            continue
        p = s / base                                  # normalised, unitless
        units.append(p)
        prof.append(np.roll(p, -int(np.round(float(row["phi_rad"]) / dtheta))))

    if len(prof) < 3:
        return None
    prof = np.vstack(prof)

    # null: n_perm replicate MEAN profiles, each built the same way as the observation
    null_means = np.empty((n_perm, n_bins))
    for b in range(n_perm):
        acc = np.empty((len(units), n_bins))
        for i, p in enumerate(units):
            q = rng.permutation(p)
            _, phi_q, _ = _resultant(th, np.clip(q, 0, None))
            sh_q = 0 if not np.isfinite(phi_q) else int(np.round(phi_q / dtheta))
            acc[i] = np.roll(q, -sh_q)
        null_means[b] = acc.mean(axis=0)

    lag_deg = (np.degrees(th) + 180) % 360 - 180
    order = np.argsort(lag_deg)
    obs = prof.mean(axis=0)
    null_hi = np.percentile(null_means, 95, axis=0)

    # Score at lag 0 only. Both the observation and the null are aligned so
    # their maximum sits at lag 0 by construction, so that is the one bin the
    # test is about; taking a max over all 36 lags instead re-introduces a
    # multiple-comparisons problem and lets a uniform field clear its own null.
    lag0 = int(np.argmin(np.abs(lag_deg)))
    return {
        "lag_deg": lag_deg[order],
        "mean": obs[order],
        "sem": (prof.std(axis=0, ddof=1) / np.sqrt(len(prof)))[order],
        "null_hi": null_hi[order],
        "null_mean": null_means.mean(axis=0)[order],
        "n_units": len(prof),
        "obs_at_axis": float(obs[lag0]),
        "null_at_axis": float(null_hi[lag0]),
        "peak_excess": float(obs[lag0] - null_hi[lag0]),
    }


def plot_aggregate_asymmetry(binned: pd.DataFrame, resultants: pd.DataFrame, config,
                             score: str = "R", n_perm: int = 200,
                             save_as: str = "aggregate_asymmetry"):
    """Whole-dataset asymmetry with direction discarded.

    (a) distribution of the per-nucleus score by shell, against the geometry
        floor and the uniform-shell null -- magnitude only, no direction.
    (b) ECDF of the same, so the whole distribution is visible rather than a
        median.
    (c) the alignment-averaged profile: what the asymmetry LOOKS like once
        every nucleus is rotated onto its own axis, with the permutation null.
    """
    if resultants is None or resultants.empty:
        print("No data to plot."); return

    shells = sorted(resultants["shell_idx"].unique())
    labels = (resultants.drop_duplicates("shell_idx")
              .set_index("shell_idx")["shell_label"].to_dict())
    cmap = LinearSegmentedColormap.from_list(
        "shells", ["#7FD1B9", "#2A9D8F", "#3D5A99", "#1B1B2F"])
    colors = {sh: cmap(i / max(len(shells) - 1, 1)) for i, sh in enumerate(shells)}

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))

    # (a) violins of R_corr, with the geometry floor overlaid
    ax = axes[0]
    data = [resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy()
            for sh in shells]
    pos = np.arange(len(shells))
    keep = [i for i, d in enumerate(data) if d.size > 1]
    if keep:
        vp = ax.violinplot([data[i] for i in keep], positions=pos[keep],
                           showmedians=True, widths=0.75)
        for i, body in zip(keep, vp["bodies"]):
            body.set_facecolor(colors[shells[i]]); body.set_alpha(0.55)
        for k in ("cmins", "cmaxes", "cbars", "cmedians"):
            if k in vp:
                vp[k].set_color("0.25"); vp[k].set_linewidth(1.0)
    for i, sh in enumerate(shells):
        g = resultants.loc[resultants.shell_idx == sh, "R_geom"].median()
        ax.hlines(g, i - 0.38, i + 0.38, color="#D7263D", lw=1.8, ls=":", zorder=5)
    ax.plot([], [], color="#D7263D", lw=1.8, ls=":", label="median $R_{geom}$")
    med_neff = resultants["n_eff"].median()
    if np.isfinite(med_neff):
        ax.axhline(R_critical(med_neff), color="k", lw=0.9, ls="--", alpha=0.6,
                   label="uniform-shell $R$ at p=0.05")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{labels[sh]}" for sh in shells], fontsize=8)
    ax.set_xlabel("shell (\u00b5m from nuclear surface)")
    ax.set_ylabel("asymmetry score $R$")
    ax.set_title("(a) Score distribution, all nuclei \u00d7 all frames")
    ax.set_ylim(0, 1); ax.legend(fontsize=7.5)
    ax.spines[["top", "right"]].set_visible(False)

    # (b) ECDF
    ax = axes[1]
    for sh in shells:
        d = np.sort(resultants.loc[resultants.shell_idx == sh, score].dropna().to_numpy())
        if d.size == 0:
            continue
        ax.step(d, np.arange(1, d.size + 1) / d.size, where="post",
                color=colors[sh], lw=1.8, label=f"{labels[sh]}  (n={d.size})")
        g = np.sort(resultants.loc[resultants.shell_idx == sh, "R_geom"].dropna().to_numpy())
        if g.size:
            ax.step(g, np.arange(1, g.size + 1) / g.size, where="post",
                    color=colors[sh], lw=1.0, ls=":", alpha=0.8)
    ax.set_xlabel("asymmetry score $R$"); ax.set_ylabel("cumulative fraction")
    ax.set_title("(b) ECDF (dotted = $R_{geom}$)")
    ax.set_xlim(0, 1); ax.legend(fontsize=7)
    ax.spines[["top", "right"]].set_visible(False)

    # (c) alignment-averaged profile
    ax = axes[2]
    any_prof = False
    for sh in shells:
        pr = aligned_mean_profile(binned, resultants, sh, n_perm=n_perm)
        if pr is None:
            continue
        any_prof = True
        ax.plot(pr["lag_deg"], pr["mean"], color=colors[sh], lw=2,
                label=f"{labels[sh]}  (n={pr['n_units']})")
        ax.fill_between(pr["lag_deg"], pr["mean"] - 1.96 * pr["sem"],
                        pr["mean"] + 1.96 * pr["sem"], color=colors[sh], alpha=0.18, lw=0)
        if pr["null_hi"] is not None:
            ax.plot(pr["lag_deg"], pr["null_hi"], color=colors[sh], lw=0.9,
                    ls="--", alpha=0.75)
    if any_prof:
        ax.axhline(1.0, color="0.4", lw=0.8)
        ax.set_xlim(-180, 180); ax.set_xticks([-180, -90, 0, 90, 180])
        ax.plot([], [], color="0.3", lw=0.9, ls="--", label="permutation null (95th pct)")
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, "no fully-covered shell units", ha="center", va="center",
                transform=ax.transAxes, fontsize=9, color="0.4")
    ax.set_xlabel("angle from each nucleus's own asymmetry axis (\u00b0)")
    ax.set_ylabel("intensity / shell mean")
    ax.set_title("(c) Alignment-averaged profile")
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Aggregate membrane asymmetry \u2014 direction discarded", fontsize=13)
    fig.tight_layout()
    save_figure(fig, save_as, config)
    plt.show()

### 37f. Run the shell-resolved analysis

In [ ]:
# ── Shell-resolved asymmetry: full run ────────────────────────────────────
radial_n_df = add_wall_normalised_radius(radial_df, cfg)

# One background floor for the whole dataset — see 37b for why it must not
# come from the profile being scored.
BG = estimate_background(radial_n_df, q=0.05)
print(f"background floor: {BG:.1f} (5th percentile of all sweep samples)")

binned_df = bin_shell_angle(radial_n_df, cfg,
                            shell_edges_um=DEFAULT_SHELL_EDGES_UM,
                            n_theta_bins=N_THETA_BINS)
shell_res_df = compute_shell_resultants(binned_df, background=BG,
                                        min_coverage=MIN_SHELL_COVERAGE)
print(shell_res_df.shape)

# 1. the rose itself, for the longest-lived nucleus
_longest = int(tracked_df.groupby("track_id")["t"].nunique().idxmax())
plot_shell_rose_facets(binned_df, shell_res_df, cfg, track_id=_longest,
                       value="mean_bgsub", radius_scale="sqrt", share_r="row",
                       save_as=f"shell_rose_track{_longest}")

# 2. normalised asymmetry score — table + graph
shell_score_table = plot_asymmetry_score_summary(shell_res_df, cfg)
display(shell_score_table)

# 3. aggregate across the whole dataset, direction discarded
plot_aggregate_asymmetry(binned_df, shell_res_df, cfg, n_perm=200)

# exports
shell_res_df.to_csv(cfg.exports_dir / "shell_asymmetry_per_nucleus.csv", index=False)
shell_score_table.to_csv(cfg.exports_dir / "shell_asymmetry_score_table.csv", index=False)
binned_df.to_pickle(cfg.analysis_dir / "shell_angle_binned.pkl")
print("figures ->", figures_dir(cfg))

In [ ]:
# ── Sanity read of the geometry control ───────────────────────────────────
# Any shell where R does not clear R_geom is reporting droplet placement,
# not membrane. Read this before interpreting the rose.
chk = (shell_res_df.groupby("shell_label")
       .agg(n=("R", "size"),
            R=("R", "median"),
            R_geom=("R_geom", "median"),
            excess=("R_excess", "median"),
            coverage=("coverage", "min"),
            frac_sig=("significant", "mean"))
       .round(3))
display(chk)

flagged = chk[chk.R_geom >= 0.5 * chk.R]
if len(flagged):
    print("\nShells where sampling geometry accounts for half or more of the "
          "measured asymmetry — do not quote these as membrane asymmetry:")
    display(flagged)
else:
    print("\nNo shell has R_geom within 2x of R.")

In [ ]:
PIPELINE_VERSION = "v16.3"  # matches this notebook; bump manually per release
DB_EXPORT_DIR = cfg.output_dir / "database_export"
DB_EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def _fmt_slice_id(z: int) -> str:
    """Match the S#### convention used in the schema workbook."""
    return f"S{int(z):04d}"


def _fmt_centroid_xyz(x: float, y: float, z: float) -> str:
    return f"({x:.1f}, {y:.1f}, {z:.1f})"


In [ ]:
def build_nuclei_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per Nucleus_ID (track_id) per Time_Frame — analysis-ready table."""
    if halo_df.empty:
        return pd.DataFrame(columns=[
            "Nucleus_ID", "Droplet_ID", "FOV_ID", "Experiment_ID",
            "Time_Frame", "Time_Real_Minutes", "Stage_Classification",
            "Selected_Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Volume_3D_um3", "NC_Ratio",
            "Segmentation_Pipeline_Version", "Model_Version",
            "QC_Flag", "QC_Notes", "Source_File_Path",
        ])

    experiment_id = Path(cfg.input_image_name).stem
    rows = []
    for row in halo_df.itertuples(index=False):
        qc_pass = bool(getattr(row, "valid_single_nucleus", True))
        rows.append({
            "Nucleus_ID":                    int(row.track_id),
            "Droplet_ID":                    "",  # not tracked as a distinct ID by this pipeline
            "FOV_ID":                        "FOV01",  # single stitched image per run
            "Experiment_ID":                 experiment_id,
            "Time_Frame":                    int(row.t),
            "Time_Real_Minutes":             float(getattr(row, "true_time_min", np.nan)),
            "Stage_Classification":          "",  # not computed by this pipeline
            "Selected_Slice_ID":             _fmt_slice_id(row.z),
            "Centroid_XYZ_px":               _fmt_centroid_xyz(row.centroid_x_px, row.centroid_y_px, row.z),
            "Cross_Sectional_Area_um2":      float(getattr(row, "nucleus_area_um2", np.nan)),
            "Volume_3D_um3":                 np.nan,  # 2D best-Z pipeline only
            "NC_Ratio":                      float(getattr(row, "nc_ratio", np.nan)),
            "Segmentation_Pipeline_Version": PIPELINE_VERSION,
            "Model_Version":                 cfg.model_name,
            "QC_Flag":                       "PASS" if qc_pass else "FAIL",
            "QC_Notes":                      "" if qc_pass else "excluded: multiple nuclei within exclusion radius",
            "Source_File_Path":              cfg.input_image_name,
        })
    return pd.DataFrame(rows)


nuclei_export_df = build_nuclei_table(halo_df, cfg)
print(nuclei_export_df.shape)
display(nuclei_export_df.head())


In [ ]:
def build_zstack_table(grouped_z_df: pd.DataFrame, best_z_df: pd.DataFrame,
                       tracked_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """One row per Z-plane per nucleus per timepoint — full sweep behind the best-Z pick."""
    cols = ["Nucleus_ID", "Time_Frame", "Slice_ID", "Centroid_XYZ_px",
            "Cross_Sectional_Area_um2", "Is_Selected_Max"]
    if grouped_z_df.empty or best_z_df.empty or tracked_df.empty:
        return pd.DataFrame(columns=cols)

    track_lookup = tracked_df[["t", "nucleus_3d_id", "track_id"]].drop_duplicates()
    best_z_lookup = best_z_df[["t", "nucleus_3d_id", "z"]].rename(columns={"z": "best_z"})

    merged = grouped_z_df.merge(track_lookup, on=["t", "nucleus_3d_id"], how="inner")
    merged = merged.merge(best_z_lookup, on=["t", "nucleus_3d_id"], how="left")

    rows = []
    for row in merged.itertuples(index=False):
        area_um2 = float(row.area_px) * (cfg.pixel_size_um ** 2)
        rows.append({
            "Nucleus_ID":               int(row.track_id),
            "Time_Frame":               int(row.t),
            "Slice_ID":                 _fmt_slice_id(row.z),
            "Centroid_XYZ_px":          _fmt_centroid_xyz(row.centroid_x_px, row.centroid_y_px, row.z),
            "Cross_Sectional_Area_um2": area_um2,
            "Is_Selected_Max":          bool(row.z == row.best_z),
        })
    return pd.DataFrame(rows, columns=cols)


zstack_export_df = build_zstack_table(grouped_z_df, best_z_df, tracked_df, cfg)
print(zstack_export_df.shape)
display(zstack_export_df.head())


In [ ]:
def build_raw_intensities_table(halo_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Per-halo, per-channel integrated intensities at the selected Z-slice.

    NOTE: only the Mcherry/NLS channel is currently measured by
    measure_fiji_equivalent_halos (Section 18). NPC and Membrane columns,
    and all Background_* columns, are written as NaN until that function
    is extended to loop over channels.
    """
    cols = [
        "Nucleus_ID", "Time_Frame",
        "Halo1_Mcherry", "Halo2_Mcherry", "Halo3_Mcherry", "Halo4_Mcherry",
        "Halo1_NPC", "Halo2_NPC", "Halo3_NPC", "Halo4_NPC",
        "Halo1_Membrane", "Halo2_Membrane", "Halo3_Membrane", "Halo4_Membrane",
        "Background_Mcherry", "Background_NPC", "Background_Membrane",
    ]
    if halo_df.empty:
        return pd.DataFrame(columns=cols)

    rows = []
    for row in halo_df.itertuples(index=False):
        rd = row._asdict()
        rec = {
            "Nucleus_ID": int(rd["track_id"]),
            "Time_Frame": int(rd["t"]),
        }
        for i in range(1, cfg.n_halos + 1):
            rec[f"Halo{i}_Mcherry"] = rd.get(f"ring_{i}_intden", np.nan)
            rec[f"Halo{i}_NPC"] = np.nan       # not measured yet — see docstring
            rec[f"Halo{i}_Membrane"] = np.nan  # not measured yet — see docstring
        rec["Background_Mcherry"] = np.nan
        rec["Background_NPC"] = np.nan
        rec["Background_Membrane"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows, columns=cols)


raw_intensities_export_df = build_raw_intensities_table(halo_df, cfg)
print(raw_intensities_export_df.shape)
display(raw_intensities_export_df.head())


In [ ]:
def build_radial_profile_table(radial_df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    """Long-form radial sweep -> schema RadialProfile (one row per ray sample point).

    W_Wall_Proximity_Index, Is_Ridge_Point, and Cluster_ID are written as
    placeholders (NaN / False / blank) -- wall-proximity index and ridge/DBSCAN
    clustering are not computed in v16.
    """
    cols = [
        "Nucleus_ID", "Time_Frame", "Theta_deg", "Rho_Normalized",
        "W_Wall_Proximity_Index", "Channel", "Intensity",
        "Is_Ridge_Point", "Cluster_ID",
    ]
    if radial_df is None or radial_df.empty:
        return pd.DataFrame(columns=cols)

    channel_label_by_index = {
        cfg.membrane_channel_index: "Membrane",
        cfg.nuclear_channel_index:  "Mcherry",
        cfg.npc_channel_index:      "NPC",
    }
    radial_channel_label = channel_label_by_index.get(cfg.radial_channel_index, "Membrane")

    out = pd.DataFrame({
        "Nucleus_ID":             radial_df["track_id"].astype(int),
        "Time_Frame":             radial_df["t"].astype(int),
        "Theta_deg":              np.degrees(radial_df["theta_rad"].astype(float)),
        "Rho_Normalized":         radial_df["distance_norm"].astype(float),
        "W_Wall_Proximity_Index": np.nan,
        "Channel":                radial_channel_label,
        "Intensity":              radial_df["intensity"].astype(float),
        "Is_Ridge_Point":         False,
        "Cluster_ID":             "",
    })
    return out[cols]


radial_profile_export_df = build_radial_profile_table(radial_df, cfg)
print(radial_profile_export_df.shape)
display(radial_profile_export_df.head())


In [ ]:
def build_experimental_cfg_table(cfg: PipelineConfig) -> pd.DataFrame:
    experiment_id = Path(cfg.input_image_name).stem
    channel_labels = {0: "Membrane", 1: "NLS", 2: "NPC"}
    row = {
        "Experiment_ID":                 experiment_id,
        "FOV_ID":                        "FOV01",
        "Droplet_ID":                    "",
        "Date":                          datetime.utcnow().strftime("%Y-%m-%d"),
        "Pixel_Size_um":                 cfg.pixel_size_um,
        "Z_Step_um":                     cfg.z_step_um,
        "Num_Z_Planes":                  int(img_5d.shape[1]) if "img_5d" in dir() else np.nan,
        "Frame_Interval_min":            cfg.minutes_per_tile * cfg.tile_rows * cfg.tile_cols,
        "Channel0_Label":                channel_labels.get(0, ""),
        "Channel1_Label":                channel_labels.get(1, ""),
        "Channel2_Label":                channel_labels.get(2, ""),
        "Microscope":                    "",
        "Objective":                     "",
        "Operator":                      "",
        "Segmentation_Pipeline_Version": PIPELINE_VERSION,
        "Model_Version":                 cfg.model_name,
        "Raw_File_Path":                 str(cfg.input_image_path),
        "Notes":                         "",
    }
    return pd.DataFrame([row])


experimental_cfg_export_df = build_experimental_cfg_table(cfg)
display(experimental_cfg_export_df)


In [ ]:
def export_database_schema_tables(export_dir: Path) -> dict:
    tables = {
        "Nuclei":            nuclei_export_df,
        "NucleusZStack":     zstack_export_df,
        "RawIntensities":    raw_intensities_export_df,
        "RadialProfile":     radial_profile_export_df,
        "Experimental_Cfg":  experimental_cfg_export_df,
    }
    written = {}
    for name, df in tables.items():
        out_path = export_dir / f"{name}.csv"
        df.to_csv(out_path, index=False)
        written[name] = str(out_path)
        print(f"{name}: {len(df)} rows -> {out_path}")
    return written


RUN_DB_SCHEMA_EXPORT = True

if RUN_DB_SCHEMA_EXPORT:
    written_paths = export_database_schema_tables(DB_EXPORT_DIR)


In [ ]:
r = (d.assign(frac=d.nuc_px/d.drop_px)
       .groupby("t")["frac"]
       .describe(percentiles=[.5,.9,.95,.99])[["50%","90%","95%","99%","max"]])
print(r.round(3))

In [ ]:
rows = []
for (t, z), grp in b.groupby(["t", "z"]):          # drop the t>=7 filter
    plane = np.asarray(drop[t, z])
    props = {r.label: r.area for r in measure.regionprops(plane)}
    for r in grp.itertuples():
        did = plane[int(r.centroid_y_px), int(r.centroid_x_px)]
        rows.append({"t": t, "nuc_px": r.area_px,
                     "drop_px": props.get(did, np.nan),
                     "assigned": did > 0})
dall = pd.DataFrame(rows)
dall["frac"] = dall.nuc_px / dall.drop_px
print(dall.groupby("t")["frac"].describe(percentiles=[.5,.9,.95,.99])[["50%","90%","95%","99%","max"]].round(3))
print("\nunassigned (droplet_id=0) by t:")
print(dall.groupby("t")["assigned"].apply(lambda s: 1 - s.mean()).round(3))

In [ ]:
import pandas as pd

tm = pd.read_pickle(cfg.track_dir / "best_z_nuclei_timed.pkl")
tm["area_um2"] = tm.area_px * cfg.pixel_size_um**2

EARLY_MIN, EARLY_MAX = 50.0, 200.0     # true_time_min <= 20
LATE_MIN,  LATE_MAX  = 200.0, 500.0    # true_time_min >  20

early = tm.true_time_min <= 20
keep = ((early  & tm.area_um2.between(EARLY_MIN, EARLY_MAX)) |
        (~early & tm.area_um2.between(LATE_MIN,  LATE_MAX)))

tm_f = tm[keep].copy()
tm_f.to_pickle(cfg.track_dir / "best_z_nuclei_timed_filtered.pkl")

print(f"kept {len(tm_f)} / {len(tm)}  ({100*len(tm_f)/len(tm):.1f}%)")
print("\nretained by frame:")
print(pd.DataFrame({"before": tm.groupby("t").size(),
                    "after":  tm_f.groupby("t").size()}).fillna(0).astype(int))
print("\nmedian area µm² after filter:")
print(tm_f.groupby("t")["area_um2"].median().round(1))

In [ ]:
import pandas as pd

halo_df = pd.read_pickle(cfg.analysis_dir / "halo_analysis.pkl")

# recompute area from px so it always reflects the current calibration
if "nucleus_area_px" in halo_df.columns:
    halo_df["nucleus_area_um2"] = halo_df.nucleus_area_px * cfg.pixel_size_um**2

early = halo_df.true_time_min <= 20
keep = ((early  & halo_df.nucleus_area_um2.between(50, 200)) |
        (~early & halo_df.nucleus_area_um2.between(200, 500)))
hf = halo_df[keep].copy()

print(f"kept {len(hf)} / {len(halo_df)} ({100*len(hf)/len(halo_df):.1f}%)")
print(pd.DataFrame({"before": halo_df.groupby("t").size(),
                    "after":  hf.groupby("t").size()}).fillna(0).astype(int))

plot_area_vs_time(hf)
plot_nc_vs_time(hf)
plot_largest_cross_sectional_area_vs_time(hf, cfg)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col, lab in [(ax[0], "nucleus_area_um2", "Nuclear area (µm²)"),
                    (ax[1], "nc_ratio_fraction", "N / (N + C)")]:
    g = hf.groupby("true_time_min")[col]
    med, lo, hi = g.median(), g.quantile(.25), g.quantile(.75)
    a.fill_between(med.index, lo, hi, alpha=.25)
    a.plot(med.index, med, lw=2)
    a.set_xlabel("True acquisition time (min)"); a.set_ylabel(lab)
a.figure.suptitle("Median ± IQR, filtered nuclei"); plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd

# nuclei per frame and per true acquisition minute
counts = pd.DataFrame({
    "rows":          hf.groupby("t").size(),
    "unique_tracks": hf.groupby("t")["track_id"].nunique(),
    "unique_droplets": hf.groupby("t")["droplet_id"].nunique() if "droplet_id" in hf else None,
}).fillna(0).astype(int)
counts["dupes"] = counts["rows"] - counts["unique_tracks"]
print(counts)

print("\nper true acquisition minute:")
print(hf.groupby("true_time_min")["track_id"].nunique())

# how persistent are tracks across the timecourse?
span = hf.groupby("track_id")["t"].nunique()
print("\ntimepoints per track:")
print(span.value_counts().sort_index())
print(f"\ntotal unique nuclei (any t): {hf.track_id.nunique()}")
print(f"tracks present at all 10 t:   {(span == hf.t.nunique()).sum()}")

In [ ]:
import numpy as np, tifffile as tiff
from skimage import measure
drop = tiff.memmap(cfg.droplet_instance_hyperstack_path, mode="r")
for t in range(drop.shape[0]):
    z = int(hf[hf.t == t]["z"].median()) if (hf.t == t).any() else drop.shape[1]//2
    n = len([r for r in measure.regionprops(np.asarray(drop[t, z])) if r.area > 5000])
    print(f"t={t}  droplets={n:4d}  nuclei_kept={int((hf.t==t).sum()):4d}")

In [ ]:
import numpy as np, pandas as pd, tifffile as tiff
from skimage import measure

prob = tiff.memmap(cfg.segmentation_probability_hyperstack_path, mode="r")  # (T,Z,C,Y,X)
NUC = cfg.nucleus_class_index

tm = pd.read_pickle(cfg.track_dir / "best_z_nuclei_timed.pkl")
tm["area_um2"] = tm.area_px * cfg.pixel_size_um**2

recs = []
for (t, z), grp in tm.groupby(["t", "z"]):
    lab = np.load(cfg.seg_dir / f"nuclear_mask_t{int(t):03d}_z{int(z):03d}.npy")
    lab = measure.label(lab > 0) if lab.dtype == bool else lab
    pplane = np.asarray(prob[int(t), int(z), NUC]).astype(np.float32)
    tbl = pd.DataFrame(measure.regionprops_table(
        lab, intensity_image=pplane,
        properties=("label", "mean_intensity", "area")))
    lut = dict(zip(tbl.label, tbl.mean_intensity))
    for r in grp.itertuples():
        l = int(lab[int(r.centroid_y_px), int(r.centroid_x_px)])
        recs.append({"t": r.t, "z": r.z, "track_id": getattr(r, "track_id", np.nan),
                     "centroid_x_px": r.centroid_x_px, "centroid_y_px": r.centroid_y_px,
                     "mean_prob": lut.get(l, np.nan)})

pr = pd.DataFrame(recs)
tm = tm.merge(pr, on=["t","z","centroid_x_px","centroid_y_px"], how="left")

print(tm.groupby("t")["mean_prob"].describe(percentiles=[.1,.25,.5,.75,.9])
        [["10%","25%","50%","75%","90%","max"]].round(3))
print("\nfraction with mean_prob > 0.9, by t:")
print(tm.groupby("t")["mean_prob"].apply(lambda s: (s > 0.9).mean()).round(3))

In [ ]:
tmp = tm[tm.mean_prob > 0.7].copy()
print(f"kept {len(tmp)}/{len(tm)}; unique nuclei/t:")
print(tmp.groupby("t")["track_id"].nunique())

plot_largest_cross_sectional_area_vs_time(tmp, cfg)

hf_p = halo_df.merge(tmp[["t","track_id"]].drop_duplicates(), on=["t","track_id"])
plot_nc_vs_time(hf_p)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

radial_df = pd.read_pickle(cfg.analysis_dir / "radial_sweep.pkl")

# --- pick the longest-lived track (or set TRACK manually) ---
span = radial_df.groupby("track_id")["t"].nunique().sort_values(ascending=False)
TRACK = int(span.index[0])
print(f"track {TRACK}: present at t = {sorted(radial_df[radial_df.track_id==TRACK].t.unique())}")

sub = radial_df[radial_df.track_id == TRACK].copy()

# --- radial axis: normalised (nucleus edge -> droplet edge) ---
RBINS = 60
sub["rbin"] = pd.cut(sub.distance_norm, np.linspace(0, 1, RBINS + 1), labels=False)

ts = sorted(sub.t.unique())
grids = {t: (sub[sub.t == t]
             .pivot_table(index="theta_index", columns="rbin",
                          values="intensity", aggfunc="mean")
             .reindex(index=range(cfg.radial_n_angles), columns=range(RBINS)))
         for t in ts}

allv = np.concatenate([g.to_numpy().ravel() for g in grids.values()])
vmin, vmax = np.nanpercentile(allv, [2, 98])

ncol = min(5, len(ts)); nrow = int(np.ceil(len(ts) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 2.9*nrow),
                         squeeze=False, constrained_layout=True)
for ax, t in zip(axes.ravel(), ts):
    im = ax.imshow(grids[t].to_numpy(), aspect="auto", origin="lower",
                   extent=[0, 1, 0, 360], vmin=vmin, vmax=vmax, cmap="inferno")
    ax.set_title(f"t={t}", fontsize=9)
    ax.set_xlabel("r (nucleus edge → droplet edge)", fontsize=7)
    ax.set_ylabel("θ (deg)", fontsize=7)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(ts):]:
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.7, label="membrane intensity")
fig.suptitle(f"Radial membrane profile — track {TRACK}", fontsize=11)
plt.show()

In [ ]:
# ============================================================================
#  POST-REPAIR SMALL-NUCLEUS CONTACT SHEET
#
#  Same idea as before, one difference that matters: repaired masks were never
#  written back to nucleus_instance_hyperstack, so a repaired nucleus has no
#  mask on disk to contour. This regenerates it by re-running repair_plane at
#  the stored z — deterministic, so what you see is exactly the mask that
#  produced the number in the table.
#
#  Contour colour tells you which path produced it:
#    cyan   = repaired (watershed)
#    red    = original mask, flagged fragmented but repair was REJECTED
#    green  = original mask, never flagged
#    yellow = droplet
#
#  Requires the v17 repair stage (repair_plane, _resolve_instance) in scope.
# ============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt


def small_nucleus_sheet_v17(df, img_5d, nucleus_instance_4d,
                            droplet_instance_4d, cfg,
                            area_max=250.0, n_per_t=8, t_min=None,
                            win_um=45.0, tile_in=2.2, area_col=None,
                            id_col=None, exclude_edge=True):
    """
    Grid of every nucleus below `area_max`, cropped at its own centroid.

    Rows = timepoint, columns = the n smallest at that timepoint.
    """
    d = df.copy()
    if exclude_edge and "exclude_edge" in d.columns:
        d = d[~d.exclude_edge]
    if area_col is None:
        area_col = ("nucleus_area_um2" if "nucleus_area_um2" in d.columns
                    else "area_um2")
    if id_col is None:
        for c in ("track_id", "nucleus_3d_id", "Nucleus_ID"):
            if c in d.columns:
                id_col = c; break
    if t_min is not None:
        d = d[d.t >= t_min]

    small = d[d[area_col] < area_max].copy()
    print(f"{len(small)} of {len(d)} nuclei below {area_max:.0f} µm² "
          f"({100*len(small)/max(len(d),1):.1f}%)")
    if "repair_status" in small.columns:
        print(small.repair_status.value_counts().to_string())
    zero = int((small[area_col] <= 1.0).sum())
    if zero:
        print(f"\n{zero} have area <= 1 µm² — those are not nuclei at all, they are "
              f"empty or failed rows. Check them first.")
    if small.empty:
        return small

    picks = (small.sort_values(["t", area_col])
                  .groupby("t").head(n_per_t).reset_index(drop=True))
    ts = sorted(picks.t.unique())
    ncol = int(picks.groupby("t").size().max())
    half = int(round((win_um / cfg.pixel_size_um) / 2))
    H, W = img_5d.shape[-2], img_5d.shape[-1]
    px2 = cfg.pixel_size_um ** 2

    fig, ax = plt.subplots(len(ts), ncol,
                           figsize=(tile_in * ncol, tile_in * 1.30 * len(ts)))
    ax = np.atleast_2d(ax).reshape(len(ts), ncol)

    for ri, t in enumerate(ts):
        sub = picks[picks.t == t].reset_index(drop=True)
        for ci in range(ncol):
            a = ax[ri, ci]
            a.set_xticks([]); a.set_yticks([])
            for s in a.spines.values():
                s.set_visible(False)
            if ci >= len(sub):
                a.axis("off"); continue

            r = sub.iloc[ci]
            cx, cy, z = float(r.centroid_x_px), float(r.centroid_y_px), int(r.z)
            r0, c0 = max(int(cy) - half, 0), max(int(cx) - half, 0)
            r1, c1 = min(int(cy) + half, H), min(int(cx) + half, W)
            sl = (slice(r0, r1), slice(c0, c1))

            nls = np.asarray(img_5d[t, z, cfg.nuclear_channel_index],
                             np.float32)[sl]
            lo, hi = np.percentile(nls, [1, 99.5])
            a.imshow(np.clip((nls - lo) / (hi - lo + 1e-8), 0, 1), cmap="gray")

            dpl = np.asarray(droplet_instance_4d[t, z])
            did = _resolve_instance(dpl, cx, cy)
            if did:
                a.contour((dpl == did)[sl], levels=[0.5], colors="#ffd700",
                          linewidths=0.7)

            status = str(r.get("repair_status", "")) if hasattr(r, "get") else ""
            drawn = np.nan
            if status == "repaired":
                mask, _, off = repair_plane(t, z, cx, cy, img_5d,
                                            nucleus_instance_4d,
                                            droplet_instance_4d, cfg)
                if mask.any():
                    full = np.zeros((H, W), bool)
                    full[off[0]:off[0] + mask.shape[0],
                         off[1]:off[1] + mask.shape[1]] = mask
                    a.contour(full[sl], levels=[0.5], colors="c", linewidths=1.5)
                    drawn = float(mask.sum()) * px2
                colour_note = "repaired"
            else:
                inst = np.asarray(nucleus_instance_4d[t, z])
                lb = _resolve_instance(inst, cx, cy)
                col = "red" if status.startswith("rejected") else "lime"
                if lb:
                    m = (inst == lb)[sl]
                    if m.any():
                        a.contour(m, levels=[0.5], colors=col, linewidths=1.5)
                        drawn = float(m.sum()) * px2
                colour_note = ("repair rejected" if status.startswith("rejected")
                               else "not flagged")

            nid = r[id_col] if id_col else ci
            cap = f"{id_col}={nid}  z={z}\n{r[area_col]:.0f} µm²  ({colour_note})"
            if np.isfinite(drawn) and abs(drawn - r[area_col]) > 0.05 * max(r[area_col], 1):
                cap += f"\nmask now {drawn:.0f} µm²"
            cap += f"\nx={cx:.0f} y={cy:.0f}"
            a.set_title(cap, fontsize=6.0, pad=2)

        ax[ri, 0].set_ylabel(f"t = {t}", fontsize=11, rotation=0,
                             labelpad=26, va="center")

    fig.suptitle(f"Nuclei below {area_max:.0f} µm² after repair — "
                 f"cyan repaired, red repair-rejected, green never flagged, "
                 f"yellow droplet", fontsize=11, y=0.998)
    plt.tight_layout(rect=[0, 0, 1, 0.985]); plt.show()
    return picks


def small_nucleus_breakdown(df, cfg, area_max=250.0, area_col=None,
                            exclude_edge=True):
    """Where do the small ones come from? Answer before looking at tiles."""
    d = df.copy()
    if exclude_edge and "exclude_edge" in d.columns:
        d = d[~d.exclude_edge]
    if area_col is None:
        area_col = ("nucleus_area_um2" if "nucleus_area_um2" in d.columns
                    else "area_um2")
    d["is_small"] = d[area_col] < area_max
    out = d.groupby(["t", "is_small"]).size().unstack(fill_value=0)
    out.columns = ["large", "small"] if out.shape[1] == 2 else out.columns
    out["pct_small"] = (100 * out["small"] / out.sum(axis=1)).round(1)
    print(out.to_string())
    if "repair_status" in d.columns:
        print("\nsmall nuclei by repair status:")
        print(d[d.is_small].repair_status.value_counts().to_string())
        print("\nlarge nuclei by repair status:")
        print(d[~d.is_small].repair_status.value_counts().to_string())
    if "ratio_vs_npc" in d.columns:
        s = d[d.is_small & d.ratio_vs_npc.notna()]
        if len(s):
            print(f"\nsmall repaired nuclei, median ratio_vs_npc: "
                  f"{s.ratio_vs_npc.median():.3f}")
            print("If this is near 0.9 the mask matches its own NPC ring — the")
            print("nucleus really is small. If it is ~0.5 the repair fell short.")
    return d

In [ ]:
small_nucleus_breakdown(best_z_df, cfg, area_max=250.0)

picks = small_nucleus_sheet_v17(best_z_df, img_5d, nucleus_instance_4d,
                                droplet_instance_4d, cfg,
                                area_max=250.0, n_per_t=8)

In [ ]:
d = best_z_df[~best_z_df.exclude_edge]
print(d.repair_status.str.split(":").str[0].value_counts())

rej = d[d.repair_status.str.startswith("rejected")]
print(f"\n{len(rej)} rejected, median area {rej.nucleus_area_um2.median():.0f} µm²")
print(rej.repair_status.value_counts().head(10).to_string())

small = d[d.nucleus_area_um2 < 250]
print(f"\nof {len(small)} nuclei under 250: "
      f"{int(small.repair_status.str.startswith('rejected').sum())} are rejected repairs")

In [ ]:
# ============================================================================
#  FOCUS METRIC + FINAL AREA PLOT
#
#  Two things:
#
#  1. edge_sharpness() — an explicit focus measure, since a defocused nucleus
#     has a soft boundary while an in-focus one is sharply defined. Measured as
#     the 10-90% rise distance of the intensity step across the mask boundary,
#     in um. In focus this approaches the PSF width; 3 planes off it broadens.
#
#     This also explains the rejection pattern: ALL 44 rejections were
#     "low signed contrast" at 1.04-1.15, clustered just under the 1.15 cut.
#     A blurred edge spills nuclear signal into the shell just outside the
#     mask, which lowers the interior/shell ratio. So that gate was detecting
#     defocus indirectly. Measuring it directly is better: it separates "this
#     repair is wrong" from "this plane is out of focus", which need different
#     handling.
#
#  2. plot_final_area() — the growth curve with failed repairs REMOVED rather
#     than carried at fragment size. A rejected repair is an UNMEASURED
#     nucleus, not a small one; leaving it in plots a known-wrong number.
#
#  NOTE on z-selection: the z-search takes maximum AREA. A defocused plane can
#  measure larger than the true equator because blur spreads the intensity
#  outward, so max-area selection can actively prefer out-of-focus planes.
#  select_sharpest_z() below re-selects on sharpness among planes near the
#  area peak, which is the safer criterion.
# ============================================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import measure, morphology


def edge_sharpness(mask, nls, cfg, band_um=3.0, n_samples=400):
    """
    10-90% rise distance across the mask boundary, in um. Small = sharp.

    Samples the signed distance from the boundary and fits where the intensity
    profile crosses 10% and 90% of the inside-outside step. Independent of
    absolute brightness, so it is comparable across timepoints.
    """
    if not mask.any():
        return np.nan
    band_px = max(int(round(band_um / cfg.pixel_size_um)), 2)
    d_in = ndi.distance_transform_edt(mask)
    d_out = ndi.distance_transform_edt(~mask)
    signed = np.where(mask, d_in, -d_out)          # + inside, - outside
    sel = np.abs(signed) <= band_px
    if sel.sum() < 50:
        return np.nan
    s, v = signed[sel], nls[sel].astype(np.float32)
    if s.size > n_samples:
        idx = np.random.default_rng(0).choice(s.size, n_samples, replace=False)
        s, v = s[idx], v[idx]
    order = np.argsort(s)
    s, v = s[order], v[order]
    k = max(len(s) // 12, 3)
    sm = np.convolve(v, np.ones(k) / k, mode="same")
    lo, hi = np.percentile(sm, 5), np.percentile(sm, 95)
    if hi <= lo:
        return np.nan
    t10, t90 = lo + 0.1 * (hi - lo), lo + 0.9 * (hi - lo)
    try:
        x10 = s[np.argmin(np.abs(sm - t10))]
        x90 = s[np.argmin(np.abs(sm - t90))]
    except ValueError:
        return np.nan
    return float(abs(x90 - x10) * cfg.pixel_size_um)


def add_focus_metric(d, img_5d, nucleus_instance_4d, droplet_instance_4d, cfg,
                     verbose=True):
    """Add edge_width_um to every row, regenerating repaired masks as needed."""
    d = d.copy()
    widths = []
    half = int(round((cfg.repair_win_um / cfg.pixel_size_um) / 2))
    H, W = img_5d.shape[-2], img_5d.shape[-1]
    for r in d.itertuples():
        t, z = int(r.t), int(r.z)
        cx, cy = float(r.centroid_x_px), float(r.centroid_y_px)
        r0, c0 = max(int(cy) - half, 0), max(int(cx) - half, 0)
        r1, c1 = min(int(cy) + half, H), min(int(cx) + half, W)
        sl = (slice(r0, r1), slice(c0, c1))
        nls = np.asarray(img_5d[t, z, cfg.nuclear_channel_index], np.float32)[sl]
        if str(getattr(r, "repair_status", "")) == "repaired":
            mask, _, off = repair_plane(t, z, cx, cy, img_5d,
                                        nucleus_instance_4d,
                                        droplet_instance_4d, cfg)
            if mask.shape != nls.shape:
                widths.append(np.nan); continue
        else:
            inst = np.asarray(nucleus_instance_4d[t, z])
            lb = _resolve_instance(inst, cx, cy)
            mask = (inst == lb)[sl] if lb else np.zeros(nls.shape, bool)
        widths.append(edge_sharpness(mask, nls, cfg))
    d["edge_width_um"] = widths
    if verbose:
        v = d.edge_width_um.dropna()
        print(f"edge width: median {v.median():.2f} um  "
              f"p10 {v.quantile(.1):.2f}  p90 {v.quantile(.9):.2f}")
        if "repair_status" in d.columns:
            print("\nby repair status:")
            print(d.groupby(d.repair_status.str.split(":").str[0])
                  .edge_width_um.median().round(2).to_string())
    return d


def select_sharpest_z(row, img_5d, nucleus_instance_4d, droplet_instance_4d,
                      cfg, z_window=3, area_tol=0.15):
    """
    Among planes within `area_tol` of the area peak, take the SHARPEST.

    Max-area selection can prefer a defocused plane, because blur spreads
    intensity outward and inflates the measured area. Sharpness breaks that
    tie on the right criterion.
    """
    t, z0 = int(row.t), int(row.z)
    cx, cy = float(row.centroid_x_px), float(row.centroid_y_px)
    Z = img_5d.shape[1]
    half = int(round((cfg.repair_win_um / cfg.pixel_size_um) / 2))
    H, W = img_5d.shape[-2], img_5d.shape[-1]
    r0, c0 = max(int(cy) - half, 0), max(int(cx) - half, 0)
    r1, c1 = min(int(cy) + half, H), min(int(cx) + half, W)
    sl = (slice(r0, r1), slice(c0, c1))

    cands = []
    for z in range(max(z0 - z_window, 0), min(z0 + z_window + 1, Z)):
        mask, m, _ = repair_plane(t, z, cx, cy, img_5d, nucleus_instance_4d,
                                  droplet_instance_4d, cfg)
        if not mask.any():
            continue
        nls = np.asarray(img_5d[t, z, cfg.nuclear_channel_index], np.float32)[sl]
        cands.append({"z": z, "area_um2": m["area_um2"],
                      "edge_width_um": edge_sharpness(mask, nls, cfg)})
    if not cands:
        return z0, np.nan, np.nan
    c = pd.DataFrame(cands)
    peak = c.area_um2.max()
    near = c[c.area_um2 >= (1 - area_tol) * peak].dropna(subset=["edge_width_um"])
    if near.empty:
        best = c.loc[c.area_um2.idxmax()]
    else:
        best = near.loc[near.edge_width_um.idxmin()]
    return int(best.z), float(best.area_um2), float(best.get("edge_width_um", np.nan))


def plot_final_area(d, cfg, time_col=None, area_col="nucleus_area_um2",
                    max_edge_um=None, figsize=(16, 5)):
    """
    The growth curve, with failed repairs removed.

    Three exclusions, all flags rather than deletions:
      exclude_edge                 clipped by the frame
      repair_status startswith rejected   unmeasured, not small
      edge_width_um > max_edge_um  out of focus (optional)
    """
    if time_col is None:
        time_col = "true_time_min" if "true_time_min" in d.columns else "t"
    d = d.copy()
    d["rejected"] = d.repair_status.astype(str).str.startswith("rejected")
    keep = ~d.exclude_edge & ~d.rejected
    if max_edge_um is not None and "edge_width_um" in d.columns:
        keep &= (d.edge_width_um <= max_edge_um) | d.edge_width_um.isna()
    kept, dropped = d[keep], d[~keep]

    print(f"{len(d)} nuclei -> {len(kept)} kept, {len(dropped)} removed")
    print(f"   edge-clipped        {int(d.exclude_edge.sum())}")
    print(f"   repair failed       {int(d.rejected.sum())}")
    if max_edge_um is not None and "edge_width_um" in d.columns:
        print(f"   out of focus (>{max_edge_um} um)  "
              f"{int((d.edge_width_um > max_edge_um).sum())}")

    fig, ax = plt.subplots(1, 3, figsize=figsize)

    a = ax[0]
    a.scatter(dropped[time_col], dropped[area_col], s=12, alpha=0.35,
              c="lightgray", label=f"removed (n={len(dropped)})")
    rep = kept[kept.repair_status == "repaired"]
    nf = kept[kept.repair_status == "not fragmented"]
    a.scatter(nf[time_col], nf[area_col], s=13, alpha=0.6, c="tab:blue",
              label=f"never fragmented (n={len(nf)})")
    a.scatter(rep[time_col], rep[area_col], s=15, alpha=0.75, c="tab:green",
              label=f"repaired (n={len(rep)})")
    a.set_xlabel(time_col); a.set_ylabel("nuclear cross-sectional area (µm²)")
    a.set_title("final population"); a.legend(fontsize=8); a.grid(alpha=0.25)

    a = ax[1]
    if "track_id" in kept.columns:
        for _, g in kept.groupby("track_id"):
            if len(g) > 1:
                g = g.sort_values(time_col)
                a.plot(g[time_col], g[area_col], "-", lw=0.6, alpha=0.35,
                       c="gray", zorder=0)
    a.scatter(kept[time_col], kept[area_col], s=13, alpha=0.7, c="tab:blue")
    a.set_xlabel(time_col); a.set_title("tracks, cleaned population")
    a.grid(alpha=0.25)

    a = ax[2]
    g = kept.groupby(time_col)[area_col]
    med, lo, hi = g.median(), g.quantile(0.25), g.quantile(0.75)
    a.fill_between(med.index, lo.values, hi.values, alpha=0.2, color="tab:blue")
    a.plot(med.index, med.values, "o-", c="tab:blue", lw=2, label="cleaned")
    gd = d.groupby(time_col)[area_col].median()
    a.plot(gd.index, gd.values, "--", c="tab:gray", lw=1.4, label="all rows")
    if "npc_area_um2" in kept.columns:
        n = kept[kept.npc_area_um2.notna()].groupby(time_col).npc_area_um2.median()
        if len(n):
            a.plot(n.index, n.values, ":", c="k", lw=1.8, label="NPC ring")
    a.set_xlabel(time_col); a.set_title("median with IQR")
    a.legend(fontsize=8); a.grid(alpha=0.25)

    plt.tight_layout(); plt.show()

    print("\nmedian area by timepoint")
    cmp = pd.DataFrame({"all_rows": d.groupby(time_col)[area_col].median(),
                        "cleaned": med, "n_kept": kept.groupby(time_col).size()})
    print(cmp.round(1).to_string())
    return kept

In [ ]:
kept = plot_final_area(best_z_df, cfg)          # the plot you asked for

d = add_focus_metric(best_z_df, img_5d, nucleus_instance_4d,
                     droplet_instance_4d, cfg)

kept = plot_final_area(d, cfg, max_edge_um=2.5)

In [ ]:
d = d.copy()          # the frame returned by add_focus_metric

In [ ]:
d = add_focus_metric(best_z_df, img_5d, nucleus_instance_4d,
                     droplet_instance_4d, cfg)

In [ ]:
d["rejected"] = d.repair_status.astype(str).str.startswith("rejected")
keep = ~d.exclude_edge & ~d.rejected
if "edge_width_um" in d.columns:
    keep &= (d.edge_width_um <= 2.5) | d.edge_width_um.isna()
else:
    print("no edge_width_um — plotting without the focus filter (n=408 version)")
kept = d[keep]

In [ ]:
import matplotlib.pyplot as plt

# --- focus metric: reuse if present, else compute ----------------------
if "edge_width_um" in best_z_df.columns:
    d = best_z_df.copy()
else:
    d = add_focus_metric(best_z_df, img_5d, nucleus_instance_4d,
                         droplet_instance_4d, cfg)

# --- attach true acquisition time --------------------------------------
if "true_time_min" not in d.columns:
    src = tracked_df if "tracked_df" in dir() else timed_df
    keys = [k for k in ("t", "nucleus_3d_id") if k in src.columns and k in d.columns]
    cols = keys + [c for c in ("true_time_min", "track_id")
                   if c in src.columns and c not in d.columns]
    d = d.merge(src[cols].drop_duplicates(keys), on=keys, how="left")
    print("missing times:", int(d.true_time_min.isna().sum()))

# --- population ---------------------------------------------------------
d["rejected"] = d.repair_status.astype(str).str.startswith("rejected")
keep = ~d.exclude_edge & ~d.rejected
if "edge_width_um" in d.columns:
    keep &= (d.edge_width_um <= 2.5) | d.edge_width_um.isna()
kept = d[keep]
print(f"{len(d)} -> {len(kept)} kept")

# --- scatter ------------------------------------------------------------
xcol = "true_time_min" if "true_time_min" in kept.columns else "t"
fig, ax = plt.subplots(figsize=(11, 6))
if "track_id" in kept.columns:
    for _, g in kept.groupby("track_id"):
        if len(g) > 1:
            g = g.sort_values(xcol)
            ax.plot(g[xcol], g.nucleus_area_um2, "-", lw=0.7, alpha=0.45, zorder=1)
rep = kept[kept.repair_status == "repaired"]
nf  = kept[kept.repair_status == "not fragmented"]
ax.scatter(nf[xcol], nf.nucleus_area_um2, s=26, alpha=0.8, c="tab:blue",
           zorder=2, label=f"never fragmented (n={len(nf)})")
ax.scatter(rep[xcol], rep.nucleus_area_um2, s=30, alpha=0.85, c="tab:green",
           zorder=3, label=f"repaired (n={len(rep)})")
ax.set_xlabel("True acquisition time (min)" if xcol == "true_time_min" else "t")
ax.set_ylabel("Nuclear area (µm²)")
ax.set_title("Nuclear cross-sectional area by track")
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

In [ ]:
def count_window(d, img_5d, cfg, y0=1500, x0=2500, size=800, z=None,
                 time_col=None, figsize=(18, 9)):
    """Fixed spatial window at every timepoint, with the pipeline's kept nuclei
    marked. Count the ones you'd accept, compare to n_kept."""
    if time_col is None:
        time_col = "true_time_min" if "true_time_min" in d.columns else "t"
    ts = sorted(d.t.unique())
    ncol = 3; nrow = int(np.ceil(len(ts) / ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=figsize)
    ax = np.atleast_1d(ax).ravel()
    rows = []
    for k, t in enumerate(ts):
        sub = d[(d.t == t)
                & d.centroid_y_px.between(y0, y0 + size)
                & d.centroid_x_px.between(x0, x0 + size)]
        zz = z if z is not None else int(sub.z.median()) if len(sub) else 14
        nls = np.asarray(img_5d[t, zz, cfg.nuclear_channel_index],
                         np.float32)[y0:y0+size, x0:x0+size]
        lo, hi = np.percentile(nls, [1, 99.5])
        ax[k].imshow(np.clip((nls - lo) / (hi - lo + 1e-8), 0, 1), cmap="gray")
        ax[k].scatter(sub.centroid_x_px - x0, sub.centroid_y_px - y0,
                      s=60, facecolors="none", edgecolors="lime", lw=1.4)
        ax[k].set_title(f"t={t}  z={zz}  pipeline kept: {len(sub)}", fontsize=10)
        ax[k].axis("off")
        rows.append({"t": t, "z_shown": zz, "pipeline_kept": len(sub)})
    for a in ax[len(ts):]:
        a.axis("off")
    plt.tight_layout(); plt.show()
    return pd.DataFrame(rows)

counts = count_window(kept, img_5d, cfg, y0=1500, x0=2500, size=800)

In [ ]:
for t in (1, 2, 3):
    for stage, df in (("objects", objects_df), ("grouped", grouped_z_df),
                      ("best_z", best_z_df)):
        n = len(df[df.t == t]) if "t" in df.columns else 0
        print(f"t={t:2d} {stage:>8}: {n}")
    print()

In [ ]:
t, z = 2, 14
p = np.asarray(prob[t, z, cfg.nucleus_class_index], np.float32)
for thr in (0.5, 0.3, 0.15, 0.05):
    m = p > thr
    print(f"p>{thr}: {int(measure.label(m).max())} objects, "
          f"{m.sum()*cfg.pixel_size_um**2:.0f} µm² total")

In [ ]:
piv = (objects_df.groupby(["t", "z"]).size().unstack(fill_value=0))
print(piv.to_string())
print("\nz of peak detection per timepoint:")
print(piv.idxmax(axis=1).to_string())

In [ ]:
t = 2
for z in range(img_5d.shape[1]):
    nls = np.asarray(img_5d[t, z, cfg.nuclear_channel_index], np.float32)
    print(f"z={z:2d}  p99.9={np.percentile(nls, 99.9):8.0f}  "
          f"std={nls.std():7.0f}  objects={int(((objects_df.t==t)&(objects_df.z==z)).sum())}")

In [ ]:
import inspect
for fn in (run_segmentation_for_all_planes, extract_objects_from_saved_masks):
    s = inspect.getsource(fn)
    for i, l in enumerate(s.split("\n")):
        if any(k in l for k in ("z_range", "range(", "[:3]", "head(", "n_z",
                                "z_start", "z_end", "focus", "best")):
            print(f"{fn.__name__}:{i}: {l.strip()}")
    print()

print([a for a in dir(cfg) if "z" in a.lower() or "focus" in a.lower()])

In [ ]:
print(seg_index_df.groupby("t").z.apply(lambda s: sorted(s.unique())).to_string())

In [ ]:
t = 2
for z in range(8, 16):
    plane = np.asarray(nucleus_instance_4d[t, z])
    c = np.bincount(plane.ravel()); c[0] = 0
    print(f"z={z:2d} instances in mask: {int((c >= 50).sum()):3d}  "
          f"objects_df: {int(((objects_df.t==t)&(objects_df.z==z)).sum()):3d}")

In [ ]:
import tifffile as tiff
prob = tiff.memmap(cfg.segmentation_probability_hyperstack_path, mode="r")
print(prob.shape)

t, ci = 2, cfg.nucleus_class_index
for z in range(8, 16):
    p = np.asarray(prob[t, z, ci], np.float32)
    print(f"z={z:2d}  p_max={p.max():.3f}  p99.9={np.percentile(p,99.9):.3f}  "
          f"px>0.5={int((p>0.5).sum()):>8d}  px>0.3={int((p>0.3).sum()):>8d}  "
          f"mask={int(((objects_df.t==t)&(objects_df.z==z)).sum())}")

In [ ]:
t = 2
for z in (9, 11, 14):
    for c in range(4):
        p = np.asarray(prob[t, z, c], np.float32)
        print(f"z={z:2d} c={c}: min={p.min():.6f} max={p.max():.6f} "
              f"mean={p.mean():.6f} nonzero={int((p != 0).sum())}")
    print()

In [ ]:
import inspect
src = inspect.getsource(run_segmentation_for_all_planes)
print(src)

In [ ]:
print([f"{a}={getattr(cfg,a)}" for a in dir(cfg)
       if not a.startswith('_') and ('z' in a.lower() or 'focus' in a.lower()
                                     or 'plane' in a.lower())])

In [ ]:
manual = pd.read_csv("<hand analysis>")     # time, area, n/c
bins = np.arange(0, 70, 5)
k = kept.copy(); k["bin"] = pd.cut(k.true_time_min, bins)
m = manual.copy();  m["bin"] = pd.cut(m.time, bins)
cmp = pd.DataFrame({
    "manual_median": m.groupby("bin").area.median(),
    "pipeline_median": k.groupby("bin").nucleus_area_um2.median(),
    "manual_n": m.groupby("bin").size(),
    "pipeline_n": k.groupby("bin").size()})
cmp["ratio"] = (cmp.pipeline_median / cmp.manual_median).round(3)
print(cmp.round(1).to_string())

In [ ]:
best_z_df["area_valid"] = ~best_z_df.repair_status.str.startswith("rejected")